# **wiki2shards**

## Phase 0: Workbench (M0)

### Step 0.1: Create the skeleton and pin Python

```py
uv python pin 3.12.14
```
- The command `uv python pin 3.12.14` sets Python version 3.12.14 as the active Python version for your current working directory and project.
- Creates/Updates `.python-version`: It writes `3.12.14` to a `.python-version` file in the current directory.
- According to Charlie Marsh (creator of uv and founder of Astral), uv officially stands for "Ultraviolet".

```sh
mkdir -p src/wiki2shards/{io,acquire,extract,clean,tokenize,pack,loader,pipeline}
```
- `-p` stands for `parent` used in parent directory

```sh
touch src/wiki2shards/{io,acquire,extract,clean,tokenize,pack,loader,pipeline}/__init__.py
```

- creates a `__init__.py` inside every single directory within the curly brackets

```sh
echo "# wiki2shards" > README.md
```
- writes `# wiki2shards` into `README.md`

### Step 0.2: pyproject.toml

`pyproject.toml`
- `pyproject.toml` is the modern standard configuration file for Python projects. It centralizes project metadata, dependencies, and configuration for various development tools into a single file, replacing the older, fragmented ecosystem of setup.py, setup.cfg, and requirements.txt.

**`[project]`**
Declares the start of the core, standardized project metadata block (defined by PEP 621).

* **`name = "wiki2shards"`**
Sets the package's name. This is what you would use to pip install it.
* **`version = "0.1.0"`**
Sets the current version of the project.
* **`description = "Wikipedia dump -> checksummed token shards, plus a deterministic, resumable, mixture-aware loader"`**
Provides a short summary of the project's purpose.
* **`readme = "README.md"`**
Specifies that the long description (usually displayed on PyPI) should be pulled from the `README.md` file.
* **`requires-python = ">=3.12,<3.13"`**
Restricts this package to only run on Python 3.12.x. It will fail to install on Python 3.11 or Python 3.13.
* **`dependencies = [`**
Starts the list of core runtime dependencies required for the package to function.
* **`"numpy>=2.0",`**
Requires NumPy version 2.0 or higher.
* **`"mwparserfromhell==0.7.2",  # output bytes depend on it (wikitext goldens)`**
Pins this Wikipedia parsing library to exactly version 0.7.2. The comment explains this strict pinning ensures the processed output remains identical across builds.
* **`"tiktoken>=0.9",`**
Requires the `tiktoken` (OpenAI tokenizer) library version 0.9 or higher.
* **`"tokenizers==0.23.3",  # output bytes depend on it (fixture tokenizer)`**
Pins Hugging Face `tokenizers` to exactly 0.23.3 to guarantee deterministic tokenization.
* **`"pydantic>=2.7",`**
Requires Pydantic version 2.7 or higher for data validation.
* **`"typer>=0.12",`**
Requires Typer version 0.12 or higher to build the command-line interface.
* **`"pyyaml>=6.0",`**
Requires PyYAML version 6.0 or higher to parse YAML files.
* **`]`**
Closes the core dependencies list.

**`[project.scripts]`**
Defines console commands (CLI entry points) that get automatically installed with the package.

* **`wiki2shards = "wiki2shards.cli:main"`**
Creates a terminal command named `wiki2shards` that, when run, executes the `main` function inside the `wiki2shards.cli` Python module.

**`[dependency-groups]`**
Defines standard dependency groups (PEP 735) for different environments, natively supported by tools like `uv`.

* **`dev = [`**
Starts a group named `dev` containing dependencies only needed for developing the package, not running it.
* **`"pytest>=8",`**
Requires the Pytest testing framework version 8 or higher.
* **`"hypothesis>=6.100",`**
Requires Hypothesis v6.100+ for property-based testing.
* **`"pytest-timeout>=2.3",`**
Requires the pytest-timeout plugin v2.3+ to forcibly stop tests that hang.
* **`"ruff>=0.6",`**
Requires the Ruff linter/formatter version 0.6 or higher.
* **`"mypy>=1.11",`**
Requires the Mypy static type checker version 1.11 or higher.
* **`"types-PyYAML>=6.0",`**
Installs type hints for PyYAML so Mypy can properly type-check YAML-related code.
* **`]`**
Closes the dev dependencies list.

**`[build-system]`**
Defines the tools pip needs to build the package from source into a wheel (PEP 518).

* **`requires = ["hatchling"]`**
Specifies that the `hatchling` package must be downloaded before the build can start.
* **`build-backend = "hatchling.build"`**
Tells the build frontend (like pip) to use Hatchling's build system to package the code.

**`[tool.hatch.build.targets.wheel]`**
Configuration specific to the Hatchling build backend regarding how to build the `.whl` (wheel) file.

* **`packages = ["src/wiki2shards"]`**
Tells Hatchling that the actual Python code to be packaged lives in the `src/wiki2shards` directory (a standard `src` layout).

**`[tool.pytest.ini_options]`**
Configuration block for the Pytest testing framework.

* **`testpaths = ["tests"]`**
Tells Pytest to only look for tests inside the `tests` directory.
* **`pythonpath = ["tests"]`**
Adds the `tests` directory to the Python path during a test run, allowing test files to import helper functions from other test files.
* **`addopts = "-p no:cacheprovider --strict-markers"`**
Default command-line flags applied to every pytest run: disables test result caching (`-p no:cacheprovider`) and throws an error if an unregistered test marker is used (`--strict-markers`).
* **`timeout = 120`**
Instructs `pytest-timeout` to automatically fail any single test that takes longer than 120 seconds.
* **`markers = [`**
Starts a list to register custom test markers (decorators like `@pytest.mark.slow`).
* **`"slow: subprocess builds, crash matrix, SIGKILL resume (excluded from the fast suite)",`**
Registers the `slow` marker and provides a description of what it's used for.
* **`"bench: benchmarks, never gating",`**
Registers the `bench` marker used for benchmarking tests.
* **`]`**
Closes the markers list.

**`[tool.ruff]`**
Configuration block for the Ruff linter and formatter.

* **`line-length = 100`**
Sets the maximum allowed characters per line to 100 (overriding the default 88).
* **`target-version = "py312"`**
Tells Ruff to assume the code is running on Python 3.12, allowing it to apply modern formatting and linting rules specific to that version.

**`[tool.ruff.lint]`**
Specific configuration for Ruff's linting features.

* **`select = ["E", "F", "W", "I", "B", "UP", "SIM", "RUF"]`**
Enables specific families of linting rules: `E` and `W` (pycodestyle errors/warnings), `F` (Pyflakes), `I` (isort for imports), `B` (flake8-bugbear for common bugs), `UP` (pyupgrade for modern syntax), `SIM` (flake8-simplify), and `RUF` (Ruff's own rules).

**`[tool.mypy]`**
Configuration block for the Mypy static type checker.

* **`strict = true`**
Turns on Mypy's most restrictive mode, requiring type hints for everything and prohibiting dynamic typing loopholes.
* **`python_version = "3.12"`**
Tells Mypy to type-check against Python 3.12 rules.
* **`files = ["src"]`**
Instructs Mypy to only check the code inside the `src` directory (ignoring tests or scripts).

**`[[tool.mypy.overrides]]`**
Starts an override block (denoted by double brackets `[[ ]]` representing an array of tables in TOML) to apply specific Mypy rules to specific libraries.

* **`module = ["mwparserfromhell", "mwparserfromhell.*"]`**
Targets the `mwparserfromhell` library and all of its submodules for this override.
* **`ignore_missing_imports = true`**
Tells Mypy to suppress errors about this library missing type hints (since it does not provide them natively and no stub package exists).

### Step 0.3: Package marker, then install

`src/wiki2shards/__init__.py`

### Step 0.4: Crash-safe writes

`src/wiki2shards/io/atomic.py`

```python
"""Crash-safe writes: tmp -> fsync -> os.replace -> fsync(dir).

A reader can never observe a partially written file under its final name: the bytes land in a
hidden tmp file next to it, and only a complete, fsynced file is renamed into place. os.replace
is atomic on POSIX within one filesystem, which is why the tmp file lives in the same directory.
"""

```

* **Lines 1-6:** A module-level docstring summarizing the entire script's purpose. It explains the core mechanism of atomic file writes: data is written to a temporary file, synced to disk, and then atomically renamed over the original file so no other program ever sees a partially written state.

```python
from __future__ import annotations

```

* **Line 8:** Instructs Python to delay evaluating type hints. This allows you to use modern type hint syntax (like `Path | str`) even in older Python versions, and prevents circular import issues with typing.

```python
import os
from collections.abc import Iterator
from contextlib import contextmanager
from pathlib import Path
from typing import BinaryIO

```

* **Line 10:** Imports the standard `os` module, providing low-level operating system interfaces (like `fsync`, `replace`, `open`, `close`).
* **Line 11:** Imports `Iterator` for type hinting the return type of the context manager.
* **Line 12:** Imports the `contextmanager` decorator, which allows writing a function that behaves like a class with `__enter__` and `__exit__` methods for use in a `with` statement.
* **Line 13:** Imports `Path` from `pathlib` for modern, object-oriented filesystem path manipulation.
* **Line 14:** Imports `BinaryIO` for type hinting a file object opened in binary mode.

```python
TMP_SUFFIX = ".tmp"

```

* **Line 16:** Defines a constant string `.tmp` that will be appended to the end of temporary files.

```python
def tmp_name(final: Path) -> Path:
    """Hidden sibling used while `final` is being written."""
    return final.with_name(f".{final.name}{TMP_SUFFIX}")

```

* **Line 19:** Defines a function `tmp_name` that takes a `Path` object (the target file) and returns a new `Path` object.
* **Line 20:** A docstring explaining the function's purpose.
* **Line 21:** Generates the temporary file path. It takes the original filename (`final.name`), prepends a dot `.` to make it a hidden file on UNIX/Linux systems, and appends the `.tmp` suffix. It replaces the original filename in the original directory (`final.with_name(...)`).

```python
def fsync_dir(path: Path) -> None:
    """Persist a directory entry (the rename) itself."""
    fd = os.open(path, os.O_RDONLY)
    try:
        os.fsync(fd)
    finally:
        os.close(fd)

```

* **Line 24:** Defines a function `fsync_dir` that takes a directory path and returns nothing (`None`).
* **Line 25:** A docstring explaining that this persists the directory metadata (necessary to guarantee a file rename survives a power loss).
* **Line 26:** Uses the low-level `os.open` to open the directory itself in read-only mode (`os.O_RDONLY`), returning an integer file descriptor (`fd`).
* **Line 27:** Starts a `try` block to ensure cleanup happens even if an error occurs.
* **Line 28:** Calls `os.fsync()` on the directory's file descriptor, forcing the operating system to flush the directory's metadata changes to physical storage.
* **Line 29:** Starts the `finally` block, which executes regardless of whether the `try` block succeeded or threw an exception.
* **Line 30:** Closes the low-level file descriptor to prevent resource leaks.

```python
@contextmanager
def atomic_open(final: Path | str) -> Iterator[BinaryIO]:
    """Open a binary writer whose content appears at `final` only if the block succeeds."""

```

* **Line 33:** Applies the `@contextmanager` decorator, converting the generator function below into a context manager that can be used with a `with` statement.
* **Line 34:** Defines the `atomic_open` function, which accepts either a `Path` or a string, and yields a binary file object (`BinaryIO`).
* **Line 35:** A docstring explaining that the file will only appear at its final destination if the code block inside the `with` statement completes without errors.

```python
    final = Path(final)
    tmp = tmp_name(final)
    f = open(tmp, "wb")  # noqa: SIM115 - closed explicitly on both paths below

```

* **Line 36:** Normalizes the input `final` by ensuring it is a `Path` object (in case a string was passed).
* **Line 37:** Calls `tmp_name()` to determine the hidden temporary file path.
* **Line 38:** Opens the temporary file for writing in binary mode (`"wb"`). The `# noqa: SIM115` comment tells a specific linter (flake8-simplify) to ignore the warning about opening a file without using a `with` statement, since this function manually manages the closing.

```python
    try:
        yield f
        f.flush()
        os.fsync(f.fileno())
        f.close()
        os.replace(tmp, final)
        fsync_dir(final.parent)

```

* **Line 39:** Starts the `try` block for the "success" path.
* **Line 40:** `yield f` hands the open temporary file object back to the caller's `with` block. Execution pauses here until the user's `with` block finishes.
* **Line 41:** Once the user's block finishes successfully, execution resumes. `f.flush()` forces Python's internal memory buffers to write to the OS.
* **Line 42:** `os.fsync(f.fileno())` tells the OS to physically write its own disk buffers to the hardware storage, guaranteeing the data is saved.
* **Line 43:** Closes the temporary file. It is now safely written to disk, but still has the `.tmp` name.
* **Line 44:** `os.replace` atomically renames the temporary file to the final filename. If a file already exists at `final`, it is silently and atomically overwritten.
* **Line 45:** Calls the previously defined `fsync_dir` on the parent directory (`final.parent`) to guarantee that the OS persists the file rename to disk.

```python
    except BaseException:
        f.close()
        tmp.unlink(missing_ok=True)
        raise

```

* **Line 46:** The `except BaseException` block catches *any* exception that occurred while the user was interacting with the yielded file in the `with` block, including system-exiting exceptions like `KeyboardInterrupt`.
* **Line 47:** Closes the file object to release the resource.
* **Line 48:** Deletes the temporary file (`unlink`). `missing_ok=True` prevents a `FileNotFoundError` if the file somehow doesn't exist.
* **Line 49:** `raise` re-throws the original exception so the calling program knows an error occurred and can handle it or crash normally.

```python
def atomic_write_bytes(final: Path | str, data: bytes) -> None:
    with atomic_open(final) as f:
        f.write(data)

```

* **Line 52:** Defines a helper function that takes a target path and a payload of raw bytes, returning nothing.
* **Line 53:** Uses the `atomic_open` context manager defined above to safely open the destination file.
* **Line 54:** Writes the entire `data` byte payload into the yielded file object. Once this block ends, `atomic_open` handles flushing, syncing, and renaming.

```python
def commit(src: Path, final: Path) -> None:
    """Move an already-complete file (written elsewhere in the same filesystem) into place."""
    with open(src, "rb") as f:
        os.fsync(f.fileno())
    os.replace(src, final)
    fsync_dir(final.parent)

```

* **Line 57:** Defines a `commit` function intended to take an existing file (`src`) and safely finalize it as `final`.
* **Line 58:** A docstring explaining its use case.
* **Line 59:** Opens the source file in read-binary mode using a standard `with` context manager.
* **Line 60:** Calls `os.fsync()` on the source file to ensure any pending writes made by whoever created the file are physically on disk before moving it.
* **Line 61:** The `with` block ends, automatically closing the source file.
* **Line 62:** Atomically renames the source file to its final destination.
* **Line 63:** Syncs the destination's parent directory to ensure the rename operation is permanently recorded on disk.

### Step 0.5: Hashing, canonical JSON and document identity

`src/wiki2shards/io/hashing.py`

> File Header & Setup

* `"""Stable hashing, canonical JSON and document identity.` — The start of a multi-line comment describing what this file does.
* `Nothing here uses the built-in hash(): str hashing is salted per process (PYTHONHASHSEED), so any` — Explains that Python's default tool for mixing data (`hash()`) changes every time you restart the program.
* `value derived from it would differ between runs. Everything persisted goes through sha256.` — Because default hashes change, this code uses `sha256` (a standard mathematical formula) to ensure digital fingerprints stay exactly the same forever.
* `"""` — The end of the multi-line comment.
* `from __future__ import annotations` — Allows the code to use modern Python type hints (labels telling you what type of data a variable is) even on older Python versions.
* `import hashlib` — Brings in Python's built-in tool for creating secure digital fingerprints (hashes).
* `import json` — Brings in the tool to read and write JSON, a very common text format used to store and send data.
* `from pathlib import Path` — Brings in a tool that makes it easy to work with computer file and folder paths.
* `from typing import Any` — Brings in a special label (`Any`) used to tell the code "this variable can hold absolutely any type of data."
* `_CHUNK = 1 << 20` — Creates a constant variable set to 1,048,576 (which is exactly 1 Megabyte). This is used to read large files in small pieces so they don't crash the computer's memory.

> Hashing Functions (Making Digital Fingerprints)

* `def sha256_hex(data: bytes) -> str:` — Defines a function named `sha256_hex` that takes raw computer bytes and returns a standard text string.
* `return hashlib.sha256(data).hexdigest()` — Creates a SHA-256 fingerprint of the bytes and hands it back as readable text (hexadecimal).
* `def sha256_file(path: Path | str) -> str:` — Defines a function that takes a file path (as a Path object or text string) and returns a text string.
* `h = hashlib.sha256()` — Starts an empty SHA-256 fingerprint calculator.
* `with open(path, "rb") as f:` — Safely opens the file in "read binary" (`rb`) mode so it can read the raw 1s and 0s.
* `while chunk := f.read(_CHUNK):` — A loop that reads the file one 1-Megabyte chunk at a time until the file is empty.
* `h.update(chunk)` — Adds that 1-Megabyte chunk into the fingerprint calculator.
* `return h.hexdigest()` — Once the whole file is read, hands back the final fingerprint as text.
* `def sha1_file(path: Path | str) -> str:` — Defines a function exactly like the one above, but for an older, faster fingerprint format called SHA-1.
* `h = hashlib.sha1()` — Starts an empty SHA-1 fingerprint calculator.
* `with open(path, "rb") as f:` — Safely opens the file in "read binary" mode.
* `while chunk := f.read(_CHUNK):` — Reads the file one 1-Megabyte chunk at a time.
* `h.update(chunk)` — Adds the chunk into the SHA-1 calculator.
* `return h.hexdigest()` — Hands back the final SHA-1 fingerprint as text.

> Strict Data Checking

* `def _check(obj: Any, where: str = "$") -> None:` — Defines a hidden function that inspects data to ensure it is perfectly safe for strict JSON. The `where` variable tracks where we are in the data structure (starting at `$`, the root).
* `"""Reject anything whose JSON encoding is not unique: floats, NaN, non-string keys."""` — A comment explaining that decimal numbers and weird dictionary keys are banned because different computers format them differently.
* `if isinstance(obj, bool) or obj is None or isinstance(obj, (int, str)):` — Checks if the data is a safe, simple type: True/False, empty (`None`), a whole number, or text.
* `return` — If it is safe, stop checking and approve it (do nothing).
* `if isinstance(obj, float):` — Checks if the data is a decimal number (a float).
* `raise TypeError(f"float at {where}: canonical JSON is integers only")` — If it is a decimal, crash the program and print an error showing exactly where the decimal was found.
* `if isinstance(obj, dict):` — Checks if the data is a dictionary (a collection of key-value pairs).
* `for k, v in obj.items():` — Loops through every single key (`k`) and value (`v`) inside the dictionary.
* `if not isinstance(k, str):` — Checks if the dictionary key is anything other than text.
* `raise TypeError(f"non-string key {k!r} at {where}")` — If the key isn't text, crash the program with an error.
* `_check(v, f"{where}.{k}")` — Takes the dictionary value and sends it back through this exact same `_check` function to inspect it. This ensures nested data is checked all the way down.
* `return` — Once the whole dictionary is checked, stop.
* `if isinstance(obj, (list, tuple)):` — Checks if the data is a list or tuple (an ordered collection of items).
* `for i, v in enumerate(obj):` — Loops through the list, keeping track of the item's position number (`i`) and the item itself (`v`).
* `_check(v, f"{where}[{i}]")` — Sends each list item back through the `_check` function to inspect it, recording its position.
* `return` — Once the whole list is checked, stop.
* `raise TypeError(f"unsupported type {type(obj).__name__} at {where}")` — If the data isn't any of the allowed types above, crash the program and say the data type is unsupported.

> JSON Formatting (Canonical JSON)

* `def canonical_json(obj: Any) -> bytes:` — Defines a function that turns data into a highly strict, squished-down JSON byte format.
* `"""Compact form: stage keys, JSONL lines, journals."""` — A comment saying this squished format is meant for tight storage and logs.
* `_check(obj)` — Runs the data through our `_check` function above to make sure it contains no banned elements.
* `text = json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(",", ":"))` — Converts the data into text JSON. It forces the keys into alphabetical order, allows foreign language characters, and removes all extra spaces.
* `return text.encode("utf-8")` — Converts that strict JSON text into raw computer bytes using the standard UTF-8 format.
* `def canonical_json_pretty(obj: Any) -> bytes:` — Defines a function that turns data into strict JSON, but formatted beautifully for human eyes.
* `"""Manifest form: sorted keys, indent 2, UTF-8, trailing newline."""` — A comment explaining this format is spaced out and easy to read.
* `_check(obj)` — Checks the data for banned elements again.
* `text = json.dumps(obj, sort_keys=True, ensure_ascii=False, indent=2, separators=(",", ": "))` — Converts the data into text JSON, sorts the keys, adds a 2-space indentation for neatness, and puts spaces after colons.
* `return (text + "\n").encode("utf-8")` — Adds an invisible "Enter" (new line) at the very end of the text, then converts it to bytes.

> Document Identity (Unique IDs)

* `def doc_uid(wiki: str, page_id: int) -> int:` — Defines a function that creates a unique ID number by combining a wiki name and a page number.
* `"""Little-endian u64 of the first 8 bytes of sha256(f"{wiki}:{page_id}")."""` — A comment explaining the math used to generate the ID.
* `digest = hashlib.sha256(f"{wiki}:{page_id}".encode()).digest()` — Mashes the wiki name and page ID together into text (like "enwiki:123"), converts it to bytes, and generates a raw SHA-256 fingerprint.
* `return int.from_bytes(digest[:8], "little")` — Chops off just the first 8 bytes of that fingerprint and converts them into a massive whole number.
* `def uid_hex(uid: int) -> str:` — Defines a function that takes that massive whole number ID and turns it into text.
* `"""JSON form of a doc_uid: hex of the integer, so string order equals numeric order."""` — Explains this makes the ID safe to save in JSON while making sure alphabetical order matches numerical order.
* `return f"{uid:016x}"` — Converts the number into a 16-character text string using letters and numbers, adding zeros to the front if it is too short.
* `def parse_uid_hex(text: str) -> int:` — Defines a function that takes the 16-character text ID and turns it back into a number.
* `if len(text) != 16:` — Checks if the text is exactly 16 characters long.
* `raise ValueError(f"doc_uid must be 16 hex digits, got {text!r}")` — If it isn't exactly 16 characters, crash the program with an error.
* `return int(text, 16)` — Converts the 16-character text string back into a standard whole number.
* `def text_sha256(text: str) -> str:` — Defines a simple function to quickly get the fingerprint of plain text.
* `return sha256_hex(text.encode("utf-8"))` — Converts the text into raw bytes and passes it to our very first function (`sha256_hex`) to get the fingerprint.

### Step 0.6: Fault injection

`src/wiki2shards/pipeline/faults.py`

| Code Line | Plain English Explanation |
| --- | --- |
| `"""Fault injection: W2S_FAULT="<stage>:<point>:<n>" kills the coordinator at the n-th hit.` | Opens a multi-line comment describing what this code does: it's a tool to intentionally crash a program by setting a specific rule. |
| `os._exit skips finally blocks, atexit handlers and buffered flushes, exactly like SIGKILL, so the` | Continues the comment, explaining that this tool forces a brutal, instant crash without giving the program any time to clean up its files or memory. |
| `crash tests see the same on-disk state a real kill would leave behind.` | Concludes the comment, noting that this helps developers test how the system reacts to sudden power cuts or forced closures. |
| `"""` | Closes the multi-line comment block. |
| `from __future__ import annotations` | A special instruction that helps developers write clearer, more modern code labels (type hints) in older versions of Python. |
| `import os` | Brings in Python's built-in operating system tools, which are needed to read computer settings and force the program to quit. |
| `FAULT_EXIT_CODE = 137` | Sets a specific error number (137) that the program will announce when it intentionally crashes itself. |
| `_hits: dict[tuple[str, str], int] = {}` | Creates an empty digital scoreboard (a dictionary) to keep a running tally of how many times different parts of the code are reached. |
| `def hit(stage: str, point: str) -> None:` | Creates a reusable block of code (a function) named `hit`. You give it a `stage` (a phase of the program) and a `point` (a specific spot). |
| `    spec = os.environ.get("W2S_FAULT")` | Looks at your computer's hidden settings (environment variables) to see if you gave it a crash instruction named "W2S_FAULT". |
| `    if not spec:` | Checks if that crash instruction is missing or empty. |
| `        return` | If there is no crash instruction, exit this block of code immediately and let the program continue running normally. |
| `    want_stage, want_point, n = spec.split(":")` | If there *is* an instruction, chop it into three pieces separated by colons: the exact stage, the exact point, and the number of times to hit it. |
| `    if (stage, point) != (want_stage, want_point):` | Checks if the program's current location matches the exact location where you told it to crash. |
| `        return` | If the locations don't match, exit this block of code and let the program continue normally. |
| `    count = _hits.get((stage, point), 0) + 1` | Checks the scoreboard to see how many times the program has visited this exact spot before, and adds 1 to that number. |
| `    _hits[(stage, point)] = count` | Updates the scoreboard with the new, higher count. |
| `    if count == int(n):` | Checks if the number of times we've visited this spot exactly matches the target crash number you set. |
| `        os._exit(FAULT_EXIT_CODE)` | If the numbers match, instantly and violently kill the program, spitting out the error number 137. |

### Step 0.7: The fixture dump generator

`scripts/make_fixture_dump.py`

Here is a plain-English breakdown of what this script does, section by section.

This script generates **fake Wikipedia database dumps** for testing software. Instead of downloading gigabytes of real Wikipedia data, developers can run this script to generate a small, predictable, "dummy" version of Wikipedia that always looks exactly the same every time it is generated.

> 1. Introduction and Setup

* **The Docstring (`""" ... """`)**: Explains what the script does. It creates files that perfectly mimic the structure of real Wikimedia dump files (XML compressed with bz2). It emphasizes that it uses no random number generators—everything is strictly calculated so the output never changes between computers.
* **Imports**: Loads built-in Python tools needed for the job: `argparse` (to read command-line commands), `bz2` (to compress files), `hashlib` (for generating predictable numbers/checksums), `json` (for saving test answers), `Path` (for handling file folders), and `escape/quoteattr` (to safely format text into XML).

> 2. The Raw Ingredients (Constants)

* `PAGES_PER_STREAM = 100`: Groups 100 wiki pages into a single compressed block, just like real Wikipedia dumps.
* `BASE_DATE`: A fake date used for the file names.
* `WORDS`: A giant list of simple English words. The script uses these to generate fake sentences and page titles.
* `COUNTS`: A dictionary deciding exactly how many pages of specific "weird" types to create (e.g., 20 redirects, 10 talk pages, 1 emoji page).
* `TOTAL_PAGES = 300`: The script will generate exactly 300 pages in total.

> 3. Generating Predictable "Random" Text

Instead of using Python's `random` module (which can act differently on different machines), the script uses math to pick words.

* **`_h` function**: Takes any input (like a page name), hashes it using SHA-256, and turns it into a permanent, unique number.
* **`words` function**: Uses that unique number to pick words from the `WORDS` list over and over to form a list of fake words.
* **`prose` function**: Takes those words, capitalizes the first letter, adds periods, and groups them into 5-sentence paragraphs to look like real article text.
* **`article_wikitext` function**: Glues the fake paragraphs together with real MediaWiki formatting (like `[[links]]`, `{{infoboxes}}`, `<ref>` tags, and tables) so the testing software has complex formatting to practice reading.

> 4. Building the Pages

* **`Page` class**: A simple blueprint to hold a page's data (its title, category, namespace ID, and text).
* **`build_pages` function**:
* Figures out how many standard "article" pages to make to hit exactly 300 pages.
* Loops through every category (redirects, empty pages, short pages, foreign languages, etc.) and generates the text for them using the functions above.
* Sorts them using the `_h` math function so they are thoroughly—but predictably—mixed up.
* If `variant 2` is requested (simulating a wiki update that happens later in time), it deletes 5 pages, edits a few, and adds 5 new ones.

> 5. Keeping IDs Stable

* **`page_ids` function**: Real Wikipedia pages have ID numbers that never change, even if other pages are deleted. This function calculates permanent ID numbers for every generated page.

> 6. Formatting as XML

* **`page_xml` function**: Takes a `Page` object and wraps it in MediaWiki XML tags (like `<page>`, `<title>`, `<text>`).
* **`header_xml` function**: Creates the standard XML header that sits at the very top of the dump file, declaring the site's name as "Fixturepedia".

> 7. Creating the "Answer Key"

* **`expected_answers` function**: Since this script generates test data, it also generates a "cheat sheet" (a JSON file). This file calculates exactly how many pages are valid, how many are empty, and how many are duplicates. The testing software will use this JSON file to grade itself and make sure it parsed the wiki dump correctly.

> 8. Compressing and Saving the Files

* **`make_dump` function**: This is where the actual files are built.
* It gathers all the pages and their IDs.
* It breaks the pages into chunks of 100.
* It converts them to XML, creates an index (so software knows exactly where in the file a specific page lives), and heavily compresses them into `.bz2` format.
* It creates the folder structure (`data/fixture_dump/fixturewiki/20260901/`).
* It writes the compressed XML file, the compressed index file, a `sha1sums.txt` file (to prove the files aren't corrupted), and the `fixture_expected.json` answer key to your hard drive.

> 9. The Execution Trigger

* **`main` function**: Sets up the command-line interface. When you type `python make_fixture_dump.py --out data`, this function reads your instructions and triggers `make_dump`.
* **`if __name__ == "__main__":`**: A standard Python rule that says, "If someone runs this script directly, start the `main()` function."

### Step 0.8: Shared test setup and the first tests

#### `tests/conftest.py`

`conftest.py` registers a deterministic Hypothesis profile and blocks network sockets. It also defines session-scoped fixtures that build the fixture dataset once per test run; those use helpers that arrive in later phases and are imported lazily.

```python
"""Shared test setup: Hypothesis profiles, a network guard, and session-scoped fixture builds."""

```

This is a comment at the very top of the file (a docstring) explaining what this file does: it sets up testing tools, blocks internet access during tests, and creates reusable test data.

```python
from __future__ import annotations

```

This allows Python to use modern type-hinting features (where you specify what type of data a variable is) without crashing on older Python versions.

```python
import atexit

```

Imports a built-in Python tool that lets you schedule tasks to run right before the program finishes and closes.

```python
import os

```

Imports a built-in tool to interact with your operating system, like reading environment variables or navigating folders.

```python
import shutil

```

Imports a built-in tool for managing files and folders, like deleting entire directories.

```python
import socket

```

Imports a built-in tool that Python uses to create network and internet connections.

```python
import tempfile

```

Imports a built-in tool used to create temporary files and folders that won't clutter your hard drive.

```python
from pathlib import Path

```

Imports a modern tool that makes it easier to work with file and folder paths across different operating systems (Windows, Mac, Linux).

```python
import pytest

```

Imports `pytest`, which is the main framework being used to write and run the tests.

```python
from hypothesis import HealthCheck, settings

```

Imports specific tools from `hypothesis`, a library that automatically generates test data to help find bugs.

```python
from hypothesis.configuration import set_hypothesis_home_dir

```

Imports a function to change the folder where `hypothesis` saves its background files and caches.

```python
REPO = Path(__file__).resolve().parents[1]

```

Calculates the exact folder path to the main project directory. It looks at the current file's path (`__file__`), gets the absolute path (`resolve`), and goes up two folder levels (`parents[1]`).

```python
os.chdir(REPO)  # configs use CWD-relative paths (tokenizer, language codes)

```

Changes the current working directory to the main project folder so that any files the tests need to read are found in the right place.

```python
# Hypothesis keeps caches (constants, unicode data) in ./.hypothesis even with database=None.
# Tests write only under temporary directories, so point its home at one for this session.

```

Comments explaining why the next few lines are necessary: to prevent `hypothesis` from leaving leftover cache files in the project folder.

```python
_HYPOTHESIS_HOME = tempfile.mkdtemp(prefix="w2s-hypothesis-")

```

Creates a new temporary folder on your computer starting with the name `w2s-hypothesis-` and saves its path to the variable `_HYPOTHESIS_HOME`.

```python
atexit.register(shutil.rmtree, _HYPOTHESIS_HOME, True)

```

Tells Python: "When the tests are completely finished, use `shutil.rmtree` to delete this temporary folder so it doesn't leave junk on the computer."

```python
set_hypothesis_home_dir(_HYPOTHESIS_HOME)

```

Tells `hypothesis` to use that temporary folder for its background files instead of the default project folder.

```python
# "ci" is deterministic and writes no example database (tests write only under tmp_path).

```

A comment explaining that the next block configures a testing profile for "CI" (Continuous Integration - the automated server that runs tests). It says it will be predictable and won't save files to the hard drive.

```python
settings.register_profile(

```

Starts the process of saving a specific configuration setup (a "profile") for `hypothesis`.

```python
    "ci",

```

Names this specific configuration profile "ci".

```python
    max_examples=60,

```

Tells `hypothesis` to generate and run a maximum of 60 random test cases per test.

```python
    deadline=None,

```

Turns off the time limit, meaning tests won't automatically fail if they run a little slowly.

```python
    derandomize=True,

```

Forces the randomly generated test data to be exactly the same every single time you run the tests, which makes debugging easier.

```python
    database=None,

```

Tells `hypothesis` not to save previously failed test cases to a database on the hard drive.

```python
    suppress_health_check=[HealthCheck.too_slow],

```

Tells `hypothesis` not to complain or throw a warning if generating the test data takes a long time.

```python
)

```

Closes the profile configuration block.

```python
settings.register_profile("explore", max_examples=500, deadline=None, database=None)

```

Creates a second configuration profile named "explore" that is more aggressive: it runs 500 examples instead of 60 to hunt for deeper bugs.

```python
settings.load_profile(os.environ.get("HYPOTHESIS_PROFILE", "ci"))

```

Activates a profile. It checks if your computer has an environment variable called `HYPOTHESIS_PROFILE` set (like "explore"). If not, it defaults to using the "ci" profile we just made.

```python
@pytest.fixture(autouse=True)

```

A special tag (decorator) that tells `pytest`: "The following function is a setup task, and you should run it automatically before every single test."

```python
def _no_network(monkeypatch: pytest.MonkeyPatch) -> None:

```

Defines a function named `_no_network` that takes a tool called `monkeypatch` (which lets you temporarily replace parts of Python). `-> None` means this function doesn't return a value.

```python
    """Tests never touch the network: refuse any internet socket connection."""

```

A comment explaining the function's purpose: to strictly block the tests from accessing the internet.

```python
    real_connect = socket.socket.connect

```

Saves a copy of Python's real, original function that connects to the internet, so it can still be used for local (non-internet) connections.

```python
    def guarded(self: socket.socket, address: object) -> None:

```

Defines a fake, custom connection function inside our setup function, which will act as a security guard.

```python
        if self.family in (socket.AF_INET, socket.AF_INET6):

```

Checks if the connection being attempted is an IPv4 or IPv6 internet connection.

```python
            raise RuntimeError(f"network access attempted in a test: {address!r}")

```

If it *is* an internet connection, this deliberately crashes the test with an error message showing what address it tried to connect to.

```python
        real_connect(self, address)  # type: ignore[arg-type]

```

If it is a safe local connection (like reading a local file socket), it goes ahead and uses the `real_connect` function we saved earlier.

```python
    monkeypatch.setattr(socket.socket, "connect", guarded)

```

Takes our fake `guarded` function and temporarily replaces Python's built-in internet connection tool with it for the duration of the tests.

```python
@pytest.fixture(scope="session")

```

A tag telling `pytest`: "The following setup function is expensive, so only run it once for the entire test session, and let all tests share the result."

```python
def dumps(tmp_path_factory: pytest.TempPathFactory) -> Path:

```

Defines a function named `dumps` that asks `pytest` for a tool to make temporary folders (`tmp_path_factory`). It promises to return a file `Path`.

```python
    """Fixture dumps: fixturewiki at two dates, plus a second wiki for two-source mixtures."""

```

A comment explaining that this function builds fake Wikipedia data dumps for testing purposes.

```python
    from helpers.build import make_dumps

```

Imports a custom helper function named `make_dumps` from another file in the project. (It imports it here instead of at the top of the file so it only loads if this function actually runs).

```python
    return make_dumps(tmp_path_factory.mktemp("dumps"))

```

Creates a temporary folder named "dumps", builds the fake data inside it using `make_dumps`, and hands that folder's path to any test that asks for it.

```python
@pytest.fixture(scope="session")

```

Another tag telling `pytest` to run the following setup function only once for the whole test session.

```python
def fixture_build(dumps: Path, tmp_path_factory: pytest.TempPathFactory):  # type: ignore[no-untyped-def]

```

Defines a function named `fixture_build` that requires the `dumps` folder we just made in the previous step, and the temporary folder tool.

```python
    from helpers.build import build_fixture

```

Imports another custom helper function called `build_fixture` from a local project file.

```python
    return build_fixture(tmp_path_factory.mktemp("build_a"), dumps)

```

Creates a new temporary folder named "build_a", processes the `dumps` data into a usable test environment, and returns it for tests to use.

```python
@pytest.fixture(scope="session")

```

Another tag telling `pytest` to run this setup function only once per session.

```python
def fixture_build_b(dumps: Path, tmp_path_factory: pytest.TempPathFactory):  # type: ignore[no-untyped-def]

```

Defines a function named `fixture_build_b` to create a second, slightly different test environment, also requiring the original `dumps` folder.

```python
    from helpers.build import build_fixture

```

Imports the `build_fixture` tool again.

```python
    return build_fixture(tmp_path_factory.mktemp("build_b"), dumps, wiki="fixture2wiki")

```

Creates a temporary folder named "build_b" and builds a second environment based on a different fake wiki named "fixture2wiki", making it available for tests that need two different setups at once.

#### `tests/unit/test_io_atomic.py`

This code tests a custom file-saving tool to ensure it saves files "atomically"—meaning the save is all-or-nothing. If a crash happens while saving, it guarantees you won't be left with a corrupted, half-written file.

Here is what every line does:

* `import subprocess`: Brings in a Python tool that allows this script to launch and run other programs in the background.
* `import sys`: Brings in a tool to interact with the Python system itself (used later to find the Python launcher).
* *(blank line)*
* `import pytest`: Brings in a popular testing framework used to verify that the code works as expected.
* *(blank line)*
* `from wiki2shards.io.atomic import atomic_open, atomic_write_bytes`: Loads two custom tools from the project's own code designed to safely write files without leaving them half-finished.
* *(blank line)*
* *(blank line)*
* `def test_atomic_write(tmp_path):`: Defines a new test function and gives it a temporary, disposable folder (`tmp_path`) to run tests inside safely.
* `    final = tmp_path / "out.bin"`: Creates a file path named "out.bin" inside that temporary folder.
* `    with pytest.raises(RuntimeError), atomic_open(final) as f:`: Tells the test to *expect* a `RuntimeError` crash to happen in the next block, and opens the file so we can start writing to it.
* `        f.write(b"partial")`: Writes the word "partial" into the file (the `b` means it is written as raw bytes, not standard text).
* `        raise RuntimeError("boom")`: Deliberately forces the program to crash with an error named "boom" right in the middle of writing.
* `    assert not final.exists()`: Checks to ensure the final "out.bin" file does not exist, proving the failed save didn't leave a half-written file behind.
* `    assert list(tmp_path.iterdir()) == []  # the tmp file is gone too`: Looks inside the temporary folder to guarantee no hidden leftover junk files were left behind either.
* *(blank line)*
* `    atomic_write_bytes(final, b"v1")`: Successfully writes the text "v1" into the file using the safe saving tool.
* `    with pytest.raises(RuntimeError), atomic_open(final) as f:`: Again, sets up a block that expects a crash, and opens the file to try and overwrite it.
* `        f.write(b"v2, half written")`: Starts writing new data over the old file.
* `        raise RuntimeError("boom")`: Deliberately forces another crash before the save can finish.
* `    assert final.read_bytes() == b"v1"  # the old content survives a failed rewrite`: Reads the file and verifies it still perfectly contains the original "v1" data, untouched by the failed overwrite attempt.
* *(blank line)*
* `    # A hard kill mid-write runs no cleanup at all; still nothing appears under the final name.`: A comment explaining that the next test will brutally kill the program without letting it clean up its mess.
* `    killed = tmp_path / "killed.bin"`: Creates a new file path named "killed.bin" in the temporary folder.
* `    code = (`: Begins a block of text that contains a miniature Python script.
* `        "import os, sys\nfrom wiki2shards.io.atomic import atomic_open\n"`: The mini-script loads necessary system tools and the custom safe-open tool.
* `        "with atomic_open(sys.argv[1]) as f:\n    f.write(b'x' * 4096)\n    f.flush()\n"`: The mini-script opens the file, writes 4,096 "x" characters, and forces the computer to physically push that data to the hard drive.
* `        "    os._exit(137)\n"`: The mini-script instantly and violently kills itself (using error code 137), completely bypassing any normal file-closing or cleanup steps.
* `    )`: Closes the block of text containing the mini-script.
* `    proc = subprocess.run([sys.executable, "-c", code, str(killed)])`: Uses the system to launch a completely separate background process to run that mini-script, pointing it at the "killed.bin" file path.
* `    assert proc.returncode == 137`: Verifies that the background process actually died violently with the 137 error code.
* `    assert not killed.exists()`: Checks to ensure that even though the program was killed brutally mid-save, the "killed.bin" file was never officially created, preventing data corruption.

#### `tests/unit/test_io_hashing.py`

Here is a line-by-line breakdown of what this Python test code is doing in plain English:

**`import pytest`**
Brings in a popular testing tool called `pytest`, which is used to check if the code is working correctly.

**`from wiki2shards.io.hashing import (`**
Starts importing specific tools (functions) from a file inside the project called `hashing.py`.

**`    canonical_json,`**
Imports a function that formats data into a standardized (canonical) JSON format so it always looks exactly the same.

**`    canonical_json_pretty,`**
Imports a function that formats data into a standardized JSON format, but spaced out so it's easy for humans to read.

**`    doc_uid,`**
Imports a function that creates a unique ID number for a document.

**`    parse_uid_hex,`**
Imports a function that translates a text-based (hexadecimal) ID back into a regular number.

**`    uid_hex,`**
Imports a function that translates a regular ID number into a text-based hexadecimal format (numbers 0-9 and letters a-f).

**`)`**
Closes the list of things being imported.

**`def test_canonical_json():`**
Defines a new test function meant to check if the JSON formatting tools work properly.

**`    a = {"b": [1, 2, {"z": None, "a": True}], "a": "é", "c": {"y": 1, "x": 2}}`**
Creates a bundle of data (a dictionary) named `a` containing mixed information like numbers, text with a special character (`é`), true/false values, and blank values (`None`).

**`    b = {"c": {"x": 2, "y": 1}, "a": "é", "b": [1, 2, {"a": True, "z": None}]}`**
Creates a second bundle of data named `b` with the exact same information as `a`, but written in a completely different order.

**`    assert canonical_json(a) == canonical_json(b)`**
Checks (asserts) that if you run both `a` and `b` through the formatting tool, the result is completely identical, proving the tool automatically sorts out scrambled orders.

**`    assert canonical_json(a) == '{"a":"é","b":[1,2,{"a":true,"z":null}],"c":{"x":2,"y":1}}'.encode()`**
Checks that the formatted version of `a` exactly matches this specific, correctly sorted text string that has been converted into computer bytes (which is what `.encode()` does).

**`    assert canonical_json_pretty(a).endswith(b"}\n")`**
Checks that the "pretty" human-readable version of the data ends properly with a closing bracket `}` and an invisible "new line" character `\n`.

**`    assert canonical_json_pretty(a) == canonical_json_pretty(b)`**
Checks that the "pretty" human-readable versions of `a` and `b` are also perfectly identical despite starting in different orders.

**`    for bad in (1.5, {"x": [0.0]}, {"x": float("nan")}, {1: "non-string key"}, {"s": {1, 2}}):`**
Starts a loop that goes through a list of invalid data types (like decimal numbers, "Not a Number" values, or keys that aren't text), temporarily calling each one `bad`.

**`        with pytest.raises(TypeError):`**
Tells the testing tool to expect the very next line of code to crash and throw a specific error called a `TypeError`.

**`            canonical_json(bad)`**
Tries to run the bad data through the formatting tool. If it successfully throws the expected error, the test passes. If it doesn't crash, the test fails.

**`def test_doc_uid_golden():`**
Defines a second test function to check if the unique document ID generators are working correctly.

**`    # Pinned values: a built-in hash() implementation would fail here in-process.`**
This is just a note for developers reading the code. It explains that they are testing against specific, hard-coded results because Python's default randomization would make the IDs change every time the test runs.

**`    assert uid_hex(doc_uid("enwiki", 12)) == "6b94dc5bf1a07c42"`**
Checks that generating a unique ID for document number 12 in the English Wiki ("enwiki") and converting it to text always results in exactly "6b94dc5bf1a07c42".

**`    assert uid_hex(doc_uid("simplewiki", 12)) == "0e055fda4b07921c"`**
Checks that doing the same thing for document number 12 in a different wiki ("simplewiki") results in a completely different, but expected, specific text string.

**`    assert parse_uid_hex(uid_hex(2**64 - 1)) == 2**64 - 1`**
Checks that if you take the maximum possible 64-bit number (`2**64 - 1`), convert it to text, and then immediately translate it back to a number, you get the exact same number you started with (proving the translation works perfectly in reverse).

### Validation

Here is the line-by-line explanation of what these commands do:

1. **`uv run python scripts/make_fixture_dump.py --out data/fixture_dump`**
This uses `uv` (a fast Python runner) to execute a script named `make_fixture_dump.py`. The `--out` part tells the script to save the files it generates into a folder called `data/fixture_dump`.
2. **`uv run python scripts/make_fixture_dump.py --out data/fixture_dump_again`**
This runs the exact same script again, but this time it saves the generated files into a different folder called `data/fixture_dump_again`.
3. **`diff -r data/fixture_dump data/fixture_dump_again && echo 'fixture dump: byte-identical'`**
This compares the contents of the two folders (`diff -r`) to see if there are any differences. The `&&` means "if they match perfectly, do the next thing." If they are exactly identical, it prints the message "fixture dump: byte-identical" to the screen. (This is testing to ensure the script produces reliable, identical results every time it runs).
4. **`(cd data/fixture_dump/fixturewiki/20260901 && shasum -a 256 *)`**
This temporarily goes into a specific folder (`cd ...`) and generates a unique digital fingerprint called a SHA-256 hash (`shasum -a 256`) for every file (`*`) in that folder. The parentheses ensure that once this is done, you are automatically put back in your original starting folder.
5. **`uv run pytest -q tests/unit/test_io_atomic.py tests/unit/test_io_hashing.py`**
This runs automated software tests on two specific files (`test_io_atomic.py` and `test_io_hashing.py`) using a Python testing tool called `pytest`. The `-q` stands for "quiet," meaning it will output a simplified, less cluttered summary of the test results.
6. **`uv run ruff check . && uv run ruff format --check . && uv run mypy src`**
This runs three different code-quality checks in a row, only moving to the next one if the previous one passes:
* `ruff check .` scans the code for bugs and bad practices (linting).
* `ruff format --check .` checks if the code is styled and spaced cleanly without actually altering it.
* `mypy src` checks the main code folder (`src`) to ensure all variables and data types are used correctly (type checking).

```sh
uv run python scripts/make_fixture_dump.py --out data/fixture_dump
data/fixture_dump/fixturewiki/20260901
```
---
```sh
uv run python scripts/make_fixture_dump.py --out data/fixture_dump_again
data/fixture_dump/fixturewiki/20260901
```
---
```sh
diff -r data/fixture_dump data/fixture_dump_again && echo 'fixture dump: byte-identical'
fixture dump: byte-identical
```
---
```sh
(cd data/fixture_dump/fixturewiki/20260901 && shasum -a 256 *)

2c11b79c7600e0060103ba5f19bb324234a3eaa4626fda030971a0e4a6148edc  fixture_expected.json
da6b5bdbe15a86f1600cef2b8b8118230e0d40f0cbb4ce5b57dd37424474834a  fixturewiki-20260901-pages-articles-multistream-index.txt.bz2
245358bf9f73289ed2f9266f00c47c68bf16eaef1773ea73a26a18c60983a77f  fixturewiki-20260901-pages-articles-multistream.xml.bz2
5ff7e35574c64508a57f348ca9a1bda75cb9e4d522fe85c26d2a65dec84223ca  fixturewiki-20260901-sha1sums.txt
```
---
```sh
uv run pytest -q tests/unit/test_io_atomic.py tests/unit/test_io_hashing.py
...                                                                                                                                                                                                                [100%]
3 passed in 0.04s
```
---
```sh
uv run ruff check . && uv run ruff format --check . && uv run mypy src
All checks passed!
19 files already formatted
Success: no issues found in 12 source files
```
---

## Phase 1: Contracts, then freeze (M1)

`src/wiki2shards/io/binfmt.py`

### Step 1.1: .bin/.idx, the dtype guard and shard checkers

```python
"""The .bin/.idx shard format (SPEC 1.4). Frozen at contracts-v1.

.bin: n_rows x row_len tokens, row-major, <u2 or <u4, no header, no trailing bytes.
.idx: 128-byte little-endian header, then row_seg_ptr <u8[n_rows+1], seg_doc_uid <u8[n_seg],
      seg_start <u4[n_seg], seg_len <u4[n_seg], seg_piece <u4[n_seg].
"""

```

This is a multi-line comment (docstring) explaining what this code does. It defines a specific way to store data in two files: a `.bin` file containing raw data tokens (words/pieces of text converted to numbers) and an `.idx` file containing an index (like a table of contents) to navigate the `.bin` file.

```python
from __future__ import annotations

```

Allows the code to use modern Python type hinting features before they are fully loaded by the interpreter.

```python
import struct
from dataclasses import dataclass
from pathlib import Path

import numpy as np
from numpy.typing import NDArray

```

These lines import tools the code needs:

* `struct`: Translates Python values into raw computer bytes (and vice versa).
* `dataclass`: A shortcut for creating classes that primarily store data.
* `Path`: A tool for working with file paths (like `C:/folder/file.txt`).
* `numpy as np`: A powerful math and array library, abbreviated as `np`.
* `NDArray`: A type hint for Numpy arrays.

```python
IDX_MAGIC = b"W2S-IDX\0"
IDX_VERSION_MAJOR = 1
IDX_VERSION_MINOR = 0

```

Defines constants used to identify a valid `.idx` file. The "magic" string is a unique byte signature at the start of the file ensuring it's the correct file type. It also sets the format version to 1.0.

```python
IDX_STRUCT = struct.Struct("<8sHHBBBBIIQQQQ32s32s8s")
HEADER_SIZE = 128
assert IDX_STRUCT.size == HEADER_SIZE

```

Creates a blueprint (`IDX_STRUCT`) for exactly how the 128-byte header is laid out in memory (using specific byte sizes like `<8s` for an 8-byte string, `H` for 2-byte integers, `I` for 4-byte integers, etc.). It then strictly verifies that this blueprint equals exactly 128 bytes.

```python
PACK_MODES = {"concat": 0, "bfd": 1}
SPLITS = {"train": 0, "val": 1}

```

Creates lookup dictionaries for different settings: how data is packed (`concat` vs `bfd`) and whether the data is for AI training (`train`) or validation (`val`).

```python
U2 = np.dtype("<u2")
U4 = np.dtype("<u4")
U8 = np.dtype("<u8")

```

Sets up shortcuts for specific Numpy data types: 16-bit (`U2`), 32-bit (`U4`), and 64-bit (`U8`) unsigned integers. The `<` means "little-endian" (how the computer reads the bytes).

```python
class FormatError(Exception):
    """Base class: the bytes on disk do not match the format."""

class BadMagic(FormatError):
    pass

class UnsupportedVersion(FormatError):
    pass

class SizeMismatch(FormatError):
    pass

class DtypeRangeError(ValueError):
    """A token id does not fit the target dtype (astype would wrap silently)."""

```

Defines custom error messages so if the file is broken, the program can crash with a highly specific explanation (e.g., `BadMagic` if the file signature is wrong, `SizeMismatch` if the file is cut off).

```python
def dtype_for_vocab(vocab_size: int) -> np.dtype[np.unsignedinteger]:
    return U2 if vocab_size <= 65_536 else U4

```

A function that looks at the total number of unique words/tokens (`vocab_size`). If there are 65,536 or fewer, it uses 2 bytes (`U2`) to save space. Otherwise, it uses 4 bytes (`U4`).

```python
def checked_cast(
    ids: NDArray[np.integer], dtype: np.dtype[np.unsignedinteger]
) -> NDArray[np.unsignedinteger]:
    """Cast token ids, refusing values astype would wrap (70000 -> 4464 for <u2)."""
    if ids.size:
        lo, hi = int(ids.min()), int(ids.max())
        info = np.iinfo(dtype)
        if lo < 0 or hi > info.max:
            raise DtypeRangeError(f"token ids span [{lo}, {hi}], outside {dtype.str} range")
    return ids.astype(dtype)

```

A safety function. When converting data types, it checks the lowest (`lo`) and highest (`hi`) numbers in the array. If any number is too big to fit in the new data type, it raises an error instead of silently corrupting the data.

```python
@dataclass(frozen=True)
class IdxHeader:

```

Creates a data container for the 128-byte header. `frozen=True` means once this header is created, its values cannot be accidentally changed.

```python
    itemsize: int
    pack_mode: int
    split: int
    row_len: int
    shard_index: int
    n_rows: int
    n_segments: int
    n_pad_tokens: int
    first_global_row: int
    bin_sha256: bytes
    tokenizer_fp: bytes
    version_major: int = IDX_VERSION_MAJOR
    version_minor: int = IDX_VERSION_MINOR

```

Lists all the individual pieces of information stored in the header, such as how many rows exist (`n_rows`), the size of the data (`itemsize`), and digital fingerprints (`bin_sha256`) to verify the file hasn't been tampered with.

```python
    def pack(self) -> bytes:
        return IDX_STRUCT.pack(
            IDX_MAGIC,
            self.version_major,
            self.version_minor,
            self.itemsize,
            self.pack_mode,
            self.split,
            0,
            self.row_len,
            self.shard_index,
            self.n_rows,
            self.n_segments,
            self.n_pad_tokens,
            self.first_global_row,
            self.bin_sha256,
            self.tokenizer_fp,
            b"\0" * 8,
        )

```

A method that takes all the Python variables listed above and squashes them into exactly 128 bytes of raw computer data using the `IDX_STRUCT` blueprint defined earlier. The `b"\0" * 8` is empty padding at the end.

```python
    @classmethod
    def unpack(cls, raw: bytes) -> IdxHeader:
        if len(raw) < HEADER_SIZE:
            raise SizeMismatch(f".idx shorter than its {HEADER_SIZE}-byte header")

```

A method that does the exact opposite: it takes raw computer bytes and turns them back into a Python `IdxHeader` object. First, it ensures the raw data is at least 128 bytes long.

```python
        (
            magic, vmaj, vmin, itemsize, pack_mode, split, _r, row_len,
            shard_index, n_rows, n_seg, n_pad, first_row, bin_sha, tok_fp, _r2,
        ) = IDX_STRUCT.unpack(raw[:HEADER_SIZE])

```

Slices out the first 128 bytes and extracts all the individual variables based on the blueprint.

```python
        if magic != IDX_MAGIC:
            raise BadMagic(f"bad .idx magic {magic!r}")
        if vmaj != IDX_VERSION_MAJOR:
            raise UnsupportedVersion(f".idx major version {vmaj}, reader supports {IDX_VERSION_MAJOR}")
        if itemsize not in (2, 4) or pack_mode not in (0, 1) or split not in (0, 1):
            raise FormatError(f"bad header enum: itemsize={itemsize} mode={pack_mode} split={split}")

```

Performs safety checks on the unpacked data: ensures the file signature matches, the version is supported, and the format settings are valid (e.g., itemsize must be 2 or 4).

```python
        return cls(itemsize, pack_mode, split, row_len, shard_index, n_rows, n_seg, n_pad, first_row, bin_sha, tok_fp, vmaj, vmin)

```

Returns the fully built `IdxHeader` object populated with the extracted values.

```python
@dataclass(frozen=True)
class Segments:
    row_seg_ptr: NDArray[np.uint64]
    doc_uid: NDArray[np.uint64]
    start: NDArray[np.uint32]
    length: NDArray[np.uint32]
    piece: NDArray[np.uint32]

```

Creates another data container that holds Numpy arrays representing "segments" (chunks of text). It tracks where chunks start, how long they are, and which document they belong to.

```python
    @property
    def n_rows(self) -> int:
        return int(self.row_seg_ptr.shape[0]) - 1

    @property
    def n_segments(self) -> int:
        return int(self.doc_uid.shape[0])

```

Helper properties that calculate the total number of rows and segments by checking the length of the arrays.

```python
def _counts(row_seg_ptr: NDArray[np.uint64]) -> NDArray[np.int64]:
    """Segments per row, from the CSR pointer."""
    ptr = row_seg_ptr.astype(np.int64)
    return ptr[1:] - ptr[:-1]

```

A math trick that takes an array of pointers (which show where things begin) and subtracts each pointer from the next one to figure out how many items exist in each row.

```python
def idx_size(n_rows: int, n_segments: int) -> int:
    return HEADER_SIZE + 8 * (n_rows + 1) + 20 * n_segments

```

A math formula calculating exactly how many bytes the `.idx` file should be, based on the number of rows and segments.

```python
def encode_idx(header: IdxHeader, seg: Segments) -> bytes:
    if seg.n_rows != header.n_rows or seg.n_segments != header.n_segments:
        raise ValueError("header counts disagree with the segment arrays")

```

A function to convert the entire index (header + segments) into raw bytes. It first checks that the header's counts match the actual data arrays.

```python
    return b"".join([
        header.pack(),
        seg.row_seg_ptr.astype(U8).tobytes(),
        seg.doc_uid.astype(U8).tobytes(),
        seg.start.astype(U4).tobytes(),
        seg.length.astype(U4).tobytes(),
        seg.piece.astype(U4).tobytes(),
    ])

```

Combines the packed header and converts all the Numpy arrays into raw byte strings, squishing them all together into one long string of bytes to be saved to disk.

```python
def decode_idx(raw: bytes) -> tuple[IdxHeader, Segments]:
    header = IdxHeader.unpack(raw)
    want = idx_size(header.n_rows, header.n_segments)
    if len(raw) != want:
        raise SizeMismatch(f".idx is {len(raw)} bytes, header implies {want}")

```

A function to convert a raw byte string back into a Header and Segments. It unpacks the header, calculates how long the file *should* be, and throws an error if the actual file size is different.

```python
    r, s = header.n_rows, header.n_segments
    off = HEADER_SIZE

```

Sets up variables `r` (rows) and `s` (segments), and creates an offset counter (`off`) starting at 128 (skipping the header).

```python
    def take(dtype: np.dtype[np.unsignedinteger], count: int) -> NDArray[np.unsignedinteger]:
        nonlocal off
        arr = np.frombuffer(raw, dtype=dtype, count=count, offset=off)
        off += dtype.itemsize * count
        return arr

```

A helper function that bites off chunks of the raw byte string. It reads `count` amount of numbers of type `dtype`, moving the `off` (offset) tracker forward so the next read starts in the correct place.

```python
    seg = Segments(take(U8, r + 1), take(U8, s), take(U4, s), take(U4, s), take(U4, s))
    return header, seg

```

Uses the `take` function five times to extract all the Numpy arrays, packages them into a `Segments` object, and returns both the header and the segments.

```python
def read_idx(path: Path | str) -> tuple[IdxHeader, Segments]:
    return decode_idx(Path(path).read_bytes())

```

A convenience function that takes a file path, opens it, reads all the bytes, and passes them to `decode_idx`.

```python
def read_idx_header(path: Path | str) -> IdxHeader:
    with open(path, "rb") as f:
        return IdxHeader.unpack(f.read(HEADER_SIZE))

```

Another convenience function that opens a file, but *only* reads the first 128 bytes to grab the header, saving memory if you don't need the massive segment arrays yet.

```python
def open_bin(
    path: Path | str, itemsize: int, n_rows: int, row_len: int
) -> NDArray[np.unsignedinteger]:
    dtype = U2 if itemsize == 2 else U4
    if n_rows == 0:
        return np.zeros((0, row_len), dtype=dtype)
    return np.memmap(path, dtype=dtype, mode="r", shape=(n_rows, row_len))

```

A function to read the `.bin` data file. Instead of loading potentially gigabytes of data into RAM at once, it uses `np.memmap`. This creates a virtual Numpy array that reads data directly off the hard drive only when requested. (If the file has 0 rows, it just returns an empty array).

```python
def check_segments(
    seg: Segments,
    *,
    row_len: int,
    pack_mode: int,
    allow_partial_last_row: bool = False,
) -> list[str]:
    """Structural invariants of SPEC 1.4 that need only the .idx arrays."""
    errors: list[str] = []

```

A massive validation function that checks the `Segments` arrays for mathematical and structural errors, building a list of error messages.

```python
    ptr = seg.row_seg_ptr.astype(np.int64)
    n_rows, n_seg = seg.n_rows, seg.n_segments
    if ptr[0] != 0 or ptr[-1] != n_seg:
        return [f"row_seg_ptr must run from 0 to {n_seg}, got {ptr[0]}..{ptr[-1]}"]

```

Checks that the pointers indicating where rows begin start perfectly at 0 and end perfectly at the total number of segments.

```python
    counts = ptr[1:] - ptr[:-1]
    if n_rows and counts.min() < 1:
        return ["row_seg_ptr is not strictly increasing (a row without segments)"]
    if n_seg == 0:
        return errors

```

Calculates segments per row. Checks if any row is totally empty (has 0 segments). If the whole file has 0 segments, it stops checking here and returns no errors.

```python
    start = seg.start.astype(np.int64)
    length = seg.length.astype(np.int64)
    if length.min() < 1:
        errors.append("segment with zero length")

```

Ensures every segment contains at least 1 token (no useless zero-length segments).

```python
    row_of = np.repeat(np.arange(n_rows), counts)
    first = ptr[:-1]
    if np.any(start[first] != 0):
        errors.append("a row's first segment does not start at position 0")

```

Figures out which row every segment belongs to. It then verifies that the very first segment in any row always starts at position index 0.

```python
    contiguous = np.ones(n_seg, dtype=bool)
    contiguous[first] = False
    prev_end = np.concatenate([[0], start[:-1] + length[:-1]])
    if np.any(start[contiguous] != prev_end[contiguous]):
        errors.append("segments within a row are not sorted and contiguous")

```

Checks that segments within a row connect perfectly back-to-back without gaps or overlapping (e.g., if segment 1 ends at position 5, segment 2 must start at position 5).

```python
    used = np.bincount(row_of, weights=length, minlength=n_rows).astype(np.int64)
    if np.any(used > row_len):
        errors.append("a row holds more than row_len tokens")

```

Adds up the length of all segments in a row and checks that no row overflows the maximum allowed width (`row_len`).

```python
    if pack_mode == PACK_MODES["concat"]:
        full = used == row_len
        if allow_partial_last_row and n_rows:
            full[-1] = True
        if not np.all(full):
            errors.append("concat row whose segments do not sum to row_len")
    return errors

```

If the file is using "concat" mode (where data is tightly packed together), it verifies that every single row is filled exactly to its maximum capacity. It can optionally forgive the very last row for being partially empty. It finally returns the list of any errors found.

```python
def check_tokens(
    tokens: NDArray[np.unsignedinteger],
    seg: Segments,
    *,
    vocab_size: int,
    pad_id: int,
) -> list[str]:
    """Token-level invariants: range, and pad_id after each row's last segment."""
    errors: list[str] = []

```

A second validation function that checks the actual raw tokens in the `.bin` file against the index rules.

```python
    if tokens.size and int(tokens.max()) >= vocab_size:
        errors.append(f"token id {int(tokens.max())} >= vocab_size {vocab_size}")

```

Ensures that no raw token number in the data is higher than the maximum allowed vocabulary size (e.g., if your dictionary has 50,000 words, you can't have a token ID of 50,001).

```python
    counts = _counts(seg.row_seg_ptr)
    row_of = np.repeat(np.arange(seg.n_rows), counts)
    used = np.bincount(row_of, weights=seg.length.astype(np.int64), minlength=seg.n_rows)

```

Recalculates how much space in each row is actually occupied by valid data segments.

```python
    row_len = tokens.shape[1] if tokens.ndim == 2 else 0
    pad_mask = np.arange(row_len)[None, :] >= used.astype(np.int64)[:, None]
    if np.any(tokens[pad_mask] != pad_id):
        errors.append("a padding position holds something other than pad_id")
    return errors

```

Calculates exactly which spots in the file are "empty" space at the ends of rows. It then verifies that every single empty spot contains the official "padding" token (e.g., a specific number like 0), and returns the list of errors.

### Step 1.2: The manifest

`src/wiki2shards/io/manifest.py`

This script defines the structure and rules for reading, writing, and validating a dataset's metadata file (called a "manifest"). Here is the line-by-line breakdown:

> Documentation and Imports

```python
"""dataset.manifest: schema, reader, writer (SPEC 1.4). Frozen at contracts-v1.

The manifest is canonical JSON written last; its presence commits the dataset and the SHA-256 of
its bytes is the dataset id (never stored inside it).
"""

```

* **Lines 1-5:** A multi-line comment (docstring) explaining what this file does. It states this is version 1.4 of the specification, the manifest is written in a strict JSON format, and the unique ID of the dataset is generated by hashing this file's contents.

```python
from __future__ import annotations

```

* **Line 7:** Allows Python to read type hints (like `list[SourceFile]`) without causing errors if the referenced types are defined later in the code.

```python
import json
from pathlib import Path
from typing import Any, Literal

from pydantic import BaseModel, ConfigDict, Field, ValidationError

from wiki2shards.io.atomic import atomic_write_bytes
from wiki2shards.io.hashing import canonical_json_pretty, sha256_hex

```

* **Line 9:** Imports the standard library tool for reading and writing JSON data.
* **Line 10:** Imports `Path` to easily handle file and folder paths.
* **Line 11:** Imports `Any` (allowing any data type) and `Literal` (restricting a value to specific exact choices) for type hinting.
* **Line 13:** Imports tools from the `pydantic` library, which is used to define strict data structures and validate that incoming data matches those structures.
* **Line 15:** Imports a custom function to safely write files so they don't get corrupted if the process is interrupted.
* **Line 16:** Imports custom functions to format JSON strictly consistently (`canonical_json_pretty`) and to calculate file hashes (`sha256_hex`).

> Constants and Error Handling

```python
MANIFEST_NAME = "dataset.manifest"
FORMAT = "wiki2shards"
FORMAT_VERSION = (1, 0)

```

* **Line 18:** Sets the standard filename for the manifest.
* **Line 19:** Sets the expected format name.
* **Line 20:** Sets the current version of the format as a tuple (Major 1, Minor 0).

```python
class ManifestError(Exception):
    pass

```

* **Lines 23-24:** Creates a custom error type specifically for manifest-related problems. `pass` means it does nothing special beyond acting as a standard error.

> Base Model Definition

```python
class _Model(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True, frozen=True)

```

* **Line 27:** Creates a foundational data model class based on Pydantic's `BaseModel`.
* **Line 28:** Configures all models using this base to reject any unlisted fields (`extra="forbid"`), strictly enforce data types (`strict=True`), and prevent changes after creation (`frozen=True`).

> Data Structure Definitions (Schemas)

```python
class BuildInfo(_Model):
    wiki2shards_version: str
    git_commit: str
    lockfile_sha256: str | None
    python: str
    unicode_version: str
    config_sha256: str

```

* **Lines 31-37:** Defines the `BuildInfo` section of the manifest. It requires strings detailing the software version, the exact Git code commit, the Python version, the text encoding version, and the settings hash. The lockfile hash can be a string or `None`.

```python
class SourceFile(_Model):
    name: str
    bytes: int
    sha1: str

```

* **Lines 40-43:** Defines a single downloaded source file, requiring its filename, size in bytes, and its SHA-1 hash for security.

```python
class SourceInfo(_Model):
    wiki: str
    dump_date: str
    files: list[SourceFile]

```

* **Lines 46-49:** Defines the `SourceInfo` section, tracking which Wikipedia version was used, the date of the dump, and a list of the `SourceFile` objects defined above.

```python
class TokenizerInfo(_Model):
    spec: str
    fingerprint: str
    vocab_size: int
    eos_id: int
    pad_id: int
    dtype: Literal["<u2", "<u4"]

```

* **Lines 52-58:** Defines settings for how text was chopped into tokens. It tracks the exact rules (`spec`, `fingerprint`), the total number of unique tokens (`vocab_size`), special token IDs (end-of-sequence and padding), and limits the data type (`dtype`) to specifically 16-bit or 32-bit unsigned integers.

```python
class PackingInfo(_Model):
    mode: Literal["concat", "bfd"]
    seq_len: int
    row_len: int
    doc_order: Literal["doc_uid_asc"]
    last_row: Literal["drop", "pad"]

```

* **Lines 61-66:** Defines how data was packed together. It forces `mode` to be one of two choices, sets integer lengths, forces document order to be ascending, and dictates whether the leftover last row should be deleted or padded with empty space.

```python
class SelectionInfo(_Model):
    policy: str
    soft_filters: list[str]
    dedup: str
    val_per_10k: int

```

* **Lines 69-73:** Details how data was filtered, listing the policy name, a list of applied filter names, the deduplication method used, and how many items per 10,000 are held back for testing/validation.

```python
class ShardEntry(_Model):
    index: int
    n_rows: int
    bin: str
    bin_bytes: int
    bin_sha256: str
    idx: str
    idx_bytes: int
    idx_sha256: str

```

* **Lines 76-84:** Defines a single "shard" (a broken-up piece of the dataset). It tracks the shard's sequence number, row count, file name for the raw data (`bin`), its size and hash, along with the file name, size, and hash for its corresponding index file (`idx`).

```python
class SplitInfo(_Model):
    n_rows: int
    n_tokens: int
    n_pad_tokens: int
    n_docs: int
    n_tokens_dropped_tail: int
    shards: list[ShardEntry]

```

* **Lines 87-93:** Defines a dataset split (like "train" or "validation"). It tallies up total rows, tokens, blank tokens, documents, discarded tokens, and holds a list of all the `ShardEntry` pieces that belong to this split.

```python
class SideFile(_Model):
    name: str
    bytes: int
    sha256: str

```

* **Lines 96-99:** Defines any extra supplementary files (like configurations), requiring a name, size, and hash.

> The Main Manifest Model

```python
class Manifest(_Model):
    format: Literal["wiki2shards"]
    format_version: list[int] = Field(min_length=2, max_length=2)
    name: str
    build: BuildInfo
    source: SourceInfo
    tokenizer: TokenizerInfo
    packing: PackingInfo
    selection: SelectionInfo
    splits: dict[Literal["train", "val"], SplitInfo]
    side_files: list[SideFile]

```

* **Line 102:** Starts the main `Manifest` structure that combines all the previous pieces.
* **Line 103:** Requires the format name to be exactly "wiki2shards".
* **Line 104:** Requires the version to be a list of exactly two numbers (e.g., `[1, 0]`).
* **Lines 105-112:** Embeds all the previously defined models (Build, Source, Tokenizer, etc.) as required sections. `splits` is required to be a dictionary with specific keys ("train" and "val"), mapping to the `SplitInfo` details.

```python
    def to_bytes(self) -> bytes:
        return canonical_json_pretty(self.model_dump(mode="python"))

```

* **Line 114:** Defines a function to convert the populated manifest model into raw bytes.
* **Line 115:** Extracts the data back into standard Python format, formats it into strict, pretty-printed JSON, and returns it.

> Helper and Processing Functions

```python
def _drop(data: Any, loc: tuple[int | str, ...]) -> None:
    for key in loc[:-1]:
        data = data[key]
    del data[loc[-1]]

```

* **Line 118:** Defines a helper function to delete a specific piece of data buried inside nested dictionaries.
* **Lines 119-120:** Loops through the path to find the exact nested location, stopping just before the final item.
* **Line 121:** Deletes the final item at that specific location.

```python
def parse_manifest(raw: bytes) -> Manifest:
    """Validate manifest bytes. Unknown keys are errors at minor 0 and ignored at higher minors."""

```

* **Lines 124-125:** Starts a function to take raw bytes and turn them into a valid `Manifest` object, with a comment explaining that newer file versions will ignore unrecognized fields rather than crashing.

```python
    try:
        data = json.loads(raw.decode("utf-8"))
    except (UnicodeDecodeError, json.JSONDecodeError) as exc:
        raise ManifestError(f"manifest is not valid UTF-8 JSON: {exc}") from exc

```

* **Lines 126-129:** Attempts to decode the raw bytes as standard text and read it as JSON. If it fails (due to bad text encoding or broken JSON syntax), it stops the program and raises a specific `ManifestError`.

```python
    if not isinstance(data, dict) or data.get("format") != FORMAT:
        raise ManifestError("not a wiki2shards manifest")

```

* **Lines 130-131:** Checks if the loaded JSON is a standard dictionary (object) and if its "format" key matches "wiki2shards". If not, it throws an error.

```python
    version = data.get("format_version")
    if not (isinstance(version, list) and len(version) == 2 and version[0] == FORMAT_VERSION[0]):
        raise ManifestError(f"unsupported format_version {version!r}")

```

* **Lines 132-134:** Extracts the version number. It verifies that the version is a 2-item list and that the major version number (the first number) matches the software's expected major version. If it's a mismatch, it throws an error.

```python
    try:
        return Manifest.model_validate(data)

```

* **Lines 135-136:** Tries to pass the raw dictionary into the strict `Manifest` Pydantic model. If it perfectly matches, it returns the validated object.

```python
    except ValidationError as exc:
        extras = [e["loc"] for e in exc.errors() if e["type"] == "extra_forbidden"]

```

* **Lines 137-138:** If validation fails, it catches the error. It checks if the failure was caused specifically by "extra" unknown fields being in the file.

```python
        if version[1] > FORMAT_VERSION[1] and len(extras) == len(exc.errors()):
            for loc in sorted(extras, key=len, reverse=True):
                _drop(data, loc)
            return Manifest.model_validate(data)

```

* **Lines 139-142:** If the file's minor version is newer than the software's, AND the *only* errors were these extra unexpected fields, it steps backwards through the file to delete those unknown extra fields, then tries validating it again.

```python
        raise ManifestError(f"manifest schema error: {exc}") from exc

```

* **Line 143:** If validation failed for any other reason (like missing required data or wrong data types), it throws a hard error.

```python
def read_manifest(dataset_dir: Path | str) -> tuple[Manifest, str]:
    """Return the manifest and the dataset id (sha256 of its bytes)."""
    raw = (Path(dataset_dir) / MANIFEST_NAME).read_bytes()

```

* **Lines 146-148:** Starts a function to read the manifest from a folder path. It combines the folder path with the filename and reads the raw file bytes from the hard drive.

```python
    manifest = parse_manifest(raw)

```

* **Line 149:** Passes those raw bytes to the `parse_manifest` function to validate and structure the data.

```python
    if manifest.format_version == list(FORMAT_VERSION) and manifest.to_bytes() != raw:
        raise ManifestError("manifest bytes are not in canonical form")

```

* **Lines 150-151:** Checks if the manifest is the exact same version as the software. If it is, but the raw file bytes don't exactly match the strictly formatted canonical bytes, it throws an error (meaning someone edited the JSON sloppily by hand).

```python
    return manifest, sha256_hex(raw)

```

* **Line 152:** Returns the final structured manifest object alongside the cryptographic hash (dataset ID) of the raw bytes.

```python
def write_manifest(dataset_dir: Path | str, manifest: Manifest) -> str:
    raw = manifest.to_bytes()
    atomic_write_bytes(Path(dataset_dir) / MANIFEST_NAME, raw)
    return sha256_hex(raw)

```

* **Lines 155-158:** Starts a function to save a manifest to a folder. It converts the manifest object back into raw strictly-formatted bytes. It then safely writes those bytes to the `dataset.manifest` file in the given folder, and finally returns the cryptographic hash of what it just wrote.

### Step 1.3: Inter-stage records

`src/wiki2shards/io/records.py`

```python
"""JSONL records passed between stages: extracted pages and cleaned documents.

```

This is the start of a multi-line comment (called a docstring) that explains the purpose of this file: it handles "JSONL" (JSON Lines) data that is being passed between different steps of a program.

```python
Lines are canonical JSON. Readers split on b"\\n" only: raw U+2028 is legal inside a JSON string,

```

This continues the comment, explaining a technical detail: each line is perfectly formatted JSON data. It warns that when reading the file, the computer should only separate lines using a specific "newline" character (`\n`), because other invisible spacing characters might exist inside the text and shouldn't accidentally break the line in half.

```python
and str.splitlines() would break a record there.

```

This finishes the comment, noting that Python's standard tool for splitting lines (`str.splitlines()`) would make a mistake by splitting on those invisible spacing characters.

```python
"""

```

This closes the multi-line comment.

```python
from __future__ import annotations

```

This is a special Python instruction that makes it easier to write "type hints" (labels that tell the programmer what kind of data to expect, like text or numbers) without causing errors in older versions of Python.

```python
import json

```

This brings in Python's built-in tool for reading and writing JSON (a popular, easy-to-read format for storing data).

```python
from collections.abc import Iterable, Iterator, Mapping

```

This imports special labels used to describe different types of data collections, such as things you can loop over (`Iterable`, `Iterator`) or things that link keys to values like a dictionary (`Mapping`).

```python
from pathlib import Path

```

This brings in a tool called `Path` that makes it much easier to work with file names and folders on your computer.

```python
from typing import Any, TypedDict

```

This brings in tools to help label data: `Any` means "this could be any type of data," and `TypedDict` lets the programmer create a strict blueprint for a dictionary.

```python
from wiki2shards.io.atomic import atomic_open

```

This imports a custom function called `atomic_open` from another part of this project. It is used to safely open and save files so that if the computer crashes halfway through saving, the file doesn't get corrupted.

```python
from wiki2shards.io.hashing import canonical_json

```

This imports another custom function, `canonical_json`, which converts data into a very strict, standardized JSON format so it always looks exactly the same.

```python
class ExtractedPage(TypedDict):

```

This creates a blueprint (a `TypedDict`) named `ExtractedPage`. It defines exactly what information must be included whenever the program handles a page extracted from a wiki.

```python
    page_id: int

```

The extracted page must have a `page_id`, which must be a whole number (an integer).

```python
    rev_id: int

```

The extracted page must have a `rev_id` (revision ID), which must also be a whole number.

```python
    title: str

```

The extracted page must have a `title`, which must be text (a string).

```python
    wikitext: str

```

The extracted page must have `wikitext` (the raw text from the wiki), which must be text.

```python
class CleanedDoc(TypedDict):

```

This creates another blueprint named `CleanedDoc` for documents that have been processed and cleaned up.

```python
    doc_uid: str  # 16 hex digits, see hashing.uid_hex

```

The cleaned document needs a unique ID (`doc_uid`) made of text. The comment next to it notes this will be a 16-character code.

```python
    page_id: int

```

It must have a `page_id` as a whole number.

```python
    rev_id: int

```

It must have a `rev_id` as a whole number.

```python
    title: str

```

It must have a `title` made of text.

```python
    text: str

```

It must have the cleaned `text` of the document, made of text.

```python
    n_chars: int

```

It must have a count of the characters (`n_chars`), which is a whole number.

```python
    text_sha256: str

```

It must have a special digital fingerprint of the text (`text_sha256`) to prove it hasn't been tampered with, stored as text.

```python
    verdicts: dict[str, bool]

```

It must have `verdicts` (like a checklist of rules the document passed or failed), which is a dictionary linking text names to True/False values (booleans).

```python
def write_jsonl(path: Path | str, records: Iterable[Mapping[str, Any]]) -> None:

```

This defines a new function named `write_jsonl`. It takes two inputs: a file path (where to save the file) and a list of records (the data to save). The `-> None` part just means this function performs an action but doesn't hand any data back to you when it finishes.

```python
    with atomic_open(path) as f:

```

This safely opens the file at the given path so it can be written to, and temporarily names the open file `f`.

```python
        for rec in records:

```

This starts a loop that will look at every single record (piece of data) in the list of records provided.

```python
            f.write(canonical_json(dict(rec)) + b"\n")

```

For each record, this line turns it into a standard dictionary, converts it to the strict JSON format, adds a "newline" character (like hitting the Enter key) to the end, and writes that whole line into the file.

```python
def iter_jsonl(path: Path | str) -> Iterator[dict[str, Any]]:

```

This defines a new function named `iter_jsonl`. It takes a file path as input and is designed to hand back one dictionary record at a time (an Iterator) so the computer doesn't get overwhelmed reading a massive file all at once.

```python
    data = Path(path).read_bytes()

```

This reads the entire contents of the file at the given path into the computer's memory as raw, basic computer bytes.

```python
    for line in data.split(b"\n"):

```

This chops that raw data into separate pieces every time it sees a "newline" character (`\n`), and starts a loop to look at each individual line.

```python
        if line:

```

This checks to make sure the line isn't completely empty (which often happens at the very end of a file).

```python
            yield json.loads(line)

```

If the line has data, this converts the JSON text back into a usable Python dictionary and pauses to hand it back to whatever part of the program asked for it, before looping around to the next line.

### Step 1.4: Config models

`src/wiki2shards/config.py`

Here is the line-by-line explanation of the provided Python code:

> Module Docstring and Imports

`"""Build and mixture configs: pydantic models loaded from YAML. Frozen at contracts-v1.`
Starts a multi-line comment explaining this file handles configuration models loaded from YAML files, locked to a specific version.

`extra="forbid" turns typos into errors; strict=True stops YAML 1.1 coercions (an unquoted`
Explains that strict settings are used so accidental typos are rejected instead of ignored.

`20260901 is an int, an unquoted \`no` is False) from being silently accepted.`
Gives examples of how strict settings prevent strings from accidentally being converted to numbers or booleans.

`"""`
Ends the multi-line comment.

`from __future__ import annotations`
Allows the code to use modern Python type hints (like referring to a class before it is fully defined).

`from pathlib import Path`
Imports the `Path` tool to make working with file and folder paths easier.

`from typing import Any, Literal`
Imports `Any` (allowing any data type) and `Literal` (restricting a value to specific exact choices) for type hinting.

`import yaml`
Imports the `yaml` library to read YAML configuration files.

`from pydantic import BaseModel, ConfigDict, Field, model_validator`
Imports tools from the `pydantic` library used to build, validate, and restrict the configuration data models.

`from wiki2shards.io.hashing import canonical_json, sha256_file, sha256_hex`
Imports custom hashing functions from another part of the project to help generate unique fingerprints for files and data.

`MIB = 1 << 20`
Defines a constant named `MIB` equal to 1,048,576 (which is 1 Megabyte in binary bytes), calculated by shifting the number 1 left by 20 bits.

> Base Classes and Exceptions

`class ConfigError(Exception):`
Creates a custom error type called `ConfigError` specifically for when something goes wrong with the configuration.

`    pass`
Means this custom error doesn't need any special behavior beyond acting as a standard error.

`class _Strict(BaseModel):`
Creates a base configuration model called `_Strict` that inherits from Pydantic's `BaseModel`.

`    model_config = ConfigDict(extra="forbid", strict=True, frozen=True)`
Configures this base model to reject unknown fields (`extra="forbid"`), strictly enforce data types (`strict=True`), and prevent changes after creation (`frozen=True`).

> Configuration Data Models

`class SourceConfig(_Strict):`
Defines the `SourceConfig` model (inheriting the strict rules) to handle where the Wikipedia data comes from.

`    wiki: str = Field(pattern=r"^[a-z0-9_]+$")`
Requires a `wiki` string that must contain only lowercase letters, numbers, and underscores.

`    date: str = Field(pattern=r"^\d{8}$")`
Requires a `date` string that must be exactly 8 digits long (e.g., "20261010").

`    # Base URL (https://...), or a local directory laid out as <mirror>/<wiki>/<date>/`
A comment explaining what the `mirror` field should look like.

`    mirror: str = "[https://dumps.wikimedia.org](https://dumps.wikimedia.org)"`
Defines `mirror` with a default web address where Wikipedia dumps are downloaded from.

`class ExtractConfig(_Strict):`
Defines the `ExtractConfig` model for settings related to extracting data.

`    unit_mb: int = Field(default=32, ge=1)`
Defines `unit_mb` as an integer with a default of 32, which must be greater than or equal to (`ge`) 1.

`    unit_bytes: int | None = Field(default=None, ge=1)  # overrides unit_mb; for tiny fixtures`
Defines an optional `unit_bytes` that defaults to nothing, but must be at least 1 if provided (used for testing small files).

`    @property`
Marks the next function as a property, allowing it to be accessed like an attribute (e.g., `config.target_bytes`).

`    def target_bytes(self) -> int:`
Defines the `target_bytes` property which calculates and returns an integer.

`        return self.unit_bytes if self.unit_bytes is not None else self.unit_mb * MIB`
Returns `unit_bytes` if it was provided; otherwise, it calculates the bytes by multiplying `unit_mb` by 1 Megabyte (`MIB`).

`class SoftFilterConfig(_Strict):`
Defines the `SoftFilterConfig` model for rules that flag or filter out certain Wikipedia pages.

`    min_words: int | None = Field(default=50, ge=0)`
Requires pages to have at least 50 words by default, and must not be a negative number.

`    max_symbol_pct: int | None = Field(default=30, ge=0, le=100)`
Sets a rule where pages can't have more than 30% symbols by default, restricted between 0 and 100.

`    disambiguation: bool = True`
A true/false flag (defaulting to True) to handle disambiguation (multiple-meaning) pages.

`    list_page: bool = False`
A true/false flag (defaulting to False) to handle list pages.

`    def enabled(self) -> list[str]:`
Defines a method that returns a list of text strings representing which filters are turned on.

`        on = {`
Starts a dictionary called `on` to map filter names to whether they are active.

`            "min_words": self.min_words is not None,`
Checks if `min_words` has a value.

`            "max_symbol_pct": self.max_symbol_pct is not None,`
Checks if `max_symbol_pct` has a value.

`            "disambiguation": self.disambiguation,`
Checks if the disambiguation flag is True.

`            "list_page": self.list_page,`
Checks if the list_page flag is True.

`        }`
Closes the dictionary.

`        return sorted(name for name, flag in on.items() if flag)`
Loops through the dictionary and returns an alphabetically sorted list of the filter names that are currently active.

`class CleanConfig(_Strict):`
Defines the `CleanConfig` model for settings used to clean up the Wikipedia text.

`    drop_sections: list[str] = [`
Starts a list of Wikipedia section headers that should be removed from the text.

`        "References",`
Removes the "References" section.

`        "Notes",`
Removes the "Notes" section.

`        "External links",`
Removes the "External links" section.

`        "See also",`
Removes the "See also" section.

`        "Further reading",`
Removes the "Further reading" section.

`        "Bibliography",`
Removes the "Bibliography" section.

`        "Sources",`
Removes the "Sources" section.

`    ]`
Closes the list of sections to drop.

`    soft_filters: SoftFilterConfig = SoftFilterConfig()`
Includes the `SoftFilterConfig` settings inside this clean configuration, using its defaults.

`    disambiguation_templates: list[str] = ["disambiguation", "disambig", "dab", "hndis", "geodis"]`
Provides a list of common Wikipedia template names used to identify disambiguation pages.

`    language_codes_file: str = "configs/language_codes.txt"`
Specifies the default file path to find a list of valid language codes.

`class SelectConfig(_Strict):`
Defines the `SelectConfig` model to choose which cleaned pages are kept for the final dataset.

`    policy: str = Field(default="default", pattern=r"^[A-Za-z0-9_-]+$")`
Requires a policy name (defaulting to "default") containing only letters, numbers, underscores, or hyphens.

`    apply: list[str] = ["min_words", "disambiguation"]`
Specifies which soft filters (by default, "min_words" and "disambiguation") to apply when selecting pages.

`    dedup: Literal["exact"] = "exact"`
Sets the deduplication strategy to exactly "exact", meaning it will only remove exact duplicates.

`    val_per_10k: int = Field(default=50, ge=0, le=10_000)`
Sets how many pages out of every 10,000 should be held back for the validation set (default is 50).

`class PackConfig(_Strict):`
Defines the `PackConfig` model, detailing how the text data is packed together into files.

`    mode: Literal["concat", "bfd"] = "concat"`
Sets the packing mode to either "concat" (concatenation) or "bfd" (best-fit decreasing), defaulting to "concat".

`    seq_len: int = Field(ge=1)`
Requires a sequence length (the number of tokens in a batch) that must be at least 1.

`    last_row: Literal["drop", "pad"] | None = None  # concat default: drop; bfd always pads`
Allows defining what happens to the last row of data (drop it or pad it with empty space), defaulting to None.

`    shard_mb: int = Field(default=256, ge=1)`
Sets the target file size for data shards to 256 Megabytes, which must be at least 1.

`    shard_bytes: int | None = Field(default=None, ge=1)  # overrides shard_mb; for tiny fixtures`
An optional byte-level override for the shard size, mainly used for small tests.

`    bfd_window: int = Field(default=100_000, ge=1)`
Sets a window size of 100,000 for the "bfd" packing algorithm, meaning it looks at 100,000 items at a time.

`    @model_validator(mode="after")`
Tells Pydantic to run the following validation function after all fields are initially checked.

`    def _last_row(self) -> PackConfig:`
Defines a function to enforce specific rules about how the last row is handled.

`        if self.mode == "bfd" and self.last_row == "drop":`
Checks if the user set the mode to "bfd" but incorrectly told the system to "drop" the last row.

`            raise ValueError("last_row: drop applies to concat only; bfd always pads")`
Throws an error explaining that the "bfd" mode is not allowed to drop the last row.

`        return self`
Returns the validated object if no errors were found.

`    @property`
Creates a property so it can be accessed like an attribute.

`    def row_len(self) -> int:`
Defines a function to calculate the total row length.

`        return self.seq_len + 1`
Returns the sequence length plus 1 (often used in machine learning for the "next token" prediction).

`    @property`
Creates another property.

`    def effective_last_row(self) -> Literal["drop", "pad"]:`
Defines a function to determine the final, enforced rule for handling the last row.

`        if self.mode == "bfd":`
Checks if the mode is "bfd".

`            return "pad"`
If so, forces the last row behavior to be "pad".

`        return self.last_row or "drop"`
Otherwise, returns whatever the user set, or defaults to "drop".

`    @property`
Creates another property.

`    def target_shard_bytes(self) -> int:`
Defines a function to calculate the exact target size for data shards in bytes.

`        return self.shard_bytes if self.shard_bytes is not None else self.shard_mb * MIB`
Returns the explicit `shard_bytes` if provided, otherwise calculates it from `shard_mb`.

> Main Configuration Model

`class BuildConfig(_Strict):`
Defines the master `BuildConfig` model which pulls all the previous configuration pieces together.

`    name: str = Field(pattern=r"^[A-Za-z0-9._-]+$")`
Requires a build name using only letters, numbers, dots, underscores, or hyphens.

`    source: SourceConfig`
Requires a complete `SourceConfig` object.

`    workdir: str`
Requires a string defining the working directory path for temporary files.

`    output: str`
Requires a string defining the final output directory path.

`    workers: int = Field(default=1, ge=1)`
Specifies how many processor cores (workers) to use, defaulting to 1.

`    extract: ExtractConfig = ExtractConfig()`
Includes the `ExtractConfig` settings, using its defaults if not provided.

`    clean: CleanConfig = CleanConfig()`
Includes the `CleanConfig` settings, using its defaults if not provided.

`    select: SelectConfig = SelectConfig()`
Includes the `SelectConfig` settings, using its defaults if not provided.

`    tokenizer: str = Field(pattern=r"^(tiktoken|hf):.+$")`
Requires a tokenizer (text splitter) string that must start with either "tiktoken:" or "hf:".

`    pack: PackConfig`
Requires a complete `PackConfig` object.

`    @model_validator(mode="after")`
Tells Pydantic to run the following validation function after creating the build config.

`    def _cross_checks(self) -> BuildConfig:`
Defines a function to ensure the different configuration sections make sense together.

`        enabled = set(self.clean.soft_filters.enabled())`
Gets a mathematical set of all the soft filters that are actually turned on.

`        unknown = sorted(set(self.select.apply) - enabled)`
Finds any filters that the user asked to apply in `SelectConfig`, but which were not enabled in `CleanConfig`.

`        if unknown:`
If there are any unknown/un-enabled filters found...

`            raise ValueError(f"select.apply names filters that clean does not compute: {unknown}")`
Throws an error listing the problematic filters.

`        return self`
Returns the validated build configuration object.

`    def hashed_view(self) -> dict[str, Any]:`
Defines a method that creates a dictionary meant for calculating a unique fingerprint (hash) of the configuration.

`        """The output-affecting part of the config (PLAN A1).`
A comment explaining this method extracts only the settings that physically change the final output data.

`        Excludes workers, workdir, output and source.mirror, which never change output bytes,`
Explains that local paths and CPU counts are ignored because they don't change the data itself.

`        and replaces file paths with the SHA-256 of their content.`
Explains that local files referenced in the config are replaced by a hash of their contents to ensure reproducibility.

`        """`
Ends the comment block.

`        data = self.model_dump(mode="python")`
Converts the Pydantic model into a standard Python dictionary.

`        for key in ("workers", "workdir", "output"):`
Loops through the keys that shouldn't affect the data output.

`            data.pop(key)`
Removes those keys from the dictionary.

`        data["source"].pop("mirror")`
Removes the download mirror URL from the source dictionary.

`        data["clean"]["language_codes_file"] = sha256_file(self.clean.language_codes_file)`
Reads the language codes file and replaces the file path string with a cryptographic hash of the file's contents.

`        if self.tokenizer.startswith("hf:"):`
Checks if a Hugging Face ("hf:") local tokenizer file is being used.

`            data["tokenizer"] = "hf:sha256:" + sha256_file(self.tokenizer[3:])`
Replaces the tokenizer file path with a hash of that file's contents.

`        return data`
Returns the modified dictionary ready for hashing.

`    def config_sha256(self) -> str:`
Defines a method to generate the final configuration fingerprint.

`        return sha256_hex(canonical_json(self.hashed_view()))`
Converts the hashed view dictionary into a strict JSON string, hashes it, and returns the hexadecimal string.

> Mixture Models

`class MixSource(_Strict):`
Defines a model representing a single dataset source that will be mixed with others.

`    name: str = Field(pattern=r"^[A-Za-z0-9._-]+$")`
Requires a name for the source containing only letters, numbers, dots, underscores, or hyphens.

`    path: str`
Requires a string defining the file path to this dataset.

`    weight: int = Field(ge=1)`
Requires a mixing weight (how much of this data to use compared to others) that is at least 1.

`class MixConfig(_Strict):`
Defines a configuration model for mixing multiple datasets together.

`    seed: int = Field(ge=0, lt=1 << 64)`
Requires a random seed (for reproducibility) that must be positive and less than a 64-bit maximum.

`    split: Literal["train", "val"] = "train"`
Specifies whether this mix is for training data or validation data, defaulting to "train".

`    sources: list[MixSource] = Field(min_length=1)`
Requires a list of `MixSource` objects, and the list must contain at least one item.

`    @model_validator(mode="after")`
Tells Pydantic to run validation after all fields are set.

`    def _unique(self) -> MixConfig:`
Defines a function to ensure there are no duplicate sources in the mix.

`        names = [s.name for s in self.sources]`
Extracts a list of all the source names provided.

`        if len(set(names)) != len(names):`
Checks if converting the names to a set (which removes duplicates) makes the list shorter, meaning duplicates existed.

`            raise ValueError(f"duplicate source names: {names}")`
Throws an error if duplicates were found.

`        return self`
Returns the validated mix configuration object.

> File Loading Functions

`def _load_yaml(path: Path | str) -> Any:`
Defines a helper function to read a YAML file from a given path and return its contents.

`    try:`
Starts a block to attempt reading the file, catching errors if they happen.

`        with open(path, encoding="utf-8") as f:`
Opens the file safely in UTF-8 text mode, assigning it to the variable `f`.

`            return yaml.safe_load(f)`
Reads and parses the YAML file securely, returning the resulting Python dictionary.

`    except (OSError, yaml.YAMLError) as exc:`
Catches errors related to finding/reading the file (`OSError`) or parsing bad YAML formatting (`yaml.YAMLError`).

`        raise ConfigError(f"cannot read {path}: {exc}") from exc`
Wraps the system error into the custom `ConfigError` and throws it, preserving the original error trace.

`def load_build_config(path: Path | str) -> BuildConfig:`
Defines a function to read a YAML file specifically and convert it into a `BuildConfig` object.

`    try:`
Starts a block to attempt the conversion.

`        return BuildConfig.model_validate(_load_yaml(path))`
Loads the YAML file using the helper function, validates it against the `BuildConfig` model, and returns it.

`    except ValueError as exc:  # pydantic.ValidationError is a ValueError`
Catches validation errors (like missing fields or wrong data types) raised by Pydantic.

`        raise ConfigError(f"{path}: {exc}") from exc`
Wraps the validation error into the custom `ConfigError` and throws it.

`def load_mix_config(path: Path | str) -> MixConfig:`
Defines a function to read a YAML file specifically and convert it into a `MixConfig` object.

`    try:`
Starts a block to attempt the conversion.

`        return MixConfig.model_validate(_load_yaml(path))`
Loads the YAML file, validates it against the `MixConfig` model, and returns it.

`    except ValueError as exc:`
Catches validation errors raised by Pydantic.

`        raise ConfigError(f"{path}: {exc}") from exc`
Wraps the validation error into the custom `ConfigError` and throws it.

### Step 1.5: Tokenizer protocol and fingerprints

`src/wiki2shards/tokenize/base.py`

```python
"""Tokenizer protocol and fingerprints. Frozen at contracts-v1.

Spec strings: "tiktoken:<encoding name>" or "hf:<path to tokenizer.json>" (CWD-relative).
"""

```

These lines form a multi-line comment (called a docstring) that explains the purpose of the file. It states that this code defines the rules for text tokenizers and how to create unique identifiers (fingerprints) for them. It also notes that the code is stable ("Frozen") and shows the expected text formats for loading specific tokenizers.

```python
from __future__ import annotations

```

This enables a modern Python feature that allows type hints (labels telling the code what type of data to expect) to be read as plain text before everything in the file is fully loaded, preventing setup errors.

```python
import hashlib
import importlib
import json
import struct

```

These four lines bring in standard Python toolkits. `hashlib` is used to create secure, unique digital codes. `importlib` allows the program to load other Python files while it is running. `json` provides tools to read and write structured text data files. `struct` provides tools to pack basic data into raw binary formats.

```python
from collections.abc import Mapping, Sequence
from typing import Protocol, runtime_checkable

```

These lines import special tools used to describe how data should be shaped. `Mapping` represents dictionary-like structures (key-value pairs), and `Sequence` represents lists. `Protocol` is used to create a blueprint for how a piece of code should act, and `runtime_checkable` allows the program to verify if an object follows that blueprint while it is running.

```python
import numpy as np
from numpy.typing import NDArray

```

These lines import a popular math library called `numpy` (giving it the shorthand name `np`) and import a specific type label `NDArray`, which represents a numpy array (a highly efficient list used for numbers).

```python
from wiki2shards.io.hashing import canonical_json, sha256_hex

```

This line goes into another file in the same project and grabs two specific helper functions: one to format JSON data exactly the same way every time (`canonical_json`), and another to generate a specific type of unique security code (`sha256_hex`).

```python
EOS_TEXT = "<|endoftext|>"
PAD_TEXT = "<|pad|>"

```

These lines create two permanent labels (constants) that hold special text strings. `EOS_TEXT` represents the exact marker used to signal the end of a text block. `PAD_TEXT` represents the marker used to fill empty space so multiple pieces of text can be made exactly the same length.

```python
@runtime_checkable
class Tokenizer(Protocol):

```

This sets up a mandatory blueprint called `Tokenizer`. The `@runtime_checkable` tag means the code can actively verify if an object meets all the requirements of this blueprint. Any actual tokenizer tool used later in the code must contain everything listed inside this class.

```python
    @property
    def name(self) -> str: ...  # the spec string; goes to manifest tokenizer.spec

```

This dictates that any tokenizer must have a property called `name` that provides a text string. The comment notes this string is used in a record file (manifest). The `...` just means the actual tokenizer will fill in the details.

```python
    @property
    def fingerprint(self) -> str: ...  # sha256 hex

```

This dictates that the tokenizer must have a property called `fingerprint` that provides a unique text string (a 64-character code) to identify it.

```python
    @property
    def vocab_size(self) -> int: ...

```

This dictates that the tokenizer must have a property called `vocab_size` that provides a whole number (integer) representing how many distinct text pieces (tokens) it knows.

```python
    @property
    def eos_id(self) -> int: ...

```

This dictates that the tokenizer must have a property called `eos_id` that provides the specific whole number corresponding to the "end of text" marker.

```python
    @property
    def pad_id(self) -> int | None: ...

```

This dictates that the tokenizer must have a property called `pad_id` that provides either a whole number or nothing (`None`), representing the padding marker.

```python
    def encode_batch(self, texts: Sequence[str]) -> list[NDArray[np.int64]]:
        """Encode plain text. Special-token strings inside texts are ordinary text."""
        ...

```

This dictates that the tokenizer must have an `encode_batch` function. It must accept a list of text strings and return a list of numerical arrays. The comment clarifies that if any special markers appear naturally in the user's text, they should just be treated as regular text.

```python
    def decode(self, ids: Sequence[int] | NDArray[np.integer]) -> str: ...

```

This dictates that the tokenizer must have a `decode` function that accepts a list or array of whole numbers and converts them back into a single readable text string.

```python
def effective_pad_id(tok: Tokenizer) -> int:
    """GPT-2 has no pad token, so pad = eos; padding is identified by segments, not values."""
    return tok.pad_id if tok.pad_id is not None else tok.eos_id

```

This defines a new function that takes a tokenizer object. The comment explains that some models, like GPT-2, don't have a dedicated padding marker. The function checks if a padding ID exists; if it does, it returns it. If it doesn't, it returns the end-of-text ID instead.

```python
def fingerprint_hf_json(raw: bytes) -> str:
    """sha256 of the canonical form of tokenizer.json (floats are rejected: BPE files have none)."""
    return sha256_hex(canonical_json(json.loads(raw)))

```

This defines a function that takes raw binary data. It converts the binary data into a JSON object, formats it perfectly cleanly, and then generates a unique identifier code from it. This is used to securely identify Hugging Face ("hf") format tokenizers.

```python
def fingerprint_tiktoken(
    name: str,
    pat_str: str,
    special_tokens: Mapping[str, int],
    mergeable_ranks: Mapping[bytes, int],
) -> str:

```

This starts defining a function to create a unique identifier for OpenAI ("tiktoken") tokenizers. It requires four inputs: a name, a text pattern, a dictionary of special markers, and a dictionary of standard text fragments paired with their numerical ranks. It promises to output a text string.

```python
    """sha256 over a canonical header, then (<u4 rank, <u4 len, bytes) for each rank in order."""

```

This is a comment explaining exactly how the unique identifier will be mathematically constructed piece by piece.

```python
    h = hashlib.sha256()

```

This creates a fresh, empty digital blender (a sha256 hashing object) to start mixing the data together securely.

```python
    header = {"name": name, "pat_str": pat_str, "special_tokens": dict(special_tokens)}

```

This groups the name, pattern string, and special tokens together into a single dictionary called a `header`.

```python
    h.update(canonical_json(header) + b"\n")

```

This converts the `header` dictionary into a perfectly formatted text byte string, adds a "new line" character to the end, and drops it into the hashing object.

```python
    for token, rank in sorted(mergeable_ranks.items(), key=lambda kv: kv[1]):

```

This line starts a loop that goes through every standard token fragment. Before looping, it sorts them by their numerical rank from lowest to highest.

```python
        h.update(struct.pack("<II", rank, len(token)))

```

For every token in the loop, this packs its numerical rank and the length of its text into a tiny chunk of raw binary data and drops it into the hashing object.

```python
        h.update(token)

```

This then drops the actual raw text bytes of the token itself into the hashing object.

```python
    return h.hexdigest()

```

Once the loop is finished, this asks the hashing object to blend all the dropped-in pieces and return the final unique identifier as a readable text string.

```python
_CACHE: dict[str, Tokenizer] = {}

```

This creates an empty dictionary named `_CACHE`. The underscore means it is meant for internal use only. This acts as a memory bank to save tokenizers once they are loaded so the program doesn't have to rebuild them if asked for them again.

```python
def load_tokenizer(spec: str) -> Tokenizer:
    """Construct (once per process) the tokenizer named by a spec string.

    Adapters are imported by name, so this contract module does not depend on them (they are
    written later, in tokenize/hf_tok.py and tokenize/tiktoken_tok.py).
    """

```

This begins a function that takes a specification text string and promises to build and return a fully working Tokenizer. The comment explains that the actual building tools are loaded on the fly from other files, keeping this core file independent.

```python
    if spec in _CACHE:
        return _CACHE[spec]

```

This checks if the requested tokenizer has already been built and saved in the memory bank (`_CACHE`). If it has, it immediately returns the saved version to save time.

```python
    kind, _, arg = spec.partition(":")

```

This splits the requested specification text string into three pieces at the colon. The word before the colon becomes `kind` (e.g., "tiktoken"), the colon itself is ignored (the `_`), and the rest becomes `arg` (e.g., "gpt2").

```python
    tok: Tokenizer

```

This tells Python to prepare a variable named `tok` that will eventually hold an object matching the Tokenizer blueprint.

```python
    if kind == "tiktoken":

```

This checks if the first half of the requested string was "tiktoken".

```python
        tok = importlib.import_module(
            "wiki2shards.tokenize.tiktoken_tok"
        ).TiktokenAdapter.from_name(arg)

```

If it was "tiktoken", the program opens a file named `tiktoken_tok` inside the project, grabs a specific tool inside called `TiktokenAdapter`, and uses it to build the tokenizer based on the second half of the requested string (`arg`). It assigns the result to `tok`.

```python
    elif kind == "hf":

```

If the first half of the requested string wasn't "tiktoken", this checks if it was "hf" (Hugging Face).

```python
        tok = importlib.import_module("wiki2shards.tokenize.hf_tok").HFAdapter.from_file(
            arg, spec=spec
        )

```

If it was "hf", the program opens a file named `hf_tok`, grabs a tool called `HFAdapter`, and uses it to build the tokenizer by reading a file located at the path provided in `arg`. It assigns the result to `tok`.

```python
    else:
        raise ValueError(f"unknown tokenizer spec {spec!r}")

```

If the string didn't start with "tiktoken" or "hf", the program stops working and throws an error message explaining that it doesn't recognize the request format.

```python
    _CACHE[spec] = tok

```

After successfully building the tokenizer, this saves a copy of it into the memory bank (`_CACHE`), using the original request string as the label.

```python
    return tok

```

Finally, this hands the newly built tokenizer back to whatever part of the program originally asked for it.

### Step 1.6: Public loader types

`src/wiki2shards/loader/api.py`

Here is a line-by-line breakdown of your Python code in plain English:

* **`"""Public loader types. Frozen at contracts-v1.`**
This starts a multi-line comment (called a docstring) explaining that this file defines data types for loading data, and that these rules are locked at version 1.
* **`from wiki2shards.loader import Dataset, Mixture, Loader`**
This is just an example inside the comment showing how someone might import tools from this project in the future.
* **`"""`**
This ends the multi-line comment block.
* **`from __future__ import annotations`**
This is a special Python instruction that tells the code to handle data type hints in a modern way, allowing class names to be used as types before they are fully defined.
* **`from dataclasses import dataclass`**
This brings in a helpful tool called `dataclass` that automatically writes standard setup code (like creating variables) for classes whose main job is just to hold data.
* **`import numpy as np`**
This loads the `numpy` library (a popular tool for doing math with large lists of numbers) and gives it the short nickname `np` so it is easier to type.
* **`from numpy.typing import NDArray`**
This brings in `NDArray`, which is just a label used to tell the code, "Hey, expect a numpy array (a grid of numbers) here."
* **`STATE_FORMAT = "wiki2shards.loader_state"`**
This creates a constant variable holding a text string to name the format of the saved data state.
* **`STATE_VERSION = 1`**
This creates a constant variable to track that this code is using version 1 of the state format.
* **`@dataclass(frozen=True)`**
This applies the `dataclass` tool to the blueprint directly below it. The `frozen=True` part means once this data is created, it is locked and cannot be changed.
* **`class Batch:`**
This starts the definition of a new custom object blueprint (a class) named `Batch`. It will hold a batch of training data for a machine learning model.
* **`    tokens: NDArray[np.unsignedinteger]  # [B, row_len]; inputs tokens[:, :-1], targets [:, 1:]`**
This creates a property named `tokens` that expects a numpy array of positive whole numbers. The comment explains the grid's shape (Batch size by row length) and how the data is split into inputs and targets.
* **`    loss_mask: NDArray[np.uint8]  # [B, seq_len]; 0 where the target is padding`**
This creates `loss_mask`, an array of small 8-bit numbers. The comment notes it tells the model what data to pay attention to and what data is just empty "padding" (marked as 0).
* **`    segment_ids: NDArray[np.int32] | None  # [B, row_len]; segment index in row, -1 on padding`**
This creates `segment_ids`, which can either be an array of standard numbers or `None` (empty). It helps separate different pieces of text in the same row.
* **`    sample_index: NDArray[np.int64]  # [B]; the t of each row`**
This creates `sample_index`, an array of larger 64-bit numbers used to keep track of the specific sample ID for each row in the batch.
* **`    source_index: NDArray[np.int16]  # [B]; index into the mixture's sources`**
This creates `source_index`, an array of 16-bit numbers that tracks which specific dataset or file this batch of data came from.
* **`    row_id: NDArray[np.int64]  # [B]; global row within its source`**
This creates `row_id`, an array of 64-bit numbers that remembers the exact, original row number this data lived at inside its source file.
* **`    step: int  # global batch index`**
This creates `step`, a standard whole number that tracks which batch number this is in the overall training process.
* **`class LoaderStateError(ValueError):`**
This creates a custom error message named `LoaderStateError`. By putting `ValueError` in the parentheses, it tells Python to treat this like a standard "bad value" error.
* **`    """load_state_dict refused: format, fingerprint, dataset or draw-count mismatch."""`**
This docstring explains when this error happens: it triggers if the program tries to load a saved state but the data format, dataset, or amounts do not perfectly match what is expected.

### Step 1.7: Fixture config and language codes

`configs/fixture.yaml`

* `# Offline build of the generated fixture dump. Create the dump first:`
This is a note for the programmer. It explains that this file processes a small batch of test data (a "fixture dump") locally on the computer, and reminds you that you need to generate that test data before running this configuration.
* `#   uv run python scripts/make_fixture_dump.py --out data/fixture_dump`
Another note showing the exact command you need to type into your computer's terminal to create that required test data.
* `name: fixturewiki-20260901-fixture-s64`
This gives a specific title to this data processing job so you can identify it later.
* `source: {wiki: fixturewiki, date: "20260901", mirror: data/fixture_dump}`
This tells the program where to find its starting material. It's looking for a test Wikipedia dataset ("fixturewiki") dated September 1, 2026, located in a folder called `data/fixture_dump`.
* `workdir: data/work/fixturewiki-20260901`
This tells the program to use the folder `data/work/fixturewiki-20260901` as a temporary workspace while it does its processing.
* `output: data/datasets/fixturewiki-20260901-fixture-s64`
This is the final destination. Once the program finishes cleaning and packaging the data, it will save the final result in this folder.
* `workers: 2`
This allows the program to split its tasks across two of your computer's processor cores at the same time to finish the job faster.
* `extract: {unit_bytes: 1}            # one bz2 stream per unit -> 3 units, so workers matter`
This tells the program to pull the data apart in incredibly small 1-byte chunks (mostly to test if the program handles chunking correctly). The note explains that doing this creates 3 separate streams of data, proving that using the 2 "workers" from the previous line will actually help speed things up.
* `clean:`
This acts as a header, signaling that the next few lines are rules for cleaning up the messy text.
* `  drop_sections: [References, Notes, External links, See also, Further reading, Bibliography, Sources]`
This tells the cleaner to completely delete these specific sections from the bottom of the articles, as they aren't useful for training a language model.
* `  soft_filters: {min_words: 50, max_symbol_pct: 30, disambiguation: true, list_page: true}`
This sets up a system to "flag" bad articles. An article gets flagged if it has fewer than 50 words, if more than 30% of it is made of symbols, if it's a "disambiguation" page (a page that just clarifies which similar topic you meant), or if it is just a list of items.
* `  disambiguation_templates: [disambiguation, disambig, dab, hndis, geodis]`
Wikipedia uses special hidden code words for disambiguation pages. This tells the program exactly which code words to look for so it knows when to flag a page.
* `  language_codes_file: configs/language_codes.txt`
This tells the program to look inside a file called `configs/language_codes.txt` to find the master list of accepted languages.
* `select: {policy: default, apply: [min_words, disambiguation], dedup: exact, val_per_10k: 2000}`
This decides which articles actually survive and make it into the final dataset. It applies the standard rules, throws out any articles that were flagged earlier for being too short or being disambiguation pages, deletes any exact duplicate pages, and sets aside 2,000 out of every 10,000 articles to use later as a test to see how smart the AI is getting.
* `tokenizer: hf:tests/fixtures/tokenizer.json`
AI models read text by chopping words into tiny pieces called tokens. This tells the program to use a specific dictionary file (`tokenizer.json`) to know exactly how to chop those words up.
* `pack: {mode: concat, seq_len: 64, last_row: drop, shard_bytes: 16384, bfd_window: 100000}`
This dictates how the final chopped-up text is packaged for the AI. It strings all the articles together end-to-end (`concat`), cuts that string into equal segments of 64 tokens, throws away the leftover pieces at the very end (`drop`), bundles the files so none are larger than 16,384 bytes, and shuffles the data around within a window of 100,000 items so the AI doesn't learn things in a predictable order.

`configs/language_codes.txt`

@ /IMPLEMENTATION_GUIDE.md

### Step 1.8: Test helpers and contract tests

#### `tests/helpers/build.py`

```python
"""Helpers to generate fixture dumps and build them into tmp directories."""

```

This is a docstring (a text comment) that explains what this whole file does: it provides helper tools to create fake data (fixtures) and build them in temporary folders for testing.

```python
from __future__ import annotations

```

This enables modern Python type hinting (like `list[str]`) even if an older version of Python is running.

```python
import hashlib
import importlib.util
import json
import os
import subprocess
import sys
import tempfile
from dataclasses import dataclass
from pathlib import Path
from typing import Any

```

These lines import built-in Python tools:

* `hashlib`: For creating security hashes/checksums of files.
* `importlib.util`: For loading Python code from file paths directly.
* `json`: For reading and writing JSON data format.
* `os`: For interacting with the operating system (like checking environment variables).
* `subprocess`: For running external command-line programs.
* `sys`: For interacting with the Python interpreter (like finding the current Python path).
* `tempfile`: For creating temporary files that clean themselves up.
* `dataclass`: A decorator that makes creating classes for storing data easier.
* `Path`: A tool from `pathlib` that makes working with file paths (like `/user/folder/file.txt`) much easier.
* `Any`: A type hint meaning a variable can be any type of data.

```python
from wiki2shards.config import BuildConfig, load_build_config

```

This imports specific project settings (the configuration structure and the tool to load it) from the main `wiki2shards` program.

```python
REPO = Path(__file__).resolve().parents[2]

```

This calculates the main root folder of the project. It finds the exact location of the current file (`__file__`), resolves its true path, and goes exactly two folders up (`parents[2]`), saving it as `REPO`.

```python
FIXTURE_CONFIG = REPO / "configs" / "fixture.yaml"

```

This creates a path to a specific testing configuration file named `fixture.yaml` located inside the `configs` folder of the main repository.

---

> `_load_script` Function

```python
def _load_script(name: str) -> Any:

```

Defines a function named `_load_script` that takes a script's name (text) and returns a loaded Python module.

```python
    spec = importlib.util.spec_from_file_location(name, REPO / "scripts" / f"{name}.py")

```

It looks for a Python file with that name inside the repository's `scripts` folder and creates a "specification" (a blueprint for how to load it).

```python
    assert spec and spec.loader

```

This is a safety check ensuring that the file was actually found and has a valid loader tool.

```python
    mod = importlib.util.module_from_spec(spec)

```

Creates an empty Python module object based on that blueprint.

```python
    sys.modules[name] = mod  # dataclasses need the module registered

```

Registers this new module into Python's official list of loaded modules so other parts of the code (like `dataclasses`) recognize it.

```python
    spec.loader.exec_module(mod)

```

Executes the code inside the script file to actually fill the empty module with the script's functions and variables.

```python
    return mod

```

Returns the fully loaded, usable Python module.

---

> `make_dumps` Function

```python
def make_dumps(out: Path) -> Path:

```

Defines a function that takes an output folder path and returns a folder path.

```python
    gen = _load_script("make_fixture_dump")

```

Uses the previous helper to load a specific script named `make_fixture_dump.py`.

```python
    gen.make_dump(out, "fixturewiki", "20260901", 1)
    gen.make_dump(out, "fixturewiki", "20261001", 2)
    gen.make_dump(out, "fixture2wiki", "20260901", 1)

```

Calls a function inside that loaded script three times to generate three sets of fake Wikipedia data (different wikis, different dates, different sizes) and saves them into the `out` folder.

```python
    return out

```

Returns the output folder path where the data was saved.

---

> `fixture_config` Function

```python
def fixture_config(
    root: Path, dumps: Path, wiki: str = "fixturewiki", date: str = "20260901", **changes: Any
) -> BuildConfig:

```

Defines a function to create a test configuration. It takes a root folder, a folder containing the dumps, optional wiki/date names, and allows capturing any extra custom settings via `**changes`.

```python
    data = load_build_config(FIXTURE_CONFIG).model_dump(mode="python")

```

Loads the base configuration from `fixture.yaml` and converts it into a standard Python dictionary so it can be easily edited.

```python
    data["name"] = f"{wiki}-{date}-fixture-s64"

```

Sets the overall name of this specific test run based on the wiki and date provided.

```python
    data["source"].update(wiki=wiki, date=date, mirror=str(dumps))

```

Updates the "source" section of the configuration dictionary with the provided wiki name, date, and the folder containing the fake data dumps.

```python
    data["workdir"] = str(root / "work")
    data["output"] = str(root / "out")

```

Tells the configuration to use a `work` folder for temporary processing files and an `out` folder for the final results, both located inside the provided `root` folder.

```python
    for dotted, value in changes.items():

```

Starts a loop over any extra custom settings passed into the function (e.g., `setting__subsetting=5`).

```python
        node = data

```

Creates a temporary pointer (`node`) that starts at the top level of the configuration dictionary.

```python
        *parents, leaf = dotted.split("__")

```

Splits the custom setting name by double underscores (`__`). It assigns the last part to `leaf` (the specific setting to change) and everything before it to `parents` (the folders/categories it lives in).

```python
        for p in parents:
            node = node[p]

```

Loops through the parent categories to drill down deep into the nested dictionary to find exactly where the setting belongs.

```python
        node[leaf] = value

```

Applies the new custom value to the exact specific setting (the `leaf`).

```python
    return BuildConfig.model_validate(data)

```

Validates the newly modified dictionary to ensure it strictly follows the rules of a `BuildConfig` object, and returns it.

---

> `Built` Class

```python
@dataclass
class Built:

```

Creates a simple data container class named `Built`.

```python
    cfg: BuildConfig
    root: Path

```

Declares that every `Built` object will hold two things: the configuration used (`cfg`) and the main folder where it ran (`root`).

```python
    @property
    def out(self) -> Path:
        return Path(self.cfg.output)

```

Creates a shortcut property. Typing `my_build.out` will automatically return the output folder path defined in the configuration.

```python
    @property
    def work(self) -> Path:
        return Path(self.cfg.workdir)

```

Creates another shortcut property. Typing `my_build.work` returns the temporary work folder path.

```python
    def expected(self) -> dict[str, Any]:

```

Defines a method to load the "expected" correct answers for testing purposes.

```python
        d = Path(self.cfg.source.mirror) / self.cfg.source.wiki / self.cfg.source.date

```

Constructs the exact folder path where the fake Wikipedia dump data is located.

```python
        return json.loads((d / "fixture_expected.json").read_text())  # type: ignore[no-any-return]

```

Finds a file named `fixture_expected.json` in that folder, reads its text, converts the JSON text into a Python dictionary, and returns it. The comment tells the type-checker to ignore strict return rules here.

---

> `build_fixture` Function

```python
def build_fixture(root: Path, dumps: Path, **kw: Any) -> Built:

```

Defines a function that actually runs the whole pipeline process for a test.

```python
    from wiki2shards.pipeline.runner import Build  # Phase 7; imported lazily for early phases

```

Imports the main `Build` engine from the project. It's imported here (lazily) inside the function rather than at the top of the file so that earlier tests don't crash if this module isn't ready yet.

```python
    cfg = fixture_config(root, dumps, **kw)

```

Creates the customized testing configuration using the helper function defined earlier.

```python
    Build(cfg).run()

```

Initializes the main `Build` engine with that configuration and runs the entire data processing pipeline.

```python
    return Built(cfg, root)

```

Returns a `Built` object containing the configuration and root path so the test can inspect the results.

---

> `write_config` Function

```python
def write_config(cfg: BuildConfig, path: Path) -> Path:

```

Defines a function to save a configuration object to a text file.

```python
    import yaml

```

Imports the `yaml` library, which is used for reading and writing YAML formatted text files.

```python
    path.write_text(yaml.safe_dump(cfg.model_dump(mode="python"), sort_keys=True))

```

Converts the configuration object into a standard dictionary, turns that dictionary into cleanly formatted YAML text (sorting the keys alphabetically), and writes it into the specified file path.

```python
    return path

```

Returns the path to the newly saved file.

---

> `CliResult` Class

```python
@dataclass
class CliResult:

```

Creates another data container to hold the results of running a terminal command.

```python
    returncode: int
    stdout: str
    stderr: str
    pid: int  # with new_session=True this is also the process-group id

```

It stores four pieces of data:

* `returncode`: The exit code (0 usually means success, anything else means an error).
* `stdout`: The standard printed text output of the command.
* `stderr`: The error messages printed by the command.
* `pid`: The Process ID (a unique number the computer assigns to the running program).

---

> `run_cli` Function

```python
def run_cli(
    args: list[str], env: dict[str, str] | None = None, new_session: bool = False
) -> CliResult:

```

Defines a function to simulate typing commands in the terminal (Command Line Interface). It takes a list of arguments, optional environment variables, and a flag to start a new independent process session.

```python
    """Run `python -m wiki2shards ...`. Output goes to files, not pipes: after a crash, orphaned
    pool workers may still hold inherited pipes open, and reading them would block."""

```

A docstring explaining that this runs the main program and intentionally saves output to temporary files rather than direct computer memory pipes to prevent the program from freezing (blocking) if it crashes.

```python
    full_env = {**os.environ, **(env or {})}

```

Combines the computer's standard environment variables with any custom ones passed into the function.

```python
    with tempfile.TemporaryFile("w+") as out, tempfile.TemporaryFile("w+") as err:

```

Safely creates two temporary files on the computer to capture standard output (`out`) and error output (`err`). When the block of code finishes, these files are automatically deleted.

```python
        proc = subprocess.Popen(

```

Starts running a new background process (a terminal command) without immediately waiting for it to finish.

```python
            [sys.executable, "-m", "wiki2shards", *args],

```

The exact command being run: It finds the current Python program running (`sys.executable`), tells it to run the `wiki2shards` module, and attaches all the provided custom text arguments.

```python
            cwd=REPO,

```

Sets the "Current Working Directory" (where the command is executed from) to the main project repository root.

```python
            env=full_env,

```

Passes the combined environment variables to the new command.

```python
            stdout=out,
            stderr=err,

```

Tells the command to write all its standard prints to the temporary `out` file and all errors to the temporary `err` file.

```python
            start_new_session=new_session,
        )

```

If `new_session` is true, makes this command the leader of its own process group (useful for cleanly killing it and any sub-programs it creates later).

```python
        code = proc.wait(timeout=300)

```

Pauses the current Python script and waits up to 300 seconds (5 minutes) for the background command to finish, capturing its exit `code`.

```python
        out.seek(0)
        err.seek(0)

```

Rewinds the "cursor" in both temporary files back to the very beginning so they can be read from start to finish.

```python
        return CliResult(code, out.read(), err.read(), proc.pid)

```

Reads all the text from both files, packages it up along with the exit code and process ID, and returns it as a `CliResult` object.

---

> `digests` Function

```python
def digests(root: Path, exclude: tuple[str, ...] = ("_state",)) -> dict[str, str]:

```

Defines a function that scans a folder and calculates a unique digital fingerprint (hash) for every file inside it, skipping folders named `_state` by default.

```python
    """sha256 of every file under a final name below root (relative path -> digest).
    Skips the stage journals (_state) and hidden files (.<name>.tmp and .<name>.part are
    in-progress writes, never final names)."""

```

A docstring explaining what the function does and what files it intentionally ignores (hidden files and temporary unfinished files).

```python
    out = {}

```

Creates an empty dictionary to store the results (file path mapped to its hash).

```python
    for p in sorted(root.rglob("*")):

```

Searches for absolutely every file and folder (`*`) inside the `root` folder and its subfolders (`rglob`), sorts them alphabetically, and loops through them one by one (`p`).

```python
        rel = p.relative_to(root)

```

Calculates the file's path relative to the root folder (e.g., if root is `/data/` and the file is `/data/folder/file.txt`, it calculates `folder/file.txt`).

```python
        if (
            p.is_file()
            and not any(part in exclude for part in rel.parts)
            and not p.name.startswith(".")
        ):

```

A strict conditional check that ensures three things:

1. It is an actual file (not a folder).
2. None of the folders in its path match the excluded names (like `_state`).
3. The file's name does not start with a dot (`.`), meaning it is not a hidden or temporary file.

```python
            out[str(rel)] = hashlib.sha256(p.read_bytes()).hexdigest()

```

If it passes the checks, it reads all the raw data inside the file, calculates a SHA-256 cryptographic hash of that data, converts the hash into a readable text string (`hexdigest`), and saves it in the dictionary under the file's relative path name.

```python
    return out

```

Returns the completed dictionary of file hashes.

---

> `build_digests` Function

```python
def build_digests(built_root: Path) -> dict[str, str]:

```

Defines a helper function that takes the root folder of a finished build and returns hashes for all files in specific subfolders.

```python
    """Digests of a build's stage outputs (work/) and dataset (out/)."""

```

Docstring stating that it will specifically hash the files in the `work` and `out` directories.

```python
    return {f"{k}/{rel}": d for k in ("work", "out") for rel, d in digests(built_root / k).items()}

```

This is a single-line dictionary comprehension that does the following:

1. Loops over two folder names: `"work"` and `"out"`.
2. Runs the `digests` function on each of those specific subfolders.
3. Takes the results and combines them into one massive dictionary, formatting the name so it clearly shows which main folder (`k`) the file (`rel`) came from. Returns the combined dictionary.

---

> `write_mix` Function

```python
def write_mix(
    path: Path, sources: list[tuple[str, Path, int]], seed: int = 1234, split: str = "train"
) -> Path:

```

Defines a function to create a configuration file that mixes different data sources. It requires a save path and a list of sources, with optional random seed and dataset split type (defaulting to "train").

```python
    lines = [f"seed: {seed}", f"split: {split}", "sources:"]

```

Creates a list containing the first three lines of text for a YAML configuration file: the seed number, the split name, and the start of a `sources:` list.

```python
    lines += [f"  - {{name: {n}, path: '{p}', weight: {w}}}" for n, p, w in sources]

```

Loops through the provided list of sources (where each source has a name `n`, path `p`, and weight `w`) and generates a formatted YAML text line for each one, adding them to the existing list of lines.

```python
    path.write_text("\n".join(lines) + "\n")

```

Takes all those individual text lines, glues them together into a single large block of text separated by line breaks (`\n`), adds one final blank line at the very end, and writes all of it into the specified file path.

```python
    return path

```

Returns the path to the file that was just written.

#### `tests/unit/test_io_binfmt.py`

| Code Line | Plain English Explanation |
| --- | --- |
| `import numpy as np` | Loads the NumPy library (used for high-performance lists of numbers) and nicknames it `np`. |
| `import pytest` | Loads the pytest library, which is used to run automated tests on the code. |
| `from wiki2shards.io.binfmt import (` | Starts importing specific tools and variables from a custom file in the project named `binfmt`. |
| `HEADER_SIZE,` | Imports a variable that stores the size of the file's header. |
| `U2,` | Imports a shortcut for a 2-byte unsigned integer data type. |
| `U4,` | Imports a shortcut for a 4-byte unsigned integer data type. |
| `BadMagic,` | Imports a specific error type triggered when a file is missing its correct identifying signature ("magic number"). |
| `DtypeRangeError,` | Imports an error type used when a number is too big or small to fit in a specific data type. |
| `IdxHeader,` | Imports a blueprint (class) for storing index header information. |
| `Segments,` | Imports a blueprint (class) for storing chunks (segments) of document data. |
| `SizeMismatch,` | Imports an error type triggered when a file's actual size doesn't match its expected size. |
| `UnsupportedVersion,` | Imports an error type for when the code tries to read a file version it doesn't understand. |
| `checked_cast,` | Imports a function that safely converts data types without secretly losing information. |
| `decode_idx,` | Imports a function that turns raw computer bytes back into Python data (`IdxHeader` and `Segments`). |
| `dtype_for_vocab,` | Imports a function that figures out the smallest data type needed to store a certain number of words (vocabulary). |
| `encode_idx,` | Imports a function that turns Python data into raw computer bytes to be saved. |
| `idx_size,` | Imports a function that calculates exactly how many bytes a file should be. |
| `)` | Closes the list of imports. |
| `def _sample(n_rows=3):` | Creates a helper function named `_sample` that creates fake data for testing. It defaults to creating 3 rows. |
| `seg = Segments(` | Creates a new `Segments` data container and saves it to the variable `seg`. |
| `row_seg_ptr=np.array([0, 2, 3, 5][: n_rows + 1], dtype=np.uint64),` | Creates a list of 64-bit numbers indicating where rows start, cutting the list down based on the `n_rows` requested. |
| `doc_uid=np.array([11, 12, 12, 13, 14], dtype=np.uint64),` | Creates a list of 64-bit numbers representing unique ID numbers for documents. |
| `start=np.array([0, 3, 0, 0, 4], dtype=np.uint32),` | Creates a list of 32-bit numbers showing the starting positions of text pieces. |
| `length=np.array([3, 5, 8, 4, 4], dtype=np.uint32),` | Creates a list of 32-bit numbers showing the lengths of those text pieces. |
| `piece=np.array([0, 0, 1, 0, 0], dtype=np.uint32),` | Creates a list of 32-bit numbers tracking which part of a split document this is. |
| `)` | Closes the `Segments` creation. |
| `hdr = IdxHeader(` | Creates a new `IdxHeader` container for file metadata and saves it to `hdr`. |
| `itemsize=2,` | Notes that each item takes up 2 bytes of space. |
| `pack_mode=0,` | Sets the data packing style to mode 0. |
| `split=1,` | Indicates this file represents 1 split/chunk of a larger dataset. |
| `row_len=8,` | Sets the length of a row to 8 units. |
| `shard_index=4,` | Identifies this specific file as the 4th piece (shard) in a sequence. |
| `n_rows=3,` | States that there are 3 rows of data in this file. |
| `n_segments=5,` | States that there are 5 total segments of data. |
| `n_pad_tokens=0,` | States there are zero empty filler (padding) tokens. |
| `first_global_row=120,` | Notes that the first row in this file is the 120th row in the entire global dataset. |
| `bin_sha256=bytes(range(32)),` | Creates a fake 32-byte security checksum (hash) using the numbers 0 through 31. |
| `tokenizer_fp=bytes(range(32, 64)),` | Creates a fake 32-byte fingerprint for the text processor (tokenizer) using the numbers 32 through 63. |
| `)` | Closes the `IdxHeader` creation. |
| `return hdr, seg` | Sends the newly created header and segment data back to whatever called this function. |
| `def test_idx_roundtrip():` | Defines an automated test to ensure data can be saved to bytes and read back perfectly. |
| `hdr, seg = _sample()` | Calls the helper function above to get fresh fake header and segment data. |
| `raw = encode_idx(hdr, seg)` | Converts the fake data into raw computer bytes and saves it as `raw`. |
| `assert len(raw) == idx_size(3, 5) == HEADER_SIZE + 8 * 4 + 20 * 5` | Checks that the length of the raw bytes exactly matches the mathematical formula for the expected file size. If not, the test fails. |
| `assert idx_size(1000, 1600) == 40_136  # SPEC 1.4 worked example` | Checks a known mathematical example to ensure the size calculator function (`idx_size`) works correctly. |
| `hdr2, seg2 = decode_idx(raw)` | Takes the raw bytes and converts them back into Python objects, saving them as `hdr2` and `seg2`. |
| `assert hdr2 == hdr` | Verifies that the restored header perfectly matches the original header. |
| `for name in ("row_seg_ptr", "doc_uid", "start", "length", "piece"):` | Starts a loop to check each specific list of numbers inside the segment data. |
| `assert np.array_equal(getattr(seg2, name), getattr(seg, name))` | Verifies that the restored list of numbers is completely identical to the original list of numbers. |
| `with pytest.raises(BadMagic):` | Tells the test runner to expect the very next line of code to crash with a `BadMagic` error. |
| `decode_idx(b"W2S-IDY\0" + raw[8:])` | Attempts to decode corrupted data where the file's opening signature has been changed to something invalid. |
| `bumped_major = raw[:8] + (2).to_bytes(2, "little") + raw[10:]` | Manually hacks the raw bytes to change the file's "major version" number to 2. |
| `with pytest.raises(UnsupportedVersion):` | Tells the test runner to expect the next line to crash with an `UnsupportedVersion` error. |
| `decode_idx(bumped_major)` | Tries to read the hacked bytes; it should fail because the system doesn't know how to read version 2. |
| `with pytest.raises(SizeMismatch):` | Tells the test runner to expect a `SizeMismatch` error on the next line. |
| `decode_idx(raw[:-1])` | Tries to read the bytes but chops off the very last byte, simulating an incomplete file download. |
| `with pytest.raises(SizeMismatch):` | Tells the test runner to expect a `SizeMismatch` error on the next line. |
| `decode_idx(raw + b"\0")` | Tries to read the bytes but adds one extra dummy byte at the end, simulating a file that is unexpectedly long. |
| `with pytest.raises(SizeMismatch):` | Tells the test runner to expect a `SizeMismatch` error on the next line. |
| `decode_idx(raw[:100])` | Tries to read only the first 100 bytes, simulating a severely cut-off file. |
| `higher_minor = raw[:10] + (7).to_bytes(2, "little") + raw[12:]` | Manually hacks the raw bytes to change the "minor version" number to 7. |
| `assert decode_idx(higher_minor)[0].version_minor == 7  # readers accept higher minors` | Verifies that the system successfully reads the file anyway, because minor version updates are designed to be backward-compatible. |
| `def test_dtype_guard():` | Defines an automated test to make sure large numbers don't silently wrap around (corrupt) when crammed into small memory spaces. |
| `ids = np.array([1, 2, 70_000], dtype=np.int64)` | Creates a list of three numbers, including one very large one (70,000), using a spacious 64-bit data type. |
| `assert int(ids.astype(U2)[-1]) == 4464  # what a plain cast would silently store` | Proves that standard Python/NumPy would silently corrupt 70,000 into 4,464 if forced into a tiny 2-byte space, without warning you. |
| `with pytest.raises(DtypeRangeError):` | Tells the test runner to expect a `DtypeRangeError` on the next line. |
| `checked_cast(ids, U2)` | Tries to use the custom safe-conversion function to cram the large numbers into the 2-byte space. It should intentionally crash rather than corrupting the data. |
| `with pytest.raises(DtypeRangeError):` | Tells the test runner to expect a `DtypeRangeError` on the next line. |
| `checked_cast(np.array([-1]), U4)` | Tries to safely convert a negative number into a data type that only allows positive numbers. It should intentionally crash. |
| `assert checked_cast(ids, U4).tolist() == [1, 2, 70_000]` | Verifies that the safe-conversion function successfully moves the numbers into a 4-byte space, because 4 bytes is large enough to hold 70,000. |
| `assert dtype_for_vocab(65_536) == U2 and dtype_for_vocab(65_537) == U4` | Verifies that a vocabulary size of exactly 65,536 can fit into 2 bytes, but adding just one more word (65,537) correctly forces the system to use a larger 4-byte space. |

#### `tests/unit/test_io_manifest.py`

> Setup and Imports

`import json`
Imports Python's built-in tool for reading and writing JSON data.

`import pytest`
Imports a popular testing tool used to verify that code works correctly.

`from wiki2shards.io.hashing import canonical_json_pretty`
Imports a specific function that formats JSON text into a strict, neat, and predictable layout.

`from wiki2shards.io.manifest import (`
Opens an import block to bring in multiple specific tools from a custom module dealing with "manifests" (files that describe a dataset).

`Manifest,`
Imports the main class used to represent and validate the structure of a manifest.

`ManifestError,`
Imports a specific error type that is triggered when a manifest has invalid data.

`parse_manifest,`
Imports a function used to read manifest data directly from computer memory (bytes).

`read_manifest,`
Imports a function used to open and read a manifest from a saved file.

`write_manifest,`
Imports a function used to save manifest data to a file.

`)`
Closes the import block.

`SHA = "ab" * 32`
Creates a dummy 64-character text string (repeating "ab") to stand in for a real cryptographic security hash.

> Defining the Sample Data

`def sample() -> dict:`
Defines a function named `sample` that returns a dictionary containing dummy data for testing.

`shard = {`
Starts a dictionary that represents a "shard" (a small chunk of a larger dataset).

`"index": 0,`
Labels this as the first chunk in the sequence.

`"n_rows": 2,`
States that this chunk contains exactly two rows of data.

`"bin": "train-00000.bin",`
Gives the file name for where the raw binary data is stored.

`"bin_bytes": 20,`
States that the binary file is exactly 20 bytes in size.

`"bin_sha256": SHA,`
Assigns the dummy hash to act as a security check for the binary file.

`"idx": "train-00000.idx",`
Gives the file name for the index file, which helps locate data quickly.

`"idx_bytes": 168,`
States that the index file is exactly 168 bytes in size.

`"idx_sha256": SHA,`
Assigns the dummy hash to act as a security check for the index file.

`}`
Closes the dictionary describing the data chunk.

`split = {`
Starts a dictionary that describes a "split" (like a training set or testing set).

`"n_rows": 2,`
States there are two rows of data in this entire set.

`"n_tokens": 20,`
States there are 20 text tokens (pieces of words) in this set.

`"n_pad_tokens": 0,`
States there are zero padding (filler) tokens used here.

`"n_docs": 1,`
States this set contains exactly one document.

`"n_tokens_dropped_tail": 3,`
Notes that 3 extra tokens at the end were removed or ignored.

`"shards": [shard],`
Places the previously defined data chunk into a list inside this set.

`}`
Closes the dictionary describing the data set.

`return {`
Starts the main dictionary that the `sample` function will hand back when called.

`"format": "wiki2shards",`
Declares the name of the data format being used.

`"format_version": [1, 0],`
Sets the version number of this format to 1.0.

`"name": "demo",`
Gives the dataset the name "demo".

`"build": {`
Starts a section detailing how the dataset was constructed.

`"wiki2shards_version": "0.1.0",`
Records the version of the software used to create the data.

`"git_commit": "unknown",`
States that the exact code tracking identifier is not known.

`"lockfile_sha256": None,`
Indicates there is no security hash for software dependencies.

`"python": "3.12.14",`
Records the exact version of Python used to build the data.

`"unicode_version": "15.0.0",`
Records the text encoding standard version used.

`"config_sha256": SHA,`
Assigns the dummy hash to represent the configuration file used during the build.

`},`
Closes the build details section.

`"source": {`
Starts a section detailing where the original raw text came from.

`"wiki": "fixturewiki",`
Names the specific source as "fixturewiki".

`"dump_date": "20260901",`
Records the date (September 1, 2026) when the source data was downloaded.

`"files": [{"name": "x.xml.bz2", "bytes": 1, "sha1": "cd" * 20}],`
Lists the compressed original file used, its tiny fake size (1 byte), and a fake security hash.

`},`
Closes the source details section.

`"tokenizer": {`
Starts a section explaining how text was chopped into tokens.

`"spec": "tiktoken:gpt2",`
Specifies that it uses the same text-chopping rules as the GPT-2 artificial intelligence.

`"fingerprint": SHA,`
Provides the dummy hash as a unique identifier for these rules.

`"vocab_size": 50257,`
States that the system recognizes 50,257 unique text pieces.

`"eos_id": 50256,`
Identifies the specific number used to mark the "End of a Sentence".

`"pad_id": 50256,`
Identifies the specific number used for blank padding space.

`"dtype": "<u2",`
Specifies the computer memory format (unsigned 16-bit integer) used to store the numbers.

`},`
Closes the tokenizer details section.

`"packing": {`
Starts a section explaining how the tokens are arranged into rows.

`"mode": "concat",`
States that tokens from different documents are joined end-to-end.

`"seq_len": 9,`
Sets the target length for useful data to 9 tokens per row.

`"row_len": 10,`
Sets the total space per row to 10 tokens.

`"doc_order": "doc_uid_asc",`
States documents are ordered alphabetically by their unique ID.

`"last_row": "drop",`
Instructs the system to throw away the final row if it is not completely full.

`},`
Closes the packing details section.

`"selection": {`
Starts a section explaining how data was filtered.

`"policy": "default",`
States it used standard filtering rules.

`"soft_filters": ["min_words"],`
Lists an active filter rule requiring a minimum number of words.

`"dedup": "exact_text_sha256",`
States duplicate documents were removed by comparing exact text matches.

`"val_per_10k": 50,`
States 50 out of every 10,000 items were set aside for testing/validation.

`},`
Closes the selection details section.

`"splits": {"train": split, "val": {**split, "shards": []}},`
Assigns the `split` dictionary to the "train" set, and creates a "val" (validation) set that copies the same rules but empties out the shards list.

`"side_files": [{"name": "docs.jsonl", "bytes": 1, "sha256": SHA}],`
Lists an extra, related file called "docs.jsonl" alongside its fake size and security hash.

`}`
Closes the main dummy data dictionary.

> The Testing Function

`def test_manifest_roundtrip_and_versions(tmp_path):`
Defines a test function that receives a temporary, safe folder location (`tmp_path`) to play with files.

`m = Manifest.model_validate(sample())`
Takes the dummy dictionary, checks that all the data follows the strict rules of a `Manifest`, and saves the result as `m`.

`dataset_id = write_manifest(tmp_path, m)`
Saves that valid manifest to a file in the temporary folder and captures a generated ID.

`m2, id2 = read_manifest(tmp_path)`
Reads the file back out of the temporary folder into new variables (`m2` and `id2`).

`assert m2 == m and id2 == dataset_id`
Checks that the newly loaded data perfectly matches the original data, and the IDs match. If they don't, the test fails.

`raw = (tmp_path / "dataset.manifest").read_bytes()`
Reads the raw byte data of the saved file directly from the hard drive.

`assert raw == canonical_json_pretty(sample()) and raw.endswith(b"}\n")`
Checks that the raw bytes are perfectly formatted with the custom JSON formatter and end with a closing bracket and a new line.

`data = sample()`
Grabs a fresh copy of the dummy data dictionary to run new tests on.

`data["format_version"] = [1, 3]`
Changes the data format version to a higher minor version (1.3).

`data["future_field"] = {"x": 1}`
Injects a fake, unknown data field into the main dictionary.

`data["build"]["also_new"] = "y"`
Injects another fake, unknown field inside the build section.

`assert parse_manifest(json.dumps(data).encode()).name == "demo"  # higher minor: ignored`
Converts the messy data to bytes, reads it in, and verifies the name is still "demo", proving the system successfully ignores unknown fields if the version is newer than 1.0.

`data["format_version"] = [1, 0]`
Sets the data format version back to 1.0.

`with pytest.raises(ManifestError):`
Tells the test runner to expect the next line to trigger a specific error (`ManifestError`).

`parse_manifest(json.dumps(data).encode())  # minor 0: unknown keys are errors`
Attempts to read the data containing the fake fields, which intentionally causes an error because version 1.0 is strict and forbids unknown fields.

`data["format_version"] = [2, 0]`
Changes the data format version to a completely new major version (2.0).

`with pytest.raises(ManifestError):`
Tells the test runner to expect another error on the next line.

`parse_manifest(json.dumps(data).encode())`
Attempts to read version 2.0 data, which triggers an error because the current software cannot handle major version 2.

`with pytest.raises(ManifestError):`
Tells the test runner to expect another error for the upcoming block.

`parse_manifest(`
Starts a function call to read manifest data from bytes.

`json.dumps({**sample(), "build": {**sample()["build"], "python": 3.12}}).encode()`
Takes the dummy data, changes the Python version from a correct text string ("3.12.14") to an incorrect decimal number (3.12), converts it to bytes, and feeds it to the parser, which triggers a validation error.

`)`
Closes the function call.

`(tmp_path / "dataset.manifest").write_bytes(json.dumps(sample()).encode())`
Takes the original dummy data, converts it using the standard, unformatted JSON tool, and forcefully writes it to the temporary file, overwriting the perfectly formatted version.

`with pytest.raises(ManifestError, match="canonical"):`
Tells the test runner to expect an error specifically containing the word "canonical".

`read_manifest(tmp_path)`
Tries to open the overwritten file, triggering the expected error because the file is not formatted in the strictly required "canonical" layout.

#### `tests/unit/test_config.py`

> Imports and Setup

**`import pytest`**
Imports the pytest testing tool, which is used to run all the tests in this file.

**`from helpers.build import fixture_config`**
Brings in a helper tool named `fixture_config` from another file in the project called `helpers.build`.

**`from wiki2shards.config import BuildConfig, ConfigError, load_build_config, load_mix_config`**
Brings in specific configuration blueprints (classes) and functions from the project's main configuration file so they can be tested.

> Testing if a Configuration Loads Correctly

**`def test_fixture_config_loads():`**
Starts a test function to check if a standard, pre-made configuration file loads as expected.

**`    cfg = load_build_config("configs/fixture.yaml")`**
Reads a configuration file named `fixture.yaml` and saves all its settings into a variable called `cfg`.

**`    assert cfg.pack.row_len == 65 and cfg.pack.effective_last_row == "drop"`**
Checks two things: that the loaded setting for row length is exactly 65, and that the rule for the last row is set to "drop". If either is false, the test fails.

**`    assert cfg.clean.soft_filters.enabled() == [`**
Starts a check to guarantee that the list of active cleaning filters exactly matches the list on the following lines.

**`        "disambiguation",`**
The first filter that must be active.

**`        "list_page",`**
The second filter that must be active.

**`        "max_symbol_pct",`**
The third filter that must be active.

**`        "min_words",`**
The fourth filter that must be active.

**`    ]`**
Ends the list and finishes the check for the active filters.

> Testing if the Program Rejects Bad Configurations

**`@pytest.mark.parametrize(`**
Starts a special pytest command that will run the upcoming test multiple times, feeding it a different piece of bad data each time.

**`    "patch",`**
Names the variable that will hold the bad piece of data being injected into the test.

**`    [`**
Starts the list of bad data scenarios to test.

**`        {"source": {"wiki": "x", "date": 20260901}},  # unquoted YAML date -> int`**
The first bad scenario: providing a date as a raw number instead of text. The comment notes why it is bad.

**`        {"pack": {"seq_len": 8, "typo": 1}},  # unknown key`**
The second bad scenario: sneaking in an unrecognized setting named "typo".

**`        {"pack": {"mode": "bfd", "seq_len": 8, "last_row": "drop"}},`**
The third bad scenario: providing a group of "pack" settings that conflict or are invalid together.

**`        {"select": {"apply": ["list_page"]}},  # filter clean does not compute`**
The fourth bad scenario: trying to use a filter in a place where it doesn't belong.

**`        {"select": {"val_per_10k": 10_001}},`**
The fifth bad scenario: asking for a validation rate of 10,001 out of 10,000, which is mathematically impossible.

**`    ],`**
Ends the list of bad data scenarios.

**`)`**
Finishes the pytest command for running multiple tests.

**`def test_config_rejects(patch):`**
Starts a test function that takes one of those bad pieces of data (the `patch`) as its input.

**`    base = {`**
Starts creating a dictionary of perfect, working base settings.

**`        "name": "x",`**
Sets the configuration name to "x".

**`        "source": {"wiki": "x", "date": "20260901"},`**
Sets the source settings, using a valid text-based date.

**`        "workdir": "w",`**
Sets the working folder path to "w".

**`        "output": "o",`**
Sets the output folder path to "o".

**`        "tokenizer": "tiktoken:gpt2",`**
Sets the text-reading tool to use a specific standard called "gpt2".

**`        "pack": {"seq_len": 8},`**
Sets the sequence length rule to 8.

**`        "clean": {"soft_filters": {"list_page": False}},`**
Turns off the "list_page" filter in the base settings.

**`    }`**
Ends the dictionary of perfect base settings.

**`    base.update(patch)`**
Takes the perfect base settings and overwrites them with the bad data from the current test scenario.

**`    with pytest.raises(ValueError):`**
Tells the test to expect a specific type of error (`ValueError`) to happen on the very next line. If the error doesn't happen, the test fails.

**`        BuildConfig.model_validate(base)`**
Asks the program to validate the newly corrupted settings, which should trigger the expected error.

> Testing How Configurations are Fingerprinted (Hashed)

**`def test_config_hash_ignores_execution_fields(tmp_path, dumps):`**
Starts a test to prove that changing where or how the program runs won't change its unique digital fingerprint (its hash).

**`    gpt2 = {"tokenizer": "tiktoken:gpt2"}  # hashing the config never loads the tokenizer`**
Creates a small shortcut variable holding the tokenizer setting. The comment explains that the fingerprinting process doesn't actually turn the tool on.

**`    a = fixture_config(tmp_path / "a", dumps, **gpt2)`**
Creates a first configuration object, pointing it to a folder named "a".

**`    b = fixture_config(tmp_path / "b", dumps, workers=7, **gpt2)`**
Creates a second configuration object, pointing it to folder "b" and telling it to use 7 worker processes.

**`    c = fixture_config(tmp_path / "c", dumps, pack__seq_len=32, **gpt2)`**
Creates a third configuration object, pointing it to folder "c" and fundamentally changing a core setting (the sequence length).

**`    assert a.config_sha256() == b.config_sha256()  # workers, workdir, output, mirror excluded`**
Checks that the digital fingerprints of configuration "a" and "b" are exactly identical, because folders and worker counts are ignored when creating the fingerprint.

**`    assert a.config_sha256() != c.config_sha256()`**
Checks that the fingerprints of "a" and "c" are different, because changing a core setting like sequence length must change the fingerprint.

> Testing Error Handling for Mix Configurations

**`def test_mix_config_errors(tmp_path):`**
Starts a test to check if the program correctly rejects a bad "mix" configuration file.

**`    p = tmp_path / "m.yaml"`**
Creates a temporary file path named `m.yaml`.

**`    p.write_text("seed: 1\nsources:\n  - {name: a, path: x, weight: 0}\n")`**
Writes raw text into that temporary file to mimic a bad configuration (specifically, it breaks a rule by setting a weight to 0).

**`    with pytest.raises(ConfigError):`**
Tells the test to expect a custom `ConfigError` to happen on the next line.

**`        load_mix_config(p)`**
Tries to load the bad file into the system, which should immediately trigger the expected error.

#### `tests/unit/test_tokenizer_fp.py`

> Importing Required Tools

`import json`
This brings in Python's built-in tool for working with JSON (JavaScript Object Notation), which is a common way to format data.

`from wiki2shards.tokenize.base import fingerprint_hf_json, fingerprint_tiktoken`
This grabs two specific functions (`fingerprint_hf_json` and `fingerprint_tiktoken`) from another file in this project so they can be used in this script. These functions are used to create unique digital IDs (fingerprints) for data.

> First Test: Checking JSON Fingerprints

`def test_hf_fingerprint_is_canonical():`
This defines a new test function. The word "canonical" means the standard or simplest form. This test checks if the fingerprinting tool is smart enough to recognize when two pieces of data mean the exact same thing, even if they look slightly different.

`a = json.dumps({"b": 1, "a": [1, 2]}).encode()`
This creates a piece of data (a dictionary where "b" is 1 and "a" is a list containing 1 and 2), turns it into text using JSON, converts that text into raw computer bytes, and stores it in a variable named `a`.

`b = json.dumps({"a": [1, 2], "b": 1}, indent=4).encode()`
This creates the exact same data as above, but with the items in a different order and formatted with 4 spaces of indentation so it is easier for a human to read. It then turns this into bytes and stores it in a variable named `b`.

`assert fingerprint_hf_json(a) == fingerprint_hf_json(b)`
This is the actual test. It checks (`asserts`) that the unique digital fingerprint for `a` is exactly the same as the fingerprint for `b`. If they match, the test passes, proving the tool correctly ignores superficial formatting differences.

> Second Test: Checking a Known "Golden" Value

`def test_fingerprint_goldens():`
This defines a second test function. "Goldens" refers to known, perfectly correct results that we want to compare against to ensure our code hasn't accidentally broken.

`ranks = {bytes([i]): i for i in range(256)} | {b"ab": 256}`
This creates a dictionary of basic rules for a text tokenizer (a tool that chops text into smaller pieces). It maps every single individual byte (numbers 0 through 255) to itself, and then adds one extra rule mapping the letters "ab" to the number 256. It saves this dictionary in a variable called `ranks`.

`assert fingerprint_tiktoken("tiny", r"\S+|\s+", {"<|endoftext|>": 257}, ranks) == (`
This calls the `fingerprint_tiktoken` function using some specific test settings (a name "tiny", a pattern rule, a special token dictionary, and the `ranks` we just made). It then checks if the result equals what comes on the next line.

`"57fd155206ade5e0dff18a8ccb6e13b457bf88fab53aa5385d196a10d4491fb4"`
This is the "golden" value—a specific, mathematically generated string of letters and numbers (a hash). If the fingerprint function creates this exact text, the test passes.

`)`
This simply closes the parenthesis from the `assert` statement that started two lines above.

### Step 1.9: DESIGN.md, formats chapter

    # DESIGN
    ## 1. On-disk formats (frozen at contracts-v1)
    ### .bin            n_rows × row_len tokens, row-major, <u2 or <u4, header-less (memmap at offset 0)
    ### .idx            128-byte header `<8sHHBBBBIIQQQQ32s32s8s` (table of every field), then row_seg_ptr <u8[n+1],
                    seg_doc_uid <u8, seg_start/seg_len/seg_piece <u4; size = 128 + 8(n_rows+1) + 20 n_segments
    ### dataset.manifest  canonical JSON (sort_keys, indent 2, UTF-8, trailing newline, ints only); dataset_id = sha256(bytes)
    ### docs.jsonl      one canonical line per selected doc, sorted by doc_uid; doc_uid as f"{uid:016x}"
    ### token cache     .tok.idx header `<8sHHB3xQ`, doc_uid <u8[n], offsets <u8[n+1]
    ### Versioning      major must match; higher minors use reserved bytes (.idx) or extra keys (manifest) only
    ## 2. Invariants verify checks (and the A4/A5 rules)

> DESIGN
> This marks the beginning of the design notes or documentation.

> 1. On-disk formats (frozen at contracts-v1)
> This introduces the section about how files are saved and organized on the computer's hard drive. "Frozen at contracts-v1" means these file structures are permanently locked in place for version 1 and cannot be changed.
> 
> 

> .bin            n_rows × row_len tokens, row-major, <u2 or <u4, header-less (memmap at offset 0)
> This describes the layout of a `.bin` (binary) file, which stores raw data. It stores data like a grid (number of rows multiplied by the length of each row), reading left-to-right, row-by-row. The data is saved using small numbers (2-byte or 4-byte integers). "Header-less" means there is no introductory text at the top of the file, allowing the computer to instantly load the raw data straight into memory starting from the very first byte.

> .idx            128-byte header <8sHHBBBBIIQQQQ32s32s8s (table of every field), then row_seg_ptr <u8[n+1],
> This explains the first part of an `.idx` (index) file, which acts as a map to navigate the `.bin` file. It starts with a 128-byte introduction block (the jumble of letters defines the exact size and type of each piece of intro data). After that, it lists "pointers" (using 8-byte numbers) that tell the computer exactly where each new row begins.

> ```
>                 seg_doc_uid <u8, seg_start/seg_len/seg_piece <u4; size = 128 + 8(n_rows+1) + 20 n_segments
> 
> ```
> 
> 

This continues explaining the index file. It stores a unique 8-byte ID number for each section of a document. It also uses 4-byte numbers to record where each section starts, how long it is, and which piece it is. Finally, it provides the exact mathematical formula to calculate the total size of the file in bytes.

> dataset.manifest  canonical JSON (sort_keys, indent 2, UTF-8, trailing newline, ints only); dataset_id = sha256(bytes)
> This describes a `dataset.manifest` file, which is a text file that lists the dataset's details. It must follow strict, standard formatting rules: alphabetical order for its labels, 2-space indentations, standard text encoding (UTF-8), ending with an empty line, and only using whole numbers. The unique ID for the entire dataset is generated by putting the file's contents through a standard mathematical fingerprinting tool (SHA-256).

> docs.jsonl      one canonical line per selected doc, sorted by doc_uid; doc_uid as f"{uid:016x}"
> This explains a `docs.jsonl` file. It is a text file where every single line contains the complete, standardized data for one specific document. The lines are sorted in order by the document's unique ID. This ID is formatted as a 16-character string made of numbers and lowercase letters.

> token cache      .tok.idx header <8sHHB3xQ, doc_uid <u8[n], offsets <u8[n+1]
> This describes a temporary storage system designed for speed. Its index file (`.tok.idx`) has a specific introductory header, followed by a list of 8-byte document IDs, and then a list of "offsets" (bookmarks) that tell the computer exactly where to find the data for each specific document.

> Versioning      major must match; higher minors use reserved bytes (.idx) or extra keys (manifest) only
> This explains the rules for software updates. "Major" updates (like version 1.0 vs 2.0) must be an exact match for the files to work together. "Minor" updates (like version 1.1 vs 1.2) are allowed to work together, but they can only store new information in the empty, leftover spaces of the `.idx` files or add new items to the manifest file without breaking the old layout.

> 2. Invariants verify checks (and the A4/A5 rules)
> This introduces the second main section. "Invariants" are strict rules that must always be true to guarantee the data isn't corrupted or broken. This line means the system will run tests to confirm these strict rules are met, including specific rules internally named "A4" and "A5".
> 
>

#### Validation

> Command 1: Running Specific Tests

**`uv run pytest -q tests/unit/test_io_binfmt.py tests/unit/test_io_manifest.py tests/unit/test_config.py tests/unit/test_tokenizer_fp.py`**

* **`uv run`**: Tells `uv` (a fast Python tool manager) to execute the command inside your project's isolated environment, ensuring it has all the right dependencies.
* **`pytest`**: The tool being executed, which is used to run Python tests.
* **`-q`**: Stands for "quiet." It tells pytest to print less text on the screen so the results are cleaner and easier to read.
* **`tests/unit/...`**: The specific paths to four different Python files. Instead of running every test in the entire project, this tells pytest to only run the tests located inside these four specific files.

> Command 2: Code Quality and Type Checking

**`uv run ruff check . && uv run ruff format --check . && uv run mypy src`**

This line links three separate tasks together using `&&`. The `&&` symbol means "only run the next command if the previous one succeeded without any errors."

* **`uv run ruff check .`**: Uses `ruff` (a fast code checker) to scan every file in your current folder (represented by the `.`) to find coding mistakes, bugs, or rule violations.
* **`uv run ruff format --check .`**: Uses `ruff` to verify that your code is formatted correctly (proper spacing, line breaks, etc.). The `--check` flag means it will only tell you if the formatting is wrong, but it will not actively rewrite or fix your files.
* **`uv run mypy src`**: Uses `mypy` (a type checker) to scan the code inside your `src` (source) folder. It ensures you are using data types correctly based on your type hints, such as ensuring you aren't trying to do math on a word instead of a number.

## Phase 2: Download and extract (M2)

### Step 2.1: Download with Range resume and SHA-1 verification

#### `src/wiki2shards/acquire/download.py`

> Module Overview and Setup

* `"""Fetch a dated multistream dump..."""`: A comment block describing what the script does: it safely downloads Wikipedia data, checks it for corruption, and resumes automatically if interrupted.
* `from __future__ import annotations`: Allows modern type hinting features to work in older versions of Python.
* `import os`: Imports the operating system module to interact with files and disks.
* `import time`: Imports the time module to allow the script to pause or sleep.
* `import urllib.request`: Imports Python's built-in tool for fetching data from the internet.
* `from collections.abc import Iterator`: Imports a type hint indicating that a function yields items one by one in a sequence.
* `from pathlib import Path`: Imports a tool to handle file paths easily across different operating systems (Windows, Mac, Linux).
* `from typing import Protocol`: Imports Protocol, which is used to create a blueprint that other classes must strictly follow.
* `from wiki2shards import __version__`: Imports the current version number from the main project.
* `from wiki2shards.io.atomic import commit`: Imports a custom function that safely renames a file only when it is fully finished downloading.
* `from wiki2shards.io.hashing import sha1_file`: Imports a custom function to calculate a file's fingerprint (SHA-1) to ensure it isn't corrupted.
* `from wiki2shards.pipeline import faults`: Imports a testing tool that intentionally triggers errors to make sure the script can recover.

> Constants

* `USER_AGENT = f"wiki2shards/{__version__} (+[https://github.com/BGSTA9/wiki2shards](https://github.com/BGSTA9/wiki2shards))"`: Creates an identification badge so the Wikipedia servers know exactly what software is downloading their files.
* `CHUNK = 1 << 20`: Sets the download chunk size to 1 Megabyte by shifting the number 1 twenty times in binary math.

> Custom Errors

* `class DownloadError(Exception):`: Creates a custom error specific to this program for when downloads fail.
* `pass`: Tells Python this class requires no further code inside it to work.

> Filename and URL Generators

* `def dump_names(wiki: str, date: str) -> dict[str, str]:`: Defines a function that takes a wiki name and a date, returning a dictionary of standard file names.
* `stem = f"{wiki}-{date}"`: Combines the wiki name and date into a base string (like "enwiki-20231001").
* `return {`: Starts returning a dictionary.
* `"sha1sums": f"{stem}-sha1sums.txt",`: Names the text file containing the file fingerprints.
* `"index": f"{stem}-pages-articles-multistream-index.txt.bz2",`: Names the compressed index file.
* `"xml": f"{stem}-pages-articles-multistream.xml.bz2",`: Names the main compressed XML data file.
* `}`: Closes the dictionary.
* `def source_url(mirror: str, wiki: str, date: str, name: str) -> str:`: Defines a function to build the full web address to download from.
* `return f"{mirror.rstrip('/')}/{wiki}/{date}/{name}"`: Glues the server address, wiki name, date, and file name together with slashes, removing any extra slashes at the start.

> Transport Interfaces

* `class Transport(Protocol):`: Defines a blueprint for any downloader class.
* `def open(self, url: str, start: int) -> tuple[Iterator[bytes], bool]:`: Mandates that downloaders must have an `open` method that returns data chunks and a true/false value for whether resuming worked.
* `"""Return (chunks, range_honored)..."""`: A comment explaining how the `open` method should behave.
* `...`: A placeholder meaning no actual code goes here; it is purely a rule for other classes to follow.

> HTTP Transport

* `class HttpTransport:`: Defines the downloader used for standard internet links.
* `def open(self, url: str, start: int) -> tuple[Iterator[bytes], bool]:`: Opens an internet link, trying to start at a specific byte if interrupted previously.
* `headers = {"User-Agent": USER_AGENT}`: Prepares our ID badge to send to the server.
* `if start:`: Checks if we are trying to resume a partially downloaded file.
* `headers["Range"] = f"bytes={start}-"`: If resuming, asks the server to only send data from that specific byte onward.
* `resp = urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=60)`: Connects to the server with our request, giving it 60 seconds to respond before giving up.
* `honored = start > 0 and resp.status == 206`: Checks if the server actually agreed to resume the download (HTTP status 206 means "Partial Content").
* `def chunks() -> Iterator[bytes]:`: Defines an inner function to read the incoming data piece by piece.
* `with resp:`: Ensures the internet connection is properly closed when we are finished.
* `while chunk := resp.read(CHUNK):`: Reads the incoming file 1 Megabyte at a time until there is no more data left.
* `yield chunk`: Hands over that 1 Megabyte chunk to whatever called it, pausing until the next piece is needed.
* `return chunks(), honored`: Returns the data-yielding function and the true/false value of whether the resume was successful.

> Local Transport

* `class LocalTransport:`: Defines a downloader for local files on the computer (mostly used for testing).
* `"""A mirror that is a local directory..."""`: Comment explaining what this class is for.
* `def open(self, url: str, start: int) -> tuple[Iterator[bytes], bool]:`: Opens a local file path.
* `def chunks() -> Iterator[bytes]:`: Inner function to read the local file piece by piece.
* `with open(url, "rb") as f:`: Opens the local file in read-binary mode.
* `f.seek(start)`: Skips ahead in the file to the starting byte.
* `while chunk := f.read(CHUNK):`: Reads the file in 1 Megabyte chunks.
* `yield chunk`: Hands over the chunk.
* `return chunks(), True`: Returns the data and confirms it successfully skipped ahead (local files always support resuming).

> Transport Selector

* `def transport_for(mirror: str) -> Transport:`: A function that picks the right downloader based on the address provided.
* `return HttpTransport() if mirror.startswith(("http://", "https://")) else LocalTransport()`: Uses the HTTP downloader if the address is a web link, otherwise uses the local file reader.

> Main Download Logic

* `def fetch(url: str, dest: Path, transport: Transport, expected_sha1: str | None, retries: int = 3) -> None:`: The main function that downloads the file, retries if it fails, checks its fingerprint, and saves it.
* `part = dest.with_name(f".{dest.name}.part")`: Creates a temporary, hidden filename ending in `.part` so we don't accidentally overwrite a good file with a broken download.
* `for attempt in range(retries + 1):`: Starts a loop to try downloading, giving it a set number of retries if it fails.
* `start = part.stat().st_size if part.exists() else 0`: Checks if the temporary file already exists to see how much we've already downloaded; otherwise starts at 0.
* `try:`: Starts a block of code to catch and handle errors safely.
* `chunks, honored = transport.open(url, start)`: Asks the transport to open the file and checks if it can resume.
* `with open(part, "ab" if honored and start else "wb") as f:`: Opens the temporary file. It appends to it (`"ab"`) if resuming worked, otherwise it overwrites it from the beginning (`"wb"`).
* `for i, chunk in enumerate(chunks):`: Loops through the downloaded chunks one by one, keeping track of the count.
* `f.write(chunk)`: Saves the downloaded chunk to the computer.
* `if i == 0:`: Checks if this is the very first chunk being downloaded.
* `f.flush()`: Forces the computer to clear its memory and write the data to the disk immediately.
* `faults.hit("download", "mid_file")`: A testing hook that intentionally breaks the download to ensure the resume feature works.
* `f.flush()`: Forces any remaining data in memory to write to disk.
* `os.fsync(f.fileno())`: Double-checks directly with the operating system that all data is physically recorded on the hard drive.
* `break`: The download succeeded entirely, so it breaks out of the retry loop.
* `except OSError as exc:`: Catches any network drops or file errors that happen during the download.
* `if attempt == retries:`: Checks if we have run out of our allowed retries.
* `raise DownloadError(f"{url}: {exc}") from exc`: If out of retries, it stops the program and raises our custom error with the details.
* `time.sleep(2**attempt)`: If we still have retries, pauses for an increasing amount of time (1s, 2s, 4s...) before trying again (this is called exponential backoff).
* `if expected_sha1 is not None:`: Checks if we were given a fingerprint to verify the file's integrity.
* `got = sha1_file(part)`: Calculates the fingerprint of our newly downloaded temporary file.
* `if got != expected_sha1:`: Compares our file's fingerprint to the expected one.
* `part.unlink()`: If they don't match, it deletes the corrupted temporary file.
* `raise DownloadError(f"{dest.name}: SHA-1 {got} != expected {expected_sha1}")`: Raises an error stating the file was corrupted and failed the check.
* `commit(part, dest)`: If the download finished and the fingerprint matches perfectly, it safely renames the temporary `.part` file to the final, permanent file name.

> File Parsing Logic

* `def parse_sha1sums(text: str) -> dict[str, str]:`: Defines a function to read the text inside the SHA-1 fingerprint file.
* `out = {}`: Creates an empty dictionary to hold our extracted data.
* `for line in text.splitlines():`: Loops through the text file line by line.
* `if line.strip():`: Ignores any completely blank lines.
* `sha1, name = line.split(maxsplit=1)`: Splits the line into exactly two parts: the fingerprint and the file name.
* `out[name.strip().lstrip("*")] = sha1.lower()`: Cleans up spaces or weird characters (like asterisks) from the file name, converts the fingerprint to lowercase, and saves them in the dictionary.
* `return out`: Returns the finished dictionary mapping file names to their unique fingerprints.

### Step 2.2: Stream geometry: offsets, units, siteinfo

#### `src/wiki2shards/extract/multistream.py`

Here is a plain and simple English explanation of your code, line by line.

> File Overview and Description

`"""Multistream dump geometry: stream offsets from the index, and units of consecutive streams.`
This is the start of a multi-line comment (docstring) that explains the file's purpose: it maps out how data is structured inside a large compressed file by looking at its index.

`Unit boundaries depend on the index and the target unit size only, never on the worker count,`
This line explains that the chunks of data (Units) are divided based on their size, not based on how many computer processors are reading them.

`which is what makes extract output identical for any number of workers (G1).`
This ensures that the final extracted data looks exactly the same whether you use one computer processor or many.

`"""`
This closes the multi-line comment.

> Importing Tools

`from __future__ import annotations`
This allows the code to use modern Python features (like hinting what type of data a variable holds) even if it's running on a slightly older version of Python.

`import bz2`
This loads Python's built-in tool for reading and writing heavily compressed files (specifically files ending in `.bz2`).

`import xml.etree.ElementTree as ET`
This loads a tool for reading XML data (a type of structured text) and gives it a shorter nickname, "ET", to save typing later.

`from dataclasses import dataclass`
This imports a special tool that makes it quick and easy to create "classes" (custom objects) that are strictly meant for storing data.

`from pathlib import Path`
This loads a tool that makes it easy for the code to read file and folder paths on your computer.

`from typing import Any`
This imports the word "Any", which is used to tell Python that a specific variable is allowed to hold any type of data whatsoever.

> Creating the Data "Unit" Container

`@dataclass(frozen=True)`
This is a label (decorator) that automatically builds the following object for data storage. "frozen=True" means once you put data into this object, it is locked and cannot be changed.

`class Unit:`
This defines a new custom blueprint called `Unit` to represent one chunk of file data.

`    index: int`
This says the Unit will have an `index` (like a sequence or ID number), and it must be a whole number (an integer).

`    start: int  # byte offset of the unit's first stream`
This line stores the exact starting position (in bytes) where this chunk of data begins in the file. The text after the `#` is a developer note explaining this.

`    end: int  # next unit's first offset, or the file size for the last unit`
This stores the ending position (in bytes) where this chunk stops. The developer note says it's either where the next chunk begins, or the very end of the file.

`    n_streams: int`
This stores how many smaller pieces of data (streams) are packaged inside this larger chunk.

`    @property`
This label allows the following function to be accessed simply like a regular variable, rather than having to trigger it as an action.

`    def name(self) -> str:`
This defines a function named `name` that will output a text string.

`        return f"part-{self.index:05d}"`
This line returns the word "part-" followed by the chunk's ID number, automatically adding leading zeros so it is always 5 digits long (for example: "part-00001").

> Function: Finding the Data Starts

`def stream_offsets(index_bz2: Path | str) -> list[int]:`
This defines an action named `stream_offsets`. You give it a file path, and it will return a list of whole numbers.

`    """Sorted unique stream starts from offset:page_id:title lines."""`
This comment explains that the function looks through an index file to find the unique starting positions of data streams.

`    offsets: set[int] = set()`
This creates an empty mathematical group (a set) named `offsets`. Sets are useful because they automatically ignore duplicate numbers.

`    last = -1`
This creates a tracking variable named `last` and sets it to -1. It will be used to remember the last number we looked at.

`    with bz2.open(index_bz2, "rt", encoding="utf-8") as f:`
This opens the compressed `.bz2` index file in "read text" mode (translating the computer code into standard readable text) and temporarily names the open file `f`.

`        for line in f:`
This starts a loop that will read the open file, one line at a time.

`            off = int(line.split(":", 1)[0])`
This takes the current line, chops it into pieces at the first colon (`:`), grabs the very first piece, and turns it into a whole number (the offset).

`            if off != last:  # lines are grouped by stream; skip the set lookup`
This checks if the new number we just found is different from the previous number we looked at.

`                offsets.add(off)`
If the number is different, this adds the new number to our set of starting positions.

`                last = off`
This updates our `last` tracker to be the current number, so we can compare it to the next line.

`    return sorted(offsets)`
After the loop finishes, this takes all the unique starting positions, sorts them from smallest to largest, and gives the list back to you.

> Function: Grouping Data into Units

`def plan_units(offsets: list[int], file_size: int, target_bytes: int) -> list[Unit]:`
This defines a function called `plan_units`. You give it the list of start positions, the total size of the file, and how big you want your chunks to be. It will return a list of `Unit` objects.

`    """Greedy: add streams in index order; close the unit once it reaches target_bytes."""`
This comment explains the strategy: it will bundle small data streams together until the bundle hits your target size, then it will start a new bundle.

`    if not offsets:`
This checks if the list of starting positions is totally empty.

`        return []`
If the list is empty, it stops immediately and gives you back an empty list.

`    bounds = [*offsets, file_size]`
This creates a new list called `bounds`. It takes all your starting positions and attaches the total file size to the very end of the list.

`    units: list[Unit] = []`
This creates an empty list called `units` that will eventually hold our finished bundles of data.

`    start_i = 0`
This creates a pointer named `start_i` and sets it to 0. It tracks where our current bundle begins.

`    for i in range(len(offsets)):`
This starts a loop that will count through every single starting position in our list.

`        if bounds[i + 1] - bounds[start_i] >= target_bytes or i == len(offsets) - 1:`
This checks two things: 1) Is the data between our start pointer and the next pointer bigger than our target size? OR 2) Are we looking at the very last item in the list?

`            units.append(Unit(len(units), bounds[start_i], bounds[i + 1], i + 1 - start_i))`
If either condition is true, it creates a new `Unit` bundle with an ID number, a start byte, an end byte, and the number of streams inside it, and adds it to our `units` list.

`            start_i = i + 1`
This moves our start pointer forward so we are ready to build the next bundle.

`    # The last unit runs to the end of the file: it includes the </mediawiki> footer stream.`
This developer note explains that the very last bundle needs to stretch all the way to the end of the file to grab the final closing tags.

`    last = units[-1]`
This pulls the very last `Unit` bundle out of our list so we can modify it.

`    units[-1] = Unit(last.index, last.start, file_size, last.n_streams)`
This replaces that last bundle with a brand new one that has the exact same information, except its ending position is set to the total size of the whole file.

`    return units`
Finally, this hands back the finished list of grouped data chunks.

> Function: Cleaning Up XML Tags

`def local_name(tag: str) -> str:`
This defines a function named `local_name` that takes a text string (an XML tag) and returns a cleaned-up text string.

`    """'{[http://www.mediawiki.org/xml/export-0.11/](http://www.mediawiki.org/xml/export-0.11/)}page' -> 'page' (export versions vary)."""`
This comment shows an example of what it does: it removes the messy website addresses (namespaces) from XML tags so you are left with just the simple word.

`    return tag.rsplit("}", 1)[-1]`
This splits the text from the right side at the closing curly bracket `}`, throws away everything before it, and returns just the clean word.

> Function: Reading Website Settings

`def read_siteinfo(xml_bz2: Path | str, first_offset: int) -> dict[str, Any]:`
This defines a function named `read_siteinfo`. You give it a file path and the first offset number, and it will give you back a dictionary (a list of names and their corresponding values).

`    """Parse dbname and namespaces from the header region; assert it holds no <page>."""`
This comment explains that it reads the very beginning of the file to find the website's database name, and makes sure it hasn't accidentally wandered into the actual article text.

`    with open(xml_bz2, "rb") as f:`
This opens the compressed file in "read binary" mode (reading raw computer data rather than text) and temporarily names it `f`.

`        raw = bz2.decompress(f.read(first_offset))`
This reads only the exact amount of bytes up to the `first_offset`, and decompresses that small piece of the file into raw text.

`    parser: ET.XMLPullParser[ET.Element] = ET.XMLPullParser(events=("start", "end"))`
This creates an automated XML reader (parser) that will notify us every time it finds the start or the end of a data tag.

`    parser.feed(raw)`
This feeds the decompressed text into the automated reader so it can start scanning it.

`    info: dict[str, Any] = {"dbname": None, "namespaces": {}}`
This sets up an empty dictionary called `info` that will store the database name and a sub-list for namespaces.

`    for item in parser.read_events():`
This starts a loop that will look at every single tag event (start or end) the reader found.

`        event, elem = item[0], item[-1]  # typeshed: some events are 1-tuples`
This unpacks the reader's notification, putting the action ("start" or "end") into `event`, and the actual data tag into `elem`.

`        if not isinstance(elem, ET.Element):`
This checks to ensure the tag we pulled out is actually a valid XML data element.

`            continue`
If it isn't valid, this tells the loop to skip the rest of the steps and move to the next item.

`        name = local_name(elem.tag)`
This runs the tag through our `local_name` function to clean off the messy website addresses, giving us a simple name.

`        if event == "start" and name == "page":`
This checks if the reader just found the start of an article "page".

`            raise ValueError("header region before the first indexed offset contains a <page>")`
If it did find a page, this triggers a crash/error with a message, because article pages are not supposed to be in the header section.

`        if event == "end" and name == "dbname":`
This checks if the reader just reached the end of the "dbname" (database name) tag.

`            info["dbname"] = (elem.text or "").strip()`
If it did, this extracts the text inside the tag, cuts away any extra blank spaces on the sides, and saves it to our `info` dictionary.

`        elif event == "end" and name == "namespace":`
Alternatively, if the reader just reached the end of a "namespace" tag...

`            info["namespaces"][str(int(elem.attrib["key"]))] = elem.text or ""`
...it grabs the numerical ID associated with that namespace, turns it into text, and uses it to save the namespace's name into our dictionary.

`    if not info["dbname"]:`
After scanning everything, this checks if we failed to find a database name.

`        raise ValueError("no <dbname> in the dump header")`
If the database name is missing, this crashes the program with an error message.

`    return info`
Finally, this hands back the completed `info` dictionary containing all the website's configuration data.

> Function: Saving Website Settings

`def write_siteinfo(xml_bz2: Path, first_offset: int, out_dir: Path, wiki: str) -> list[str]:`
This defines an action named `write_siteinfo`. You give it file paths, a start point, and the name of the wiki you expect to find. It will return a list of text strings (filenames).

`    from wiki2shards.io.atomic import atomic_write_bytes`
This imports a specialized custom tool that safely saves data to your hard drive without risk of corrupting it if the computer crashes.

`    from wiki2shards.io.hashing import canonical_json_pretty`
This imports another custom tool that formats data into clean, easy-to-read JSON text.

`    info = read_siteinfo(xml_bz2, first_offset)`
This uses the previous function we made (`read_siteinfo`) to extract the website settings and saves them in a variable called `info`.

`    if info["dbname"] != wiki:`
This checks if the database name we extracted does NOT match the wiki name you asked for.

`        raise ValueError(f"dump dbname {info['dbname']!r} does not match source.wiki {wiki!r}")`
If they don't match, this intentionally crashes the program to warn you that you are looking at the wrong file.

`    atomic_write_bytes(out_dir / "siteinfo.json", canonical_json_pretty(info))`
This takes the settings, formats them nicely, and safely writes them to a new file named "siteinfo.json" inside your output folder.

`    return ["siteinfo.json"]`
This gives you back a list containing the name of the file it just successfully created.

> Function: Reading a Specific Chunk of Data

`def read_unit_bytes(xml_bz2: Path | str, unit: Unit) -> bytes:`
This defines an action named `read_unit_bytes`. You hand it the file and a specific `Unit` bundle object, and it will return the raw, decompressed bytes of data.

`    """Decompress [start, end); bz2.decompress handles concatenated streams."""`
This comment explains that it pulls only the data starting and ending at the exact bytes specified by your Unit, and decompresses it.

`    with open(xml_bz2, "rb") as f:`
This opens the compressed file in "read binary" mode.

`        f.seek(unit.start)`
This fast-forwards the file reader straight to the exact starting byte of your specific bundle of data.

`        data = bz2.decompress(f.read(unit.end - unit.start))`
This reads only the exact amount of data that belongs in your bundle (by subtracting the start byte from the end byte), decompresses it, and saves it into `data`.

`    stripped = data.rstrip()`
This creates a version of the data where any invisible blank spaces or new lines at the very end are deleted.

`    if stripped.endswith(b"</mediawiki>"):`
This checks if the very end of our data contains the main closing tag `</mediawiki>`.

`        data = stripped[: -len(b"</mediawiki>")]`
If it does contain that tag, this cuts exactly that tag off the end of our data so the data chunk remains clean.

`    return data`
Finally, this hands back the fully processed, decompressed text for that specific bundle of data.

### Step 2.3: Page streaming and filters

#### `src/wiki2shards/extract/xmlpages.py`

Here is the line-by-line explanation of your Python script in plain and simple English.

> Introduction and Setup

```python
"""Stream <page> elements out of one unit and keep ns=0, non-redirect, wikitext pages.

Measured: a page parsed from one stream is tagged `page`, but parsed from the whole file it is
`{http://www.mediawiki.org/xml/export-0.11/}page`. Every tag is matched by local name.
"""

```

This is a multi-line comment (called a docstring) explaining what the script does. It reads Wikipedia data one piece at a time, keeping only standard articles (ns=0) that are not redirects to other pages, and are formatted in standard Wikipedia text. The second part is a note for the programmer explaining a technical quirk about how the XML tags are named depending on how the file is read.

```python
from __future__ import annotations

```

This enables a newer Python feature that allows the programmer to write hints about what type of data is being used (like saying a variable will hold a "string" or a "list") in a cleaner way, even if they are using an older version of Python.

```python
import xml.etree.ElementTree as ET
from collections.abc import Iterator
from pathlib import Path

```

These three lines bring in standard Python tools. `ET` is a tool for reading and navigating XML data (which is how Wikipedia exports its pages). `Iterator` is used to create lists of items one by one. `Path` makes it easier to work with computer file and folder paths.

```python
from wiki2shards.extract.multistream import Unit, local_name, read_unit_bytes
from wiki2shards.io.atomic import atomic_write_bytes
from wiki2shards.io.hashing import canonical_json_pretty
from wiki2shards.io.records import ExtractedPage, write_jsonl

```

These four lines load custom tools built specifically for this project (`wiki2shards`). They pull in tools to handle reading data chunks, safely saving files to the hard drive, formatting text nicely, and a specific blueprint for what an `ExtractedPage` should look like.

> Setting Up Constants

```python
_FEED = 1 << 20

```

This creates a constant setting named `_FEED` and sets it to exactly 1,048,576 (which is 1 Megabyte). This tells the program how much data to read into the computer's memory at one time.

```python
SKIP_REASONS = ("skipped_model", "skipped_namespace", "skipped_redirect")

```

This creates a permanent list of three reasons why a Wikipedia page might be ignored: it has the wrong format, it isn't a main article, or it's just a redirect link to another page.

> Helper Function: Finding a Single Item

```python
def _child(elem: ET.Element, name: str) -> ET.Element | None:

```

This creates a tool (function) named `_child`. You give it a piece of XML data and a name to look for. It promises to give you back either the specific piece of data you asked for, or nothing (`None`).

```python
    for c in elem:

```

This starts a loop to look at every single item inside the provided piece of XML data.

```python
        if local_name(c.tag) == name:

```

This checks if the name of the current item exactly matches the name we are searching for.

```python
            return c

```

If the names match, it immediately stops looking and hands that item back to you.

```python
    return None

```

If the loop checks everything and finds no match, it hands back `None` (meaning nothing was found).

> Helper Function: Finding Multiple Items

```python
def _children(elem: ET.Element, name: str) -> list[ET.Element]:

```

This creates another tool named `_children`. Like the last one, you give it XML data and a name to look for, but this one promises to give you back a list of *all* matching items.

```python
    return [c for c in elem if local_name(c.tag) == name]

```

This creates and hands back a list containing every item inside the XML data that matches the name we are looking for.

> Main Function: Reading the Pages

```python
def iter_pages(data: bytes, stats: dict[str, int]) -> Iterator[ExtractedPage]:

```

This creates a function named `iter_pages`. It takes raw data and a dictionary (a checklist for counting statistics) as inputs. It promises to spit out extracted Wikipedia pages one by one.

```python
    """Yield kept pages in stream order; count the rest by reason in `stats`."""

```

A comment explaining that this function will output the good pages in the order it finds them, and count up the bad pages based on why they were rejected.

```python
    parser: ET.XMLPullParser[ET.Element] = ET.XMLPullParser(events=("start", "end"))

```

This sets up a parser (a reader). It is configured to read data bit by bit and alert the program every time it sees the "start" of an XML tag or the "end" of an XML tag.

```python
    parser.feed(b"<root>")

```

This feeds a fake starting tag (`<root>`) into the parser so it knows a valid document is beginning.

```python
    root: ET.Element | None = None
    pos = 0

```

This creates a variable `root` (starting as nothing) to remember the very top level of the document, and a tracker called `pos` starting at 0 to remember exactly where we are in the data file.

```python
    while True:

```

This starts an endless loop that will keep running until we explicitly tell it to stop.

```python
        chunk = data[pos : pos + _FEED]

```

This slices out a 1-Megabyte chunk of raw data starting from our current position.

```python
        pos += len(chunk)

```

This moves our position tracker forward by the exact size of the chunk we just sliced out.

```python
        parser.feed(chunk if chunk else b"</root>")

```

This gives the chunk of data to the parser to read. If the chunk is empty (meaning we ran out of data), it feeds a fake ending tag (`</root>`) to finish the document.

```python
        for item in parser.read_events():

```

This starts a loop to go through every single alert (event) the parser found in the chunk we just fed it.

```python
            event, elem = item[0], item[-1]  # typeshed: some events are 1-tuples

```

This splits the parser's alert into two parts: the `event` (was it a start or end tag?) and the `elem` (the actual data attached to it). The comment explains a technical fix for how the parser sometimes packages this data.

```python
            if not isinstance(elem, ET.Element):
                continue

```

This checks if the data we found is actually an XML element. If it isn't, we skip the rest of the loop and move to the next item.

```python
            if event == "start":

```

This checks if the alert is saying a tag is starting.

```python
                if root is None:
                    root = elem

```

If it is a starting tag, and we haven't saved our top-level `root` yet, we save this element as the root.

```python
                continue

```

Since it's only a starting tag, we haven't read all the content inside it yet, so we skip the rest of the loop and keep reading.

```python
            if local_name(elem.tag) != "page":
                continue

```

This checks if the name of the tag is NOT "page". If it isn't a page, we don't care about it, so we skip it.

```python
            stats["pages"] = stats.get("pages", 0) + 1

```

Because we found a page, we increase our total "pages" counting statistic by 1.

```python
            page = _page(elem, stats)

```

We take the raw XML page data and hand it to another function called `_page` to figure out if it's a good page we want to keep.

```python
            if root is not None:
                root.clear()  # free processed pages; the parser still holds the open one

```

This deletes the old data from the computer's memory so it doesn't crash from getting too full. The comment notes that the parser is smart enough to keep whatever it still needs.

```python
            if page is not None:
                yield page

```

If the `_page` function successfully gave us a clean page (it wasn't skipped), we spit it out one at a time to whatever asked for it.

```python
        if not chunk:
            parser.close()
            return

```

If the chunk of data we tried to read was completely empty, it means we are at the end of the file. We turn off the parser and stop the function completely.

> Helper Function: Getting Text

```python
def _text(elem: ET.Element | None) -> str:

```

This creates a function named `_text` that takes an XML element (or nothing) and promises to give you back text (a string).

```python
    # Never write `_child(...) or default`: an Element without children is falsy.

```

A warning to other programmers explaining that they shouldn't try a specific Python shortcut here, because empty XML tags act weirdly in Python and could break the code.

```python
    return "" if elem is None else (elem.text or "")

```

If there is no element at all, it hands back an empty string. If there is an element, it hands back the text inside it. If the element exists but has no text, it also hands back an empty string.

> Processing Function: Filtering a Single Page

```python
def _page(elem: ET.Element, stats: dict[str, int]) -> ExtractedPage | None:

```

This creates a function named `_page` that takes raw XML page data and our counting statistics. It promises to either give back a clean `ExtractedPage` object or nothing (`None`).

```python
    ns = int(_text(_child(elem, "ns")) or "-1")

```

This searches for a tag called "ns" (namespace), gets the text inside it, and turns it into a number. If it can't find anything, it defaults to the number -1.

```python
    revisions = _children(elem, "revision")

```

This finds every tag called "revision" inside the page and puts them into a list.

```python
    if len(revisions) != 1:
        raise ValueError(f"page has {len(revisions)} revisions; pages-articles has exactly one")

```

This checks if the list of revisions has exactly 1 item. If it has zero or more than one, it crashes the program with an error because standard Wikipedia article files should only have exactly one revision per page.

```python
    rev = revisions[0]

```

This grabs the first (and only) revision from the list and saves it to a variable named `rev`.

```python
    model = _text(_child(rev, "model"))

```

This looks inside that revision for a tag called "model" and gets the text out of it.

```python
    reason = None

```

This creates a variable named `reason` that starts empty. We will use this to remember why we are skipping a page, if we decide to skip it.

```python
    if ns != 0:
        reason = "skipped_namespace"

```

If the "ns" (namespace) number is not 0 (0 means standard article), we set the reason to "skipped_namespace".

```python
    elif _child(elem, "redirect") is not None:
        reason = "skipped_redirect"

```

Otherwise, if we find a tag called "redirect" inside the page, we set the reason to "skipped_redirect".

```python
    elif model != "wikitext":
        reason = "skipped_model"

```

Otherwise, if the "model" text is anything other than "wikitext", we set the reason to "skipped_model".

```python
    if reason:
        stats[reason] = stats.get(reason, 0) + 1
        return None

```

If we recorded any reason to skip the page, we increase the specific counting statistic for that reason by 1, and we return nothing (thereby throwing the page away).

```python
    stats["kept"] = stats.get("kept", 0) + 1

```

If we get to this line, it means the page survived all our tests! We increase our "kept" statistic counter by 1.

```python
    return ExtractedPage(

```

We begin creating a new, clean `ExtractedPage` object to hand back.

```python
        page_id=int(_text(_child(elem, "id"))),

```

We find the page's "id" tag, turn its text into a whole number, and save it.

```python
        rev_id=int(_text(_child(rev, "id"))),

```

We find the revision's "id" tag, turn its text into a whole number, and save it.

```python
        title=_text(_child(elem, "title")),

```

We find the page's "title" tag and save its text.

```python
        wikitext=_text(_child(rev, "text")),  # missing or deleted text -> "" (dropped by clean)

```

We find the revision's "text" tag (which contains the actual article) and save it. The comment reminds us that if the text is empty, it just saves an empty string.

```python
    )

```

This closing parenthesis finishes building our clean `ExtractedPage` object.

> Main Entry Function: Processing an Entire Unit

```python
def extract_unit(xml_bz2: Path, unit: Unit, out_dir: Path) -> list[str]:

```

This creates a function named `extract_unit`. It takes a compressed file, a specific chunk of data (unit), and an output folder location. It promises to return a list of text names (the names of the files it creates).

```python
    """Write part-NNNNN.jsonl and part-NNNNN.stats.json into out_dir; return their names."""

```

A comment explaining that this function will create a data file and a statistics file in the output folder, and then tell you their names.

```python
    stats = {"pages": 0, "kept": 0, **dict.fromkeys(SKIP_REASONS, 0)}

```

This sets up our statistics scorecard. It creates a dictionary starting with zero for "pages", zero for "kept", and zero for all three of our skip reasons.

```python
    pages = list(iter_pages(read_unit_bytes(xml_bz2, unit), stats))

```

This line does the heavy lifting. It unzips the chunk of data, feeds it into our `iter_pages` function, gathers all the clean pages that get spit out, and turns them into a large list called `pages`.

```python
    write_jsonl(out_dir / f"{unit.name}.jsonl", pages)

```

This takes our list of good pages and writes them into a newly created `.jsonl` file inside our output folder.

```python
    atomic_write_bytes(out_dir / f"{unit.name}.stats.json", canonical_json_pretty(stats))

```

This takes our final scorecard of counting statistics, formats it to look pretty, and safely writes it into a newly created `.stats.json` file in the output folder.

```python
    return [f"{unit.name}.jsonl", f"{unit.name}.stats.json"]

```

Finally, this hands back a list containing the names of the two new files we just created.

### Step 2.4: Tests

#### `tests/unit/test_extract_multistream.py`

> **Imports and Settings**

`import bz2`
This brings in Python's built-in tool for reading and writing files compressed in the `.bz2` format.

`from wiki2shards.extract.multistream import (`
This begins a multi-line statement to bring in specific tools from a custom module (likely part of a project that processes Wikipedia data).

`    local_name,`
Imports a function that probably removes extra XML formatting namespaces to get the plain name of a tag.

`    plan_units,`
Imports a function used to figure out how to divide the large file into manageable chunks (units) for reading.

`    read_siteinfo,`
Imports a function designed to extract general information about the wiki from the beginning of the file.

`    read_unit_bytes,`
Imports a function that reads the raw data from a specific chunk of the compressed file.

`    stream_offsets,`
Imports a function that reads an index file to find exactly where each compressed chunk starts.

`)`
Closes the multi-line import statement.

`from wiki2shards.extract.xmlpages import iter_pages`
Brings in a function named `iter_pages` that processes raw XML data and extracts individual wiki pages one by one.

`NS = "{[http://www.mediawiki.org/xml/export-0.11/](http://www.mediawiki.org/xml/export-0.11/)}"`
Creates a constant variable named `NS` that holds the standard XML namespace string used by MediaWiki exports.

> **Helper Function for File Paths**

`def _paths(dumps):`
Defines a helper function named `_paths` that takes a directory path (called `dumps`) as an input.

`    d = dumps / "fixturewiki" / "20260901"`
Builds a specific folder path by adding "fixturewiki" and a date ("20260901") to the base `dumps` directory.

`    return (`
Starts a multi-line statement to return two file paths back to whatever called this function.

`        d / "fixturewiki-20260901-pages-articles-multistream.xml.bz2",`
Returns the full path to the compressed XML file containing the wiki pages.

`        d / "fixturewiki-20260901-pages-articles-multistream-index.txt.bz2",`
Returns the full path to the compressed index file that maps out the contents of the XML file.

`    )`
Closes the return statement.

> **The Main Test Function**

`def test_multistream_fragment(dumps):`
Defines a test function to verify that reading fragments of the multi-stream file works correctly.

`    xml, index = _paths(dumps)`
Calls the helper function above and assigns the returned XML file path to `xml` and the index file path to `index`.

`    offsets = stream_offsets(index)`
Reads the index file to calculate the starting points (offsets) for every compressed block inside the XML file.

`    size = xml.stat().st_size`
Checks the physical file system to get the total size of the XML file in bytes.

`    assert len(offsets) == 3  # 300 pages, 100 per stream`
Checks to guarantee that exactly 3 stream starting points were found; if not, the test crashes and fails.

`    # One stream parses on its own, with un-namespaced tags.`
A comment explaining that the next section will test reading just a single stream by itself.

`    single = plan_units(offsets, size, 1)`
Calculates a reading plan where the file is split into units containing exactly 1 stream each.

`    assert [u.n_streams for u in single] == [1, 1, 1]`
Checks to guarantee that the resulting plan actually created three separate units, each containing exactly one stream.

`    stats: dict[str, int] = {}`
Creates an empty dictionary named `stats` that will map text labels to numbers, used here to track counting statistics.

`    pages = list(iter_pages(read_unit_bytes(xml, single[1]), stats))`
Reads the raw data for the second chunk (index 1), extracts the pages from it, updates the `stats` dictionary, and saves the results as a list.

`    assert stats["pages"] == 100 and len(pages) == stats["kept"]`
Checks that exactly 100 pages were found in that chunk, and that the number of pages saved in the list matches the number the system claims it kept.

`    # Unit boundaries come from the index alone; one big unit sees the same pages in order.`
A comment explaining that the next section will prove you get the exact same data if you read the whole file in one giant chunk instead of small ones.

`    whole = plan_units(offsets, size, 1 << 30)`
Calculates a new reading plan where chunks can be up to 1 gigabyte in size (which is large enough to swallow the whole test file at once).

`    assert len(whole) == 1 and whole[0].end == size`
Checks that this new plan resulted in exactly one single chunk that spans all the way to the end of the file.

`    all_pages = list(iter_pages(read_unit_bytes(xml, whole[0]), {}))`
Reads the raw data for that giant chunk, extracts all the pages, and saves them into a list called `all_pages`.

`    by_unit = [p for u in single for p in iter_pages(read_unit_bytes(xml, u), {})]`
Uses a list comprehension to go through the file chunk by chunk, extract the pages from each, and merge them all together into a single list called `by_unit`.

`    assert all_pages == by_unit`
Checks to guarantee that the pages extracted in one giant sweep exactly match the pages extracted chunk-by-chunk.

`    # Header region: siteinfo, no pages. Footer: </mediawiki> stripped from the last unit.`
A comment explaining that the top of the file contains wiki metadata, and the bottom contains an ending tag that the custom reader automatically removes.

`    assert read_siteinfo(xml, offsets[0])["dbname"] == "fixturewiki"`
Reads the wiki metadata from the very beginning of the file and checks that the database name is "fixturewiki".

`    assert not read_unit_bytes(xml, single[-1]).rstrip().endswith(b"</mediawiki>")`
Reads the very last chunk in the file, trims any blank space at the end, and checks to make sure the standard `</mediawiki>` closing tag is missing (proving the system stripped it out).

`    # Tags match by local name: the same page parsed with its namespace still matches.`
A comment explaining the next test ensures XML tags are identified correctly even if they have cumbersome namespaces attached.

`    assert local_name(f"{NS}page") == "page" == local_name("page")`
Checks that the `local_name` function successfully turns a long, complex namespace tag into the simple word "page", and leaves an already-simple word as "page".

`    with open(xml, "rb") as f:`
Opens the XML file directly in binary reading mode so the underlying raw bytes can be accessed.

`        whole_file = bz2.decompress(f.read())`
Reads every byte of the file, uncompresses the entire thing all at once, and saves the resulting text into `whole_file`.

`    import xml.etree.ElementTree as ET`
Brings in Python's standard, built-in tool for reading and navigating XML data, renaming it `ET` for brevity.

`    root = ET.fromstring(whole_file)`
Feeds the giant uncompressed text string into the standard XML reader to create a navigable tree structure of the data.

`    assert root.tag == f"{NS}mediawiki"`
Checks that the very first, outermost tag of the XML tree is exactly the MediaWiki namespace string followed by "mediawiki".

`    assert sum(1 for e in root if local_name(e.tag) == "page") == 300`
Loops through every item directly inside the root tag, counts up exactly how many of them are named "page", and checks to guarantee that the final count is exactly 300.

#### `tests/unit/test_extract_pages.py`

> The Setup

**`import json`**
Brings in the built-in tool that allows Python to read and write JSON data (a standard way to store text data).

**`from wiki2shards.extract.multistream import plan_units, stream_offsets`**
Pulls in two specific tools (`plan_units` and `stream_offsets`) from a custom project called `wiki2shards`. These tools figure out how to break down a massive Wikipedia file into smaller, manageable chunks.

**`from wiki2shards.extract.xmlpages import extract_unit`**
Pulls in a tool that actually extracts the web pages from a chunk of the Wikipedia file.

**`from wiki2shards.io.records import iter_jsonl`**
Pulls in a tool that reads `.jsonl` files (where every single line is a separate JSON object) one line at a time.

> The Function Definition

**`def test_page_filters(dumps, tmp_path):`**
Creates a new testing function named `test_page_filters`. It takes two arguments: `dumps` (the folder where the test Wikipedia data is stored) and `tmp_path` (a temporary folder to save files during the test).

> Setting Up the File Paths

**`d = dumps / "fixturewiki" / "20260901"`**
Creates a path to the specific folder holding the fake Wikipedia test data (the "fixture") for a specific date.

**`xml = d / "fixturewiki-20260901-pages-articles-multistream.xml.bz2"`**
Creates a path to the compressed, giant XML file that holds the text of the Wikipedia pages.

> Figuring Out the Chunks

**`units = plan_units(`**
Starts using the `plan_units` tool to divide the giant file into smaller chunks, saving the plan in a variable named `units`.

**`stream_offsets(d / "fixturewiki-20260901-pages-articles-multistream-index.txt.bz2"),`**
Provides the first piece of information for the plan: it reads an index file that acts like a map, showing where pages are located inside the giant compressed file.

**`xml.stat().st_size,`**
Provides the second piece of information: the total file size of the giant Wikipedia XML file.

**`1,`**
Provides the third piece of information: the number of parts (or workers) to split this job into (in this case, just 1).

**`)`**
Closes the `plan_units` setup.

> Preparing the Scorecards

**`stats: dict[str, int] = {}`**
Creates an empty dictionary (a list of key-value pairs) called `stats` to keep a running tally of how many pages were processed, skipped, or saved.

**`kept = set()`**
Creates an empty set (a collection that only holds unique items) called `kept`. This will store the unique ID numbers of the Wikipedia pages the code decides to keep.

> Processing the Data

**`for u in units:`**
Starts a loop that goes through each chunk (or "unit") of the file that we planned out earlier.

**`extract_unit(xml, u, tmp_path)`**
Reads the current chunk (`u`) from the giant file (`xml`) and extracts the Wikipedia pages, saving the results in the temporary folder (`tmp_path`).

**`for k, v in json.loads((tmp_path / f"{u.name}.stats.json").read_text()).items():`**
Opens a small statistics file that was just created for this chunk, reads the text, turns it into a Python dictionary using `json.loads`, and starts a loop to look at every statistic (where `k` is the name of the stat, and `v` is the number).

**`stats[k] = stats.get(k, 0) + v`**
Adds the numbers from the current chunk's stats to the main `stats` tally we created earlier. If the stat doesn't exist yet, it treats the starting number as 0.

**`kept |= {p["page_id"] for p in iter_jsonl(tmp_path / f"{u.name}.jsonl")}`**
Opens the file containing the pages that were successfully extracted, finds the ID number (`page_id`) for each page, and adds all of those IDs into the `kept` set.

> Checking the Results

**`expected = json.loads((d / "fixture_expected.json").read_text())`**
Opens an "answer key" file called `fixture_expected.json`, reads it, and saves it into a variable named `expected`. This contains the exact results we *should* get if the code is working perfectly.

**`ids = expected["page_ids"]`**
Grabs the specific list of page IDs from the answer key and saves it to a shorter variable name, `ids`.

**`for skipped in ("redirect", "talk", "user", "non_wikitext"):`**
Starts a loop going through four specific categories of Wikipedia pages that the system is supposed to filter out and throw away.

**`assert kept.isdisjoint(ids[skipped]), skipped`**
Checks to make sure that absolutely none of the IDs from these unwanted categories accidentally made it into our `kept` set. If even one unwanted page is in the `kept` set, the test will fail and print the name of the category that slipped through.

**`assert set(ids["article"]) <= kept and set(ids["empty"]) <= kept`**
Checks to make sure that all the valid "article" IDs and valid "empty" IDs from the answer key are fully present inside our `kept` set. If any are missing, the test fails.

**`assert stats == expected["extract"]`**
Finally, checks that our running tally of statistics exactly matches the expected statistics from the answer key. If the numbers don't match perfectly, the test fails.

#### `tests/unit/test_acquire_download.py`

> Imports and Setup

`import hashlib`
This brings in a built-in Python tool used to create security hashes (like digital fingerprints) for files or data.

`import pytest`
This brings in the `pytest` testing framework, which is used to write and run automated tests for your code.

`from wiki2shards.acquire.download import DownloadError, fetch, parse_sha1sums`
This imports three specific things from a custom file in your project (located at `wiki2shards/acquire/download.py`): an error type (`DownloadError`), a function to download things (`fetch`), and a function to read file fingerprints (`parse_sha1sums`).

> Creating a Fake Internet Connection for Testing

`class FakeTransport:`
This starts the blueprint for a "fake" internet connection. Instead of actually downloading a file from the internet, the tests will use this to pretend they are downloading something.

`    """In-memory server: honors Range only if asked to; fails once mid-file if asked to."""`
This is just a text comment explaining what this fake connection does: it pretends to be a server that can resume interrupted downloads, and it can be told to fail on purpose halfway through.

`    def __init__(self, data, honor_range=True, fail_after=None):`
This sets up the fake connection when it's first created. It takes the fake `data` to send, whether it should allow resuming (`honor_range`), and when it should pretend to crash (`fail_after`).

`        self.data, self.honor, self.fail_after, self.calls = data, honor_range, fail_after, []`
This saves those setup values inside the object so they can be used later. It also creates an empty list called `calls` to keep track of how many times a download was requested.

`    def open(self, url, start):`
This is the function that simulates opening a web address to start a download. It takes the `url` (the web address) and a `start` position (where to start downloading from).

`        self.calls.append(start)`
This records the `start` position into the `calls` list so the test can check it later to see if it tried to resume.

`        begin = start if self.honor else 0`
If this fake server supports resuming (`self.honor`), it will start sending data from the requested `start` point. Otherwise, it stubbornly starts over from the very beginning (`0`).

`        fail_after, self.fail_after = self.fail_after, None`
This grabs the point where the server is supposed to crash and then resets it to `None`. This ensures the server only crashes on the *first* try, allowing the second try to succeed.

`        def chunks():`
This creates a mini-function inside `open` that will chop the fake data into small pieces to simulate a download streaming over time.

`            for i in range(begin, len(self.data), 10):`
This loops through the fake data, starting from the `begin` point, jumping 10 bytes forward at a time.

`                if fail_after is not None and i - begin >= fail_after:`
It checks if the server is supposed to crash AND if we have downloaded enough bytes to reach the crash point.

`                    raise OSError("connection reset")`
If the crash point is reached, it throws a fake "connection reset" error, exactly like what happens when your internet drops.

`                yield self.data[i : i + 10]`
If it didn't crash, it spits out the next 10 bytes of data to the downloader.

`        return chunks(), self.honor and start > 0`
This finishes the `open` function. It gives back the stream of data (`chunks()`) and a True/False value indicating if it successfully resumed an interrupted download.

> Testing the Download Resume Feature

`@pytest.mark.parametrize("honor", [True, False])`
This tells the testing framework to run the test below two times: once where the fake server allows resuming (`True`), and once where it doesn't (`False`).

`def test_fetch_resumes_and_verifies(tmp_path, monkeypatch, honor):`
This is the test function. It is given a temporary folder to save files in (`tmp_path`), a tool to modify how Python works temporarily (`monkeypatch`), and the `honor` True/False value from above.

`    monkeypatch.setattr("time.sleep", lambda s: None)`
This tricks the code into skipping any wait/sleep times during the download retry process, making the test run instantly instead of waiting around.

`    data = bytes(range(256)) * 4`
This creates a chunk of fake binary data (1,024 bytes long) that we will pretend to download.

`    sha1 = hashlib.sha1(data).hexdigest()`
This calculates the digital fingerprint (SHA-1 hash) of our fake data so we can verify it downloaded correctly.

`    t = FakeTransport(data, honor_range=honor, fail_after=300)`
This creates our fake internet connection, loaded with our fake data. We tell it to crash after sending 300 bytes.

`    fetch("fake://x", tmp_path / "x.bin", t, sha1)`
This runs the actual `fetch` function from your project. It tries to download a fake URL, save it as "x.bin" in the temp folder, use our fake connection `t`, and verify it against our fingerprint `sha1`.

`    assert (tmp_path / "x.bin").read_bytes() == data`
This checks (asserts) that the file saved to the hard drive exactly matches the original fake data. If they match, the download worked despite the crash!

`    assert t.calls[0] == 0 and t.calls[1] > 0  # second attempt asked to resume`
This checks the logs of our fake server. It verifies the first download attempt started at byte 0, and the second attempt (after the crash) tried to start at a later byte.

`    assert not (tmp_path / ".x.bin.part").exists()`
This makes sure the temporary partial download file (usually ending in `.part`) was cleaned up and deleted after the download finished successfully.

> Testing Corrupted Downloads

`def test_fetch_rejects_bad_sha1(tmp_path):`
This starts a new test to make sure the program behaves correctly if a downloaded file is corrupted.

`    with pytest.raises(DownloadError):`
This tells the test: "I expect the next line of code to fail with a DownloadError. If it DOESN'T fail, the test should fail."

`        fetch("fake://y", tmp_path / "y.bin", FakeTransport(b"abc"), "0" * 40)`
This tries to download the letters "abc", but it gives the program a completely fake fingerprint (40 zeros). The download function should realize the file doesn't match the fingerprint and throw an error.

`    assert not (tmp_path / "y.bin").exists()`
This checks to ensure that because the file was corrupted (the fingerprint didn't match), the program deleted the bad file and didn't leave it on the hard drive.

> Testing Fingerprint Reading

`def test_parse_sha1sums():`
This starts a final test to check if the program can correctly read a text file containing file fingerprints.

`    assert parse_sha1sums("ABC  a.bz2\ndef *b.txt\n") == {"a.bz2": "abc", "b.txt": "def"}`
This passes a raw string of text (representing a typical checksum file) into the `parse_sha1sums` function. It checks if the function correctly turns that text into a dictionary mapping the filenames ("a.bz2" and "b.txt") to their lowercase fingerprints ("abc" and "def").

#### Validation

> Line-by-Line Code Explanation

`uv run python - <<'EOF'`
This uses the `uv` tool to run Python, telling it to read and execute all the code that follows until it sees the marker `EOF` (End Of File).

`import json`
This loads Python's built-in tool for reading and writing JSON files, which is a common way to format data.

`from pathlib import Path`
This loads a tool that makes it easy to handle file and folder paths on your computer.

`from wiki2shards.extract.multistream import plan_units, read_siteinfo, stream_offsets`
This brings in three specific tools from a custom library (`wiki2shards`) designed to read Wikipedia's compressed "multistream" files.

`from wiki2shards.extract.xmlpages import extract_unit`
This brings in another tool from the same library that extracts the actual Wikipedia articles from the data.

`d = Path("data/fixture_dump/fixturewiki/20260901")`
This sets a shortcut to the folder where the test Wikipedia data is stored.

`xml = d / "fixturewiki-20260901-pages-articles-multistream.xml.bz2"`
This points directly to the compressed file containing the Wikipedia articles.

`index = d / "fixturewiki-20260901-pages-articles-multistream-index.txt.bz2"`
This points to the compressed index file, which acts as a map to find specific articles inside the massive XML file.

`offsets = stream_offsets(index)`
This uses the index file to calculate the starting points (offsets) of different data blocks within the compressed Wikipedia file.

`print("stream offsets:", offsets, "| dbname:", read_siteinfo(xml, offsets[0])["dbname"])`
This displays those starting points on the screen. It also reads the very first block of the XML file to find and display the database's name.

`out = Path("data/smoke/extracted")`
This sets a shortcut to a folder where the script will save the extracted data.

`out.mkdir(parents=True, exist_ok=True)`
This physically creates the output folder on your computer. If the folder already exists, it just safely ignores the command instead of crashing.

`total: dict[str, int] = {}`
This creates an empty dictionary (like a blank scorecard) to keep a running tally of extraction statistics.

`for unit in plan_units(offsets, xml.stat().st_size, 1):`
This starts a loop. It calculates how to divide the massive file into smaller, manageable chunks (units) based on the starting points and the total file size, and then processes them one by one.

`    extract_unit(xml, unit, out)`
For the current chunk in the loop, this extracts the Wikipedia pages and saves them to the output folder.

`    for k, v in json.loads((out / f"{unit.name}.stats.json").read_text()).items():`
This opens a small statistics file that was just created for the current chunk, reads its data, and loops through the numbers.

`        total[k] = total.get(k, 0) + v`
This takes the numbers from the current chunk and adds them to the main scorecard (`total`) so we have a running grand total.

`print("extract stats:", total)`
After all chunks are finished, this prints the final grand total of the statistics to the screen.

`assert total == json.loads((d / "fixture_expected.json").read_text())["extract"]`
This is a strict test. It opens a separate file that contains the "correct" expected answers and compares it to the grand total. If they do not match exactly, the program will crash immediately.

`print("OK: matches fixture_expected.json")`
If the program survived the strict test above without crashing, this prints a success message.

`EOF`
This tells the `uv` tool that the Python code is finished.

```sh
(wiki2shards) soheilsanati@Soheils-MacBook-Pro wiki2 % uv run python - <<'EOF'
heredoc> import json 
heredoc> from pathlib import Path 
heredoc> from wiki2shards.extract.multistream import plan_units, read_siteinfo, stream_offsets
heredoc> from wiki2shards.extract.xmlpages import extract_unit
heredoc> d = Path("data/fixture_dump/fixturewiki/20260901")
heredoc> xml = d / "fixturewiki-20260901-pages-articles-multistream.xml.bz2"
heredoc> index = d / "fixturewiki-20260901-pages-articles-multistream-index.txt.bz2"
heredoc> offsets = stream_offsets(index)
heredoc> print("stream offsets:", offsets, "| dbname:", read_siteinfo(xml, offsets[0])["dbname"])
heredoc> out = Path("data/smoke/extracted")
heredoc> out.mkdir(parents=True, exist_ok=True)
heredoc> total: dict[str, int] = {}
heredoc> for unit in plan_units(offsets, xml.stat().st_size, 1):
heredoc>     extract_unit(xml, unit, out)
heredoc>     for k, v in json.loads((out / f"{unit.name}.stats.json").read_text()).items():

heredoc>         total[k] = total.get(k, 0) + v
heredoc> print("extract stats:", total)
heredoc> assert total == json.loads((d / "fixture_expected.json").read_text())["extract"]
heredoc> print("OK: matches fixture_expected.json")
heredoc> EOF
```

```txt
stream offsets: [402, 21189, 45254] | dbname: fixturewiki
extract stats: {'kept': 257, 'pages': 300, 'skipped_model': 3, 'skipped_namespace': 20, 'skipped_redirect': 20}
OK: matches fixture_expected.json
```

```sh
uv run pytest -q tests/unit/test_extract_multistream.py tests/unit/test_extract_pages.py tests/unit/test_acquire_download.py

......                                                                                                                                                                                                             [100%]
6 passed in 0.62s
```

This command tells a fast Python tool called `uv` to run a testing program (`pytest`) quietly, specifically checking three exact test files.

> Breaking down the command piece by piece:

* **uv**: A modern, very fast tool used to manage Python projects and run Python programs.
* **run**: An instruction that tells `uv` to execute the program that comes next inside your project's isolated environment.
* **pytest**: A popular tool used by developers to run tests on their Python code to make sure it works correctly.
* **-q**: Short for "quiet." It tells `pytest` to print less text on your screen, hiding the extra details and showing only the most important pass or fail results.
* **tests/unit/test_extract_multistream.py**: The folder path and file name of the first specific set of tests you want to run.
* **tests/unit/test_extract_pages.py**: The folder path and file name of the second specific set of tests you want to run.
* **tests/unit/test_acquire_download.py**: The folder path and file name of the third specific set of tests you want to run.

> Note:

from the guide, I was told to:

```
Expected: 6 passed. Then run the `lint` and type `gate`.
```

> What is `lint`? In software development, this phrase means you need to run two automated quality checks on your code to ensure it is clean and error-free before proceeding (such as before merging or deploying it).

>> Now in order to run the `lint`, Since we are using `uv` (the modern Python package manager), our project is written in `Python`. In this ecosystem, we run tools through `uv` to ensure they execute using my project's specific dependencies and isolated environment. The steps are as follow:

1. Run the Linter
> Projects using uv almost always use Ruff for linting, as they are built by the same company. Run this command in the root of your project:
```sh
uv run ruff check .
```
> I received:
```
All checks passed!
```

2. Run the Type Gate
> The standard type checker for Python is MyPy, though Pyright is also common. Run:
```sh
uv run mypy .
```
> I received:

```sh
(wiki2shards) soheilsanati@Soheils-MacBook-Pro wiki2 % uv run mypy .
tests/unit/test_tokenizer_fp.py:6: error: Function is missing a return type annotation  [no-untyped-def]
tests/unit/test_tokenizer_fp.py:6: note: Use "-> None" if function does not return a value
tests/unit/test_tokenizer_fp.py:12: error: Function is missing a return type annotation  [no-untyped-def]
tests/unit/test_tokenizer_fp.py:12: note: Use "-> None" if function does not return a value
tests/helpers/build.py:75: error: Skipping analyzing "wiki2shards.pipeline.runner": module is installed, but missing library stubs or py.typed marker  [import-untyped]
tests/helpers/build.py:75: note: See https://mypy.readthedocs.io/en/stable/running_mypy.html#missing-imports
tests/unit/test_extract_pages.py:8: error: Function is missing a type annotation  [no-untyped-def]
tests/unit/test_extract_multistream.py:15: error: Function is missing a type annotation  [no-untyped-def]
tests/unit/test_extract_multistream.py:23: error: Function is missing a type annotation  [no-untyped-def]
tests/unit/test_extract_multistream.py:24: error: Call to untyped function "_paths" in typed context  [no-untyped-call]
tests/unit/test_io_manifest.py:17: error: Missing type arguments for generic type "dict"  [type-arg]
tests/unit/test_io_manifest.py:79: error: Function is missing a type annotation  [no-untyped-def]
tests/unit/test_io_hashing.py:12: error: Function is missing a return type annotation  [no-untyped-def]
tests/unit/test_io_hashing.py:12: note: Use "-> None" if function does not return a value
tests/unit/test_io_hashing.py:24: error: Function is missing a return type annotation  [no-untyped-def]
tests/unit/test_io_hashing.py:24: note: Use "-> None" if function does not return a value
tests/unit/test_io_binfmt.py:22: error: Function is missing a type annotation  [no-untyped-def]
tests/unit/test_io_binfmt.py:46: error: Function is missing a return type annotation  [no-untyped-def]
tests/unit/test_io_binfmt.py:46: note: Use "-> None" if function does not return a value
tests/unit/test_io_binfmt.py:47: error: Call to untyped function "_sample" in typed context  [no-untyped-call]
tests/unit/test_io_binfmt.py:71: error: Function is missing a return type annotation  [no-untyped-def]
tests/unit/test_io_binfmt.py:71: note: Use "-> None" if function does not return a value
tests/unit/test_io_atomic.py:9: error: Function is missing a type annotation  [no-untyped-def]
tests/unit/test_config.py:7: error: Function is missing a return type annotation  [no-untyped-def]
tests/unit/test_config.py:7: note: Use "-> None" if function does not return a value
tests/unit/test_config.py:28: error: Function is missing a type annotation  [no-untyped-def]
tests/unit/test_config.py:43: error: Function is missing a type annotation  [no-untyped-def]
tests/unit/test_config.py:52: error: Function is missing a type annotation  [no-untyped-def]
tests/unit/test_acquire_download.py:11: error: Function is missing a type annotation  [no-untyped-def]
tests/unit/test_acquire_download.py:14: error: Function is missing a type annotation  [no-untyped-def]
tests/unit/test_acquire_download.py:19: error: Function is missing a return type annotation  [no-untyped-def]
tests/unit/test_acquire_download.py:25: error: Call to untyped function "chunks" in typed context  [no-untyped-call]
tests/unit/test_acquire_download.py:29: error: Function is missing a type annotation  [no-untyped-def]
tests/unit/test_acquire_download.py:33: error: Call to untyped function "FakeTransport" in typed context  [no-untyped-call]
tests/unit/test_acquire_download.py:40: error: Function is missing a type annotation  [no-untyped-def]
tests/unit/test_acquire_download.py:42: error: Call to untyped function "FakeTransport" in typed context  [no-untyped-call]
tests/unit/test_acquire_download.py:46: error: Function is missing a return type annotation  [no-untyped-def]
tests/unit/test_acquire_download.py:46: note: Use "-> None" if function does not return a value
Found 29 errors in 10 files (checked 34 source files)
```

>>> Following is the fix to the errors above:

---

for 2, here is the entire code

```py
def test_page_filters(dumps, tmp_path: Path) -> None:
    d = dumps / "fixturewiki" / "20260901"
    xml = d / "fixturewiki-20260901-pages-articles-multistream.xml.bz2"
    units = plan_units(
...
```

The issue is that the `dumps` parameter is missing a type annotation. Because `dumps` is being used with the `/` operator, it is a `Path` object (like a pytest fixture returning a path).

MyPy requires **every** parameter to have a type hint when strict mode is enabled.

Here is the fixed code:

```python
from pathlib import Path


def test_page_filters(dumps: Path, tmp_path: Path) -> None:
    d = dumps / "fixturewiki" / "20260901"
    xml = d / "fixturewiki-20260901-pages-articles-multistream.xml.bz2"
    units = plan_units(

```

> Why this fixes it:

* Adding `: Path` to `dumps` completes the annotations for all arguments.
* Make sure `from pathlib import Path` is imported at the top of the file.

*(If `dumps` is a custom fixture that isn't a standard `Path`, you can use `dumps: Any` after importing `from typing import Any`)*

---

solution for 
`tests/helpers/build.py:75: error: Skipping analyzing "wiki2shards.pipeline.runner": module is installed, but missing library stubs or py.typed marker  [import-untyped]`

Here are the standard solutions to fix this error, depending on whether `wiki2shards` is your own internal package or a third-party dependency.

...
> Solution 1: If `wiki2shards` is your own package (Recommended)

If `wiki2shards` is code within your repository, MyPy needs to know that the package supports type checking.

1. **Add an empty `py.typed` file** inside the package folder:
```bash
touch wiki2shards/py.typed
# or if using src layout:
touch src/wiki2shards/py.typed

```


2. **Ensure `py.typed` is included in build metadata** (if configured in `pyproject.toml`):
```toml
[tool.setuptools.package-data]
"wiki2shards" = ["py.typed"]

```

...
> Solution 2: Inline Ignore (Quickest Fix)

If you just want MyPy to ignore this single import without affecting the rest of your project, append `# type: ignore[import-untyped]` to line 75 in `tests/helpers/build.py`:

```python
from wiki2shards.pipeline import runner  # type: ignore[import-untyped]

```

....
> Solution 3: Configure MyPy in `pyproject.toml` (Project-wide Fix)

If `wiki2shards` is an external dependency or third-party library missing type stubs, tell MyPy to ignore missing imports for it across the whole project:

Add this block to your `pyproject.toml`:

```toml
[[tool.mypy.overrides]]
module = "wiki2shards.*"
ignore_missing_imports = true

```

---

tests/unit/test_extract_multistream.py:16: error: Function is missing a return type annotation  [no-untyped-def]

This error means line 16 of `tests/unit/test_extract_multistream.py` defines a function without specifying what it returns.

> How to fix it

If it is a **test function** (or a helper function that doesn't return anything), add `-> None` to the end of the function definition:

```python
# BEFORE (Line 16)
def test_extract_multistream(tmp_path: Path):
    ...

# AFTER
def test_extract_multistream(tmp_path: Path) -> None:
    ...

```

If it is a **helper function** that returns a value (e.g., returning a `Path`, `dict`, or `list`), annotate the exact return type instead:

```python
# Example if the helper returns a Path
def _get_fixture_path(name: str) -> Path:
    return Path(f"fixtures/{name}")

```

---

```py
def _paths(dumps:Path):
    d = dumps / "fixturewiki" / "20260901"
    return (
        d / "fixturewiki-20260901-pages-articles-multistream.xml.bz2",
        d / "fixturewiki-20260901-pages-articles-multistream-index.txt.bz2",
    )
```

Add `-> tuple[Path, Path]` as the return type annotation:

```python
from pathlib import Path


def _paths(dumps: Path) -> tuple[Path, Path]:
    d = dumps / "fixturewiki" / "20260901"
    return (
        d / "fixturewiki-20260901-pages-articles-multistream.xml.bz2",
        d / "fixturewiki-20260901-pages-articles-multistream-index.txt.bz2",
    )

```

This will fix both the missing return type error for `_paths` and any `Call to untyped function "_paths" in typed context` errors where this helper is used.

---


error: tests/unit/test_io_manifest.py:17: error: Missing type arguments for generic type "dict"  [type-arg]

``` py
def sample() -> dict:
    shard = {
        "index": 0,
        "n_rows": 2,
        "bin": "train-00000.bin",
        "bin_bytes": 20,
        "bin_sha256": SHA,
        "idx": "train-00000.idx",
        "idx_bytes": 168,
        "idx_sha256": SHA,
    }
```

Change `dict` to `dict[str, Any]` (or `dict[str, str | int]`), because MyPy requires you to specify the data types for dictionary keys and values.

Here is the updated code:

```python
from typing import Any


def sample() -> dict[str, Any]:
    shard = {
        "index": 0,
        "n_rows": 2,
        "bin": "train-00000.bin",
        "bin_bytes": 20,
        "bin_sha256": SHA,
        "idx": "train-00000.idx",
        "idx_bytes": 168,
        "idx_sha256": SHA,
    }
    return shard

```

> Why this fixes it:

* `str` specifies that all key names (`"index"`, `"n_rows"`, etc.) are strings.
* `Any` tells MyPy that values can be of mixed types (`int` and `str`). Make sure to import `Any` from `typing` at the top of the file.

---

error : tests/unit/test_io_binfmt.py:22: error: Function is missing a type annotation  [no-untyped-def]

``` py
def: def _sample(n_rows=3):
    seg = Segments(
        row_seg_ptr=np.array([0, 2, 3, 5][: n_rows + 1], dtype=np.uint64),
        doc_uid=np.array([11, 12, 12, 13, 14], dtype=np.uint64),
        start=np.array([0, 3, 0, 0, 4], dtype=np.uint32),
        length=np.array([3, 5, 8, 4, 4], dtype=np.uint32),
        piece=np.array([0, 0, 1, 0, 0], dtype=np.uint32),
    )
```

To resolve the `[no-untyped-def]` error from Mypy, add type annotations for both the input parameter (`n_rows: int`) and the return type (`-> Segments`):

The return type annotation `-> Segments` causes a type mismatch because the function returns a tuple containing `(hdr, seg)`. Update the return type hint to `tuple[IdxHeader, Segments]` (or `Tuple[IdxHeader, Segments]` for Python 3.8 and older):

```python
from typing import Tuple  # Only required for Python 3.8 or older

def _sample(n_rows: int = 3) -> tuple[IdxHeader, Segments]:
    seg = Segments(
        row_seg_ptr=np.array([0, 2, 3, 5][: n_rows + 1], dtype=np.uint64),
        doc_uid=np.array([11, 12, 12, 13, 14], dtype=np.uint64),
        start=np.array([0, 3, 0, 0, 4], dtype=np.uint32),
        length=np.array([3, 5, 8, 4, 4], dtype=np.uint32),
        piece=np.array([0, 0, 1, 0, 0], dtype=np.uint32),
    )
    hdr = IdxHeader(
        itemsize=2,
        pack_mode=0,
        split=1,
        row_len=8,
        shard_index=4,
        n_rows=3,
        n_segments=5,
        n_pad_tokens=0,
        first_global_row=120,
        bin_sha256=bytes(range(32)),
        tokenizer_fp=bytes(range(32, 64)),
    )
    return hdr, seg

```

> Why this happens

Mypy flags function definitions missing type hints when `--disallow-untyped-defs` or `--strict` mode is enabled. Both the parameters and the return value must be explicitly typed.

---

tests/unit/test_acquire_download.py:11: error: Function is missing a type annotation  [no-untyped-def]

```py
    def __init__(self, data, honor_range=True, fail_after=None):
        self.data, self.honor, self.fail_after, self.calls = data, honor_range, fail_after, []
```

Annotate the parameters and add `-> None` as the return type for `__init__`:

```python
from typing import Any  # or use specific types like bytes if applicable


class MockDownloader:  # or your class name
    def __init__(
        self,
        data: Any,  # Replace 'Any' with 'bytes' or 'str' if data has a specific type
        honor_range: bool = True,
        fail_after: int | None = None,  # Use Optional[int] if using Python < 3.10
    ) -> None:
        self.data, self.honor, self.fail_after, self.calls = (
            data,
            honor_range,
            fail_after,
            [],
        )

```

> Key Requirements

* **`-> None`**: `__init__` must always return `None`. Mypy requires this explicit return hint to treat the method as typed.
* **`fail_after: int | None`**: Since the default is `None`, the type must be optional/nullable.

---

tests/unit/test_config.py:28: error: Function is missing a type annotation  [no-untyped-def]

```py
def test_config_rejects(patch):
    base = {
        "name": "x",
        "source": {"wiki": "x", "date": "20260901"},
        "workdir": "w",
        "output": "o",
        "tokenizer": "tiktoken:gpt2",
        "pack": {"seq_len": 8},
        "clean": {"soft_filters": {"list_page": False}},
    }
    base.update(patch)
    with pytest.raises(ValueError):
        BuildConfig.model_validate(base)
```

Add parameter and return type annotations (`patch: dict[str, Any]` and `-> None`):

```python
from typing import Any


def test_config_rejects(patch: dict[str, Any]) -> None:
    base = {
        "name": "x",
        "source": {"wiki": "x", "date": "20260901"},
        "workdir": "w",
        "output": "o",
        "tokenizer": "tiktoken:gpt2",
        "pack": {"seq_len": 8},
        "clean": {"soft_filters": {"list_page": False}},
    }
    base.update(patch)
    with pytest.raises(ValueError):
        BuildConfig.model_validate(base)

```

> Key Requirements

* **`-> None`**: Pytest test functions do not return anything. Mypy requires `-> None` to treat the test function as typed.
* **`patch: dict[str, Any]`**: Because `patch` is passed into `base.update()`, typing it as a dictionary (or `Any` if it can be a fixture with non-dict properties) satisfies Mypy's type check. *(Use `Dict[str, Any]` from `typing` if on Python 3.8 or older).*

---